# Sky-to-Science Coefficient Transfer via Residual Emissivity Learning (RETN)

## 1. Problem

At a specific timestamp, the LVM instrument records simultaneously a **science** exposure and two **sky** exposures at different points on the sky (a near-sky and a far-sky pointing). The sky exposures are decomposed into physical components (moon+zodi continuum, mesospheric OH lines, atomic/ionospheric lines, diffuse airglow continuum, etc.) using a physical model that internally handles lunar geometry (altitude, phase, distance to sci pointing), zodiacal light geometry (solar elongation), airglow van Rhijn factors, and extinction.

The output of the sky decomposition per exposure is a vector of coefficients — one per named sky feature — that, together with the physical basis functions, reconstructs the arm's observed spectrum. For each row we have:

- $\mathbf{c}_{\rm near} \in \mathbb{R}^{442}$, $\mathbf{x}_{\rm ctx,near} \in \mathbb{R}^{25}$
- $\mathbf{c}_{\rm far}  \in \mathbb{R}^{442}$, $\mathbf{x}_{\rm ctx,far}  \in \mathbb{R}^{25}$
- $\mathbf{c}_{\rm sci}  \in \mathbb{R}^{442}$ (target at training time), $\mathbf{x}_{\rm ctx,sci} \in \mathbb{R}^{25}$

The prediction problem: given the two sky decompositions and the three pointing contexts, predict the science-pointing coefficients so the sky component of the science exposure can be reconstructed and subtracted.

## 2. Physical Prior: Intrinsic Emissivity Transfers

The airglow geometry factor

$$s_{\rm arm}(\lambda) = V_{\rm van~Rhijn}(z_{\rm arm}, h_\lambda) \cdot 10^{-0.4\, k_{\rm eff}(\lambda)\, (X_{\rm arm} - 1)}$$

is a function of altitude $z_{\rm arm}$, airmass $X_{\rm arm}$, and per-coefficient effective extinction. Applied per-coefficient per-row it gives us the intrinsic emissivity:

$$\mathbf{em}_{\rm arm} = \mathbf{c}_{\rm arm} / \mathbf{s}_{\rm arm}$$

which is the zenith-equivalent amplitude — the physical quantity that a shared airglow layer would produce independent of the arm's line of sight. Moon and continuum airglow coefficients follow different geometric factors but share the same normalization pattern.

Because the three arms sample nearby lines of sight *simultaneously*:

$$\mathbf{em}_{\rm sci} = \mathbf{em}_{\rm near} + \delta_{\rm GW} + \delta_{\rm geom-residual} + \delta_{\rm decomp-noise}$$

The dominant residual term $\delta_{\rm GW}$ is gravity-wave modulation on the mesospheric emission (typically 10% RMS over 10-minute timescales for OH). The diffuse continuum residual $\delta_{\rm geom-residual}$ is *geometry-driven* rather than gravity-wave-driven — the airglow geometry factor cancels out only to the accuracy of the fitted $k_{\rm eff}$ and shell-height assumption. The moon+zodi residual is a mix of both.

**Direct copy-near-normalized transfer** (predict $\mathbf{c}_{\rm sci} = \mathbf{em}_{\rm near} \cdot \mathbf{s}_{\rm sci}$, no learning) achieves batch median $\text{pRMSE} \approx 0.4$ display units. This is the physical baseline: any learned model must justify beating it.

## 3. Architecture: Residual Emissivity Transfer Network (RETN)

RETN encodes the physical prior as a hard skip connection over which a residual is added:

$$\mathbf{em}_{\rm pred} = \mathbf{em}_{\rm near} + f_\theta(\mathbf{em}_{\rm near}, \mathbf{em}_{\rm far}, \mathbf{x}_{\rm ctx,near}, \mathbf{x}_{\rm ctx,far}, \mathbf{x}_{\rm ctx,sci})$$
$$\mathbf{c}_{\rm pred} = \text{ReLU}\!\left(\mathbf{em}_{\rm pred} \cdot \mathbf{s}_{\rm sci}\right)$$

Components:

1. **Shared arm encoder**: $(\mathbf{em}_{\rm arm}, \mathbf{x}_{\rm ctx,arm}) \mapsto \mathbf{h}_{\rm arm} \in \mathbb{R}^{256}$. Shared weights for near and far so the encoder learns arm-agnostic features.
2. **Ctx encoder**: $\mathbf{x}_{\rm ctx,sci} \mapsto \mathbf{h}_{\rm ctx} \in \mathbb{R}^{64}$.
3. **Trunk**: fuses $[\mathbf{h}_{\rm near}, \mathbf{h}_{\rm far}, \mathbf{h}_{\rm ctx}, \mathbf{x}_{\rm ctx,sci} - \mathbf{x}_{\rm ctx,near}, \mathbf{x}_{\rm ctx,sci} - \mathbf{x}_{\rm ctx,far}]$. The explicit **ctx-difference features** are the direct handle the residual head needs for the geometry-driven diffuse continuum residual.
4. **Per-group residual heads** for `moon+zodi` (32 coefs = MoonZodi_bs joined with HO2+FeO+O2Ac since they both drive continuum flux), `mesospheric` (403 OH lines), `ionospheric` (4), `atomic` (3). Every final layer is **zero-initialized**, so an untrained RETN produces exactly $\mathbf{em}_{\rm near}$: the copy-near baseline. Any non-zero residual must be *earned* by evidence.
5. **Positivity via soft penalty + hard clip**. Loss includes $\lambda_{\rm pos}\, \langle\text{ReLU}(-\mathbf{em}_{\rm pred,phys})^2\rangle$ during training so the network is directly discouraged from producing predictions that would need clipping. The clip at inference is a defensive guard.

## 4. Training Loss

$$L = L_{\rm em} + \lambda_{\rm pos}\, L_{\rm pos} + \lambda_{\rm bias}\, L_{\rm bias}$$

$$L_{\rm em} = \text{smooth\_L1}\!\left(\mathbf{em}_{\rm pred, std},\; \mathbf{em}_{\rm true, std}\right) \cdot w_{\rm pe}$$

Heteroscedastic weights $w_{\rm pe} = 1/\sigma^2$ from the decomposition's `COEF_ERR` (propagated to emissivity space and per-column normalized). This mirrors the sigma-aware loss used by the older architecture, but now in the intrinsic-emissivity space that transfers between arms.

$$L_{\rm pos} = \langle \text{ReLU}(-\mathbf{em}_{\rm pred, phys})^2 \rangle$$

Default $\lambda_{\rm pos} = 0.1$ — small but non-negligible.

$$L_{\rm bias} = \sum_{c} \frac{1}{|\mathcal{B}|}\sum_{b} \left(\frac{\langle\, f_{\rm pred,c,b} - f_{\rm true,c,b}\, \rangle_{\rm batch}}{\sqrt{\langle f_{\rm true,c,b}^2 \rangle_{\rm batch}}}\right)^2$$

Per-component systematic-flux bias, where $f_{\rm pred,c,b} = \sum_{j \in c} I_{i,b,j}\, c_{\rm pred,i,j}$ and $I_{i,b,j}$ is the precomputed band-averaged basis integral for row $i$, band $b$, coefficient $j$ (cell 14). Component groups $c \in \{\text{moon+zodi}, \text{diffuse}, \text{lines}\}$ match the SSFR cell. Normalized per band-per-group by RMS true flux so the term is dimensionless. Default $\lambda_{\rm bias} = 100$ (calibrated so bias contributes 15–20% of total loss at convergence).

## 5. Evaluation: Sky-Subtraction Fractional Residual (SSFR)

The retired coefficient-space per-target RMSE (eRMSE) weighted all 442 coefficients equally and did not reflect flux impact — a coefficient error of 1 unit in a mesospheric OH line matters far less to reconstructed flux than the same-magnitude error in a broadband moon+zodi coefficient. **Deployment metric is flux quality**, so we now report SSFR as the headline number:

For each row, wavelength band $b$, and component group $c \in \{\text{moon+zodi}, \text{diffuse}, \text{lines}\}$:

$$\text{SSFR}_c(b) = \frac{\sqrt{\langle (f_{\rm pred,c} - f_{\rm true-recon,c})^2 \rangle_{\lambda \in b}}}{\sqrt{\langle f_{\rm obs,sci}^2 \rangle_{\lambda \in b}}}$$

$$\text{BIAS}_c(b) = \frac{\langle f_{\rm pred,c} - f_{\rm true-recon,c} \rangle_{\lambda \in b}}{\sqrt{\langle f_{\rm obs,sci}^2 \rangle_{\lambda \in b}}}$$

Aggregated flavors (same denominator for direct comparability):
- $\text{SSFR}_{\rm floor}$: RMS residual of true-recon vs observed sci — the decomposition fidelity floor, network-independent.
- $\text{SSFR}_{\rm network}$: RMS residual of predicted-recon vs true-recon — pure network error, decomposition-independent.
- $\text{SSFR}_{\rm deployment}$: RMS residual of predicted-recon vs observed sci — the sky-subtraction quality.

Bands are chosen to isolate physical processes: continuum-only windows in blue/green/red/NIR, and OH-line-dominated windows around 5000/5800/7300 Å.

**Interpretation**: $\text{SSFR} = 0.05$ means the RMS residual is 5% of the RMS sky brightness in that band.

## 6. Data + Filtering

Three physical decomposition FITS files (sky-near, sky-far, science) share a common input FITS source with matched row indices. The loader enforces numeric coercion, finite-row filtering, LMC/SMC exclusion, and reduced-$\chi^2$ gating on the decomposition fits. Splits are night-held-out so no observation from a night appearing in training also appears in val or test.


## Data loading + physical constants + reconstruction helpers

In [ ]:
import os
os.environ.setdefault('LVMCORE_DIR', '/Users/droryn/prog/lvm/lvmcore')
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from astropy.io import fits
from astropy.table import Table
from IPython.display import HTML, display
from sky_decomp.fit import reconstruct_component_spectra
from sky_decomp.lsf_surface_iterative import (
    LSFSurfaceState,
    SkyDecompLSFSurfaceIterative,
    load_lsf_surface_state,
)
from sky_decomp.moon_zodi_lsf_surface_iterative import (
    MoonZodiLSFSurfaceIterativeResult,
    SkyDecompMoonZodiLSFSurfaceIterative,
)
from sky_decomp.moon_zodi_model import MoonZodiObservation, MoonZodiState
from sky_decomp.result_io import MOON_ZODI_HDU_NAMES, load_moon_zodi_state
import re

# Journal-style axes: no gridlines, black box (mirrored) axes, outside ticks.
# Applied by mutating plotly_white in place so every explicit template='plotly_white'
# call in this notebook (and the default template) picks it up automatically.
for _ax in (pio.templates["plotly_white"].layout.xaxis,
            pio.templates["plotly_white"].layout.yaxis):
    _ax.showgrid = False
    _ax.showline = True
    _ax.mirror = True
    _ax.linecolor = "black"
    _ax.linewidth = 1
    _ax.ticks = "outside"
    _ax.zeroline = False
pio.templates.default = "plotly_white"

# Reused constants
FACTOR = 1e14
PALACE_DIR = '../'

In [ ]:
# Reused decomposition/context loading helpers
LAYER_HEIGHTS_KM = {
    'mesospheric_oh': 87.0,
    'mesospheric_atomic': 95.0,
    'ionospheric_red': 285.0,
}

TIME_FEATURE_NAMES = {
    'obstime_year_sin',
    'obstime_year_cos',
    'obstime_day_sin',
    'obstime_day_cos',
    'obstime_lunation_sin',
    'obstime_lunation_cos',
}

VAN_RHIJN_FEATURES = {
    'vanrhijn_87km': LAYER_HEIGHTS_KM['mesospheric_oh'],
    'vanrhijn_95km': LAYER_HEIGHTS_KM['mesospheric_atomic'],
    'vanrhijn_285km': LAYER_HEIGHTS_KM['ionospheric_red'],
}


def _as_array(x):
    arr = np.asarray(x)
    if arr.dtype.kind in ('U', 'S', 'O'):
        return None
    return arr.astype(np.float32)


def _coerce_coef_hdu_to_table(coef_hdu):
    data = coef_hdu.data
    if isinstance(coef_hdu, (fits.BinTableHDU, fits.TableHDU)):
        return Table(data)

    arr = np.asarray(data, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D COEF image, got shape={arr.shape}')

    n_coef = arr.shape[1]
    names = []
    for i in range(n_coef):
        key = f'COEF{i:04d}'
        names.append(str(coef_hdu.header.get(key, f'coef_{i:04d}')))
    return Table({name: arr[:, i] for i, name in enumerate(names)})


def _select_context_from_labels(meta, meta_upper, labels, base_name):
    e_key = f'SKYE_{base_name.upper()}'
    w_key = f'SKYW_{base_name.upper()}'
    if e_key not in meta_upper or w_key not in meta_upper:
        return None

    arr_e = _as_array(meta[meta_upper[e_key]])
    arr_w = _as_array(meta[meta_upper[w_key]])
    if arr_e is None or arr_w is None:
        raise ValueError(f'Labeled context columns for {base_name} are non-numeric.')

    is_e = labels == 'SKYE'
    is_w = labels == 'SKYW'
    if not np.all(is_e | is_w):
        bad = np.unique(labels[~(is_e | is_w)])
        raise ValueError(f'Unexpected label values: {bad}')

    return np.where(is_e, arr_e, arr_w).astype(np.float32)


def _table_to_float32_matrix(tbl, value_name):
    names = list(tbl.colnames)
    cols = []
    numeric_names = []
    for name in names:
        arr = _as_array(tbl[name])
        if arr is not None:
            cols.append(arr)
            numeric_names.append(name)

    if len(cols) == 0:
        raise ValueError(f'No numeric {value_name} columns found.')

    return np.column_stack(cols).astype(np.float32), numeric_names


def _altitude_to_zenith_deg(alt_deg):
    alt = np.asarray(alt_deg, dtype=np.float64)
    return np.clip(90.0 - alt, 0.0, 89.9)


def _van_rhijn_factor(alt_deg, layer_height_km, earth_radius_km=6371.0):
    z_rad = np.deg2rad(_altitude_to_zenith_deg(alt_deg))
    shell_ratio = float(earth_radius_km) / (float(earth_radius_km) + float(layer_height_km))
    denom = 1.0 - (shell_ratio * np.sin(z_rad)) ** 2
    denom = np.clip(denom, 1e-6, None)
    return (1.0 / np.sqrt(denom)).astype(np.float32)


def _extract_obstime_mjd(meta, meta_upper):
    from astropy.time import Time

    candidates = ('OBSTIME', 'MJD', 'MJD_OBS', 'MJD-OBS', 'DATE_OBS', 'DATE-OBS')
    for key in candidates:
        if key not in meta_upper:
            continue

        raw = np.asarray(meta[meta_upper[key]])
        if raw.dtype.kind in ('i', 'u', 'f'):
            return raw.astype(np.float64)

        s = np.asarray(raw).astype(str)
        try:
            return Time(s, format='isot', scale='utc').mjd.astype(np.float64)
        except ValueError:
            return Time(s).mjd.astype(np.float64)

    raise KeyError('Could not find an OBSTIME/MJD/DATE-OBS-like META column for time features')


def _build_obstime_feature(meta, meta_upper, feature_name):
    mjd = _extract_obstime_mjd(meta, meta_upper)
    two_pi = 2.0 * np.pi

    if feature_name == 'obstime_year_sin':
        return np.sin(two_pi * mjd / 365.2422).astype(np.float32)

    if feature_name == 'obstime_year_cos':
        return np.cos(two_pi * mjd / 365.2422).astype(np.float32)

    frac_day = mjd - np.floor(mjd)
    if feature_name == 'obstime_day_sin':
        return np.sin(two_pi * frac_day).astype(np.float32)

    if feature_name == 'obstime_day_cos':
        return np.cos(two_pi * frac_day).astype(np.float32)

    if feature_name == 'obstime_lunation_sin':
        return np.sin(two_pi * mjd / 29.53058867).astype(np.float32)

    if feature_name == 'obstime_lunation_cos':
        return np.cos(two_pi * mjd / 29.53058867).astype(np.float32)

    raise KeyError(f'Unsupported obstime feature name: {feature_name}')


# --- Astropy-computed pointing geometry ------------------------------------
# alt / az / airmass and sun / moon positions are computed from RA, DEC and
# obstime via astropy rather than read directly from META. Centralising the
# geometry here means a per-column error in the input META (which happens for
# online-derived quantities) does not silently propagate through the pipeline.
# `moon_phase` is intentionally not in this set: illumination fraction is not
# a pure geometry feature and is still read from META.
ASTROPY_GEOMETRY_FEATURES = {
    'alt', 'az_sin', 'az_cos', 'airmass',
    'moon_alt', 'moon_az_sin', 'moon_az_cos', 'moon_sep',
    'sun_alt', 'sun_az_sin', 'sun_az_cos', 'sun_sep',
    'sci_sep',
}

# META columns stored as an angle in degrees that wraps at 0/360.
# `<stem>_sin` / `<stem>_cos` in context_cols pull the raw column and fold it.
CYCLIC_META_DEGREE_FEATURES = {'moon_phase'}

_LCO_EARTHLOCATION = None


def _lco_earth_location():
    """LCO EarthLocation, cached at first use."""
    global _LCO_EARTHLOCATION
    if _LCO_EARTHLOCATION is not None:
        return _LCO_EARTHLOCATION
    from astropy.coordinates import EarthLocation
    import astropy.units as u
    try:
        _LCO_EARTHLOCATION = EarthLocation.of_site('Las Campanas Observatory')
    except Exception:
        # Fallback to published LCO coordinates (Baade / du Pont site).
        _LCO_EARTHLOCATION = EarthLocation(
            lat=-29.00889 * u.deg,
            lon=-70.68992 * u.deg,
            height=2380.0 * u.m,
        )
    return _LCO_EARTHLOCATION


def _pointing_ra_dec_columns(meta_upper, kind):
    """Return the META column names for the pointing RA / DEC of one kind.

    Uses SKY_NEAR_RA / SKY_NEAR_DEC and SKY_FAR_RA / SKY_FAR_DEC for the two
    sky arms, so the near/far ordering (by separation from science) and the
    east/west assignment are picked up from the same columns that already
    drive the rest of the pipeline; no consultation of SKY_NEAR_LABEL /
    SKY_FAR_LABEL is needed for the pointing geometry.
    """
    if kind == 'sci':
        ra_key = next((meta_upper[k] for k in ('SCI_RA', 'RA') if k in meta_upper), None)
        dec_key = next((meta_upper[k] for k in ('SCI_DEC', 'DEC') if k in meta_upper), None)
    elif kind == 'sky1':
        ra_key = meta_upper.get('SKY_NEAR_RA')
        dec_key = meta_upper.get('SKY_NEAR_DEC')
    elif kind == 'sky2':
        ra_key = meta_upper.get('SKY_FAR_RA')
        dec_key = meta_upper.get('SKY_FAR_DEC')
    else:
        raise ValueError(f'unexpected kind {kind!r}')
    if ra_key is None or dec_key is None:
        raise KeyError(f'RA / DEC columns for kind={kind!r} not found in META')
    return ra_key, dec_key


def _compute_astropy_geometry(meta, meta_upper, kind):
    """Compute all astropy-derived geometry features for one pointing kind.

    Returns a dict keyed by feature name (subset of ASTROPY_GEOMETRY_FEATURES)
    with float32 arrays of length n_rows. Pointing RA / DEC come from the
    kind-specific columns; obstime comes from `_extract_obstime_mjd`. Sun and
    Moon topocentric positions are computed at LCO for every obstime. Airmass
    is sec(z) = 1 / sin(alt), NaN below the horizon.
    """
    from astropy.coordinates import AltAz, SkyCoord, get_body, get_sun
    from astropy.time import Time
    import astropy.units as u

    ra_key, dec_key = _pointing_ra_dec_columns(meta_upper, kind)
    ra = np.asarray(meta[ra_key], dtype=np.float64)
    dec = np.asarray(meta[dec_key], dtype=np.float64)

    # Science pointing coords are needed for the sky-to-sci angular separation feature.
    sci_ra_key, sci_dec_key = _pointing_ra_dec_columns(meta_upper, 'sci')
    sci_ra_col = np.asarray(meta[sci_ra_key], dtype=np.float64)
    sci_dec_col = np.asarray(meta[sci_dec_key], dtype=np.float64)

    mjd = _extract_obstime_mjd(meta, meta_upper)
    time = Time(mjd, format='mjd', scale='utc')

    lco = _lco_earth_location()
    frame = AltAz(obstime=time, location=lco)

    pointing = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame='icrs')
    pointing_altaz = pointing.transform_to(frame)
    sci_pointing = SkyCoord(ra=sci_ra_col * u.deg, dec=sci_dec_col * u.deg, frame='icrs')
    sci_sep = pointing.separation(sci_pointing).to_value(u.deg)

    sun = get_sun(time)
    moon = get_body('moon', time, location=lco)
    sun_altaz = sun.transform_to(frame)
    moon_altaz = moon.transform_to(frame)

    sun_sep = pointing.separation(sun).to_value(u.deg)
    moon_sep = pointing.separation(moon).to_value(u.deg)

    alt_deg = pointing_altaz.alt.to_value(u.deg)
    sinalt = np.sin(np.deg2rad(alt_deg))
    airmass = np.where(sinalt > 1e-6, 1.0 / np.clip(sinalt, 1e-6, None), np.nan)

    az_rad = np.deg2rad(pointing_altaz.az.to_value(u.deg))
    moon_az_rad = np.deg2rad(moon_altaz.az.to_value(u.deg))
    sun_az_rad = np.deg2rad(sun_altaz.az.to_value(u.deg))

    return {
        'alt': alt_deg.astype(np.float32),
        'az_sin': np.sin(az_rad).astype(np.float32),
        'az_cos': np.cos(az_rad).astype(np.float32),
        'airmass': airmass.astype(np.float32),
        'moon_alt': moon_altaz.alt.to_value(u.deg).astype(np.float32),
        'moon_az_sin': np.sin(moon_az_rad).astype(np.float32),
        'moon_az_cos': np.cos(moon_az_rad).astype(np.float32),
        'moon_sep': moon_sep.astype(np.float32),
        'sun_alt': sun_altaz.alt.to_value(u.deg).astype(np.float32),
        'sun_az_sin': np.sin(sun_az_rad).astype(np.float32),
        'sun_az_cos': np.cos(sun_az_rad).astype(np.float32),
        'sun_sep': sun_sep.astype(np.float32),
        'sci_sep': sci_sep.astype(np.float32),
    }


# ---------------------------------------------------------------------------
# Solar activity proxies (F10.7 daily flux and Kp geomagnetic index).
#
# F10.7 is the standard EUV proxy driving thermospheric density and OI 6300 /
# OI 6364 recombination amplitude; Kp captures short-timescale auroral /
# geomagnetic driving that also modulates the 285 km group's brightness.
# Both are fetched from GFZ Potsdam's authoritative archive
#     https://kp.gfz.de/app/files/Kp_ap_Ap_SN_F107_since_1932.txt
# (definitive + quicklook, back to 1932, updated daily) and cached under
# solar_activity_cache/.  The raw file is re-downloaded if older than
# SOLAR_ACTIVITY_MAX_AGE_DAYS or missing.
# ---------------------------------------------------------------------------
SOLAR_ACTIVITY_FEATURES = {'f107', 'f107_81d', 'kp'}
SOLAR_ACTIVITY_SOURCE_URL = 'https://kp.gfz.de/app/files/Kp_ap_Ap_SN_F107_since_1932.txt'
SOLAR_ACTIVITY_CACHE_DIR = Path('solar_activity_cache')
SOLAR_ACTIVITY_RAW_PATH = SOLAR_ACTIVITY_CACHE_DIR / 'Kp_ap_Ap_SN_F107_since_1932.txt'
SOLAR_ACTIVITY_NPZ_PATH = SOLAR_ACTIVITY_CACHE_DIR / 'kp_f107.npz'
SOLAR_ACTIVITY_MAX_AGE_DAYS = 7

_SOLAR_ACTIVITY_TABLE = None  # module-level cache of parsed arrays


def _download_gfz_solar_activity(url, dest, timeout=60, verbose=True):
    """Fetch the GFZ Kp / ap / F10.7 archive to `dest` via a plain HTTP GET."""
    from urllib.request import Request, urlopen
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + '.part')
    req = Request(url, headers={'User-Agent': 'lvmsky-notebook/1.0'})
    with urlopen(req, timeout=timeout) as resp, open(tmp, 'wb') as fh:
        fh.write(resp.read())
    tmp.replace(dest)
    if verbose:
        print(f'  solar-activity archive downloaded: {dest} '
              f'({dest.stat().st_size / 1e6:.2f} MB)')


def _parse_gfz_solar_activity_raw(path):
    """Parse the GFZ text file. Returns dict of arrays: kp (3-hourly) + F10.7 (daily)."""
    df = pd.read_csv(path, comment='#', sep=r'\s+', header=None, engine='python')
    if df.shape[1] < 26:
        raise RuntimeError(
            f'Unexpected GFZ Kp/F10.7 file layout: got {df.shape[1]} columns, '
            f'expected at least 26. First data row: {df.iloc[0].tolist()!r}.'
        )
    year = df.iloc[:, 0].astype(int).to_numpy()
    month = df.iloc[:, 1].astype(int).to_numpy()
    day = df.iloc[:, 2].astype(int).to_numpy()
    # columns 3-6 = bookkeeping (days since 1932-01-01, Bartels rotation number, day of rotation)
    kp_cols = df.iloc[:, 7:15].astype(float).to_numpy()   # 8 three-hourly Kp values (Kp1..Kp8)
    # columns 15-22 = 8 ap values; 23 = Ap daily; 24 = SN; 25 = F10.7 obs; 26 = F10.7 adj
    f107_obs = df.iloc[:, 25].astype(float).to_numpy()

    from astropy.time import Time
    iso = [f'{y:04d}-{m:02d}-{d:02d}' for y, m, d in zip(year, month, day)]
    mjd_day0 = Time(iso, format='iso', scale='utc').mjd.astype(np.float64)
    # 3-hour Kp windows start at 0h, 3h, 6h, ..., 21h UT.
    hour_offsets = np.arange(8, dtype=np.float64) * (3.0 / 24.0)
    kp_mjd_start = (mjd_day0[:, None] + hour_offsets[None, :]).reshape(-1)
    kp_value = kp_cols.reshape(-1)
    # Daily F10.7 attached to noon UT so np.interp reads it as a midday sample.
    f107_mjd = mjd_day0 + 0.5
    good_kp = np.isfinite(kp_value) & (kp_value >= 0.0)
    good_f107 = np.isfinite(f107_obs) & (f107_obs > 0.0)
    return {
        'kp_mjd_start': kp_mjd_start[good_kp],
        'kp_value': kp_value[good_kp],
        'f107_mjd': f107_mjd[good_f107],
        'f107_obs': f107_obs[good_f107],
    }


def _f107_running_mean(mjd, f107, window_days=81.0):
    """Symmetric running mean of daily F10.7 over `window_days` (~3 solar rotations)."""
    order = np.argsort(mjd)
    mjd_s = mjd[order]
    f107_s = f107[order].astype(np.float64)
    n = mjd_s.size
    half = float(window_days) / 2.0
    lo = np.searchsorted(mjd_s, mjd_s - half, side='left')
    hi = np.searchsorted(mjd_s, mjd_s + half, side='right')
    csum = np.concatenate([[0.0], np.cumsum(f107_s)])
    sums = csum[hi] - csum[lo]
    counts = (hi - lo).astype(np.float64)
    with np.errstate(invalid='ignore', divide='ignore'):
        out_s = np.where(counts > 0, sums / counts, np.nan)
    inv = np.empty(n, dtype=int)
    inv[order] = np.arange(n)
    return out_s[inv]


def load_solar_activity_table(
    url=SOLAR_ACTIVITY_SOURCE_URL,
    raw_path=SOLAR_ACTIVITY_RAW_PATH,
    npz_path=SOLAR_ACTIVITY_NPZ_PATH,
    max_age_days=SOLAR_ACTIVITY_MAX_AGE_DAYS,
    force_refresh=False,
    verbose=True,
):
    """Return sorted numpy arrays for 3-hourly Kp and daily F10.7 (obs + 81-day mean).

    Downloads the GFZ archive if the raw cache is missing or older than
    `max_age_days` days.  Reuses a stale cache silently if the download fails.
    """
    global _SOLAR_ACTIVITY_TABLE
    if _SOLAR_ACTIVITY_TABLE is not None and not force_refresh:
        return _SOLAR_ACTIVITY_TABLE

    import time as _time_mod
    need_download = force_refresh or (not raw_path.exists())
    if not need_download:
        age_days = (_time_mod.time() - raw_path.stat().st_mtime) / 86400.0
        if age_days > float(max_age_days):
            if verbose:
                print(f'Solar-activity cache is {age_days:.1f} days old '
                      f'(> {max_age_days} d); refreshing.')
            need_download = True
    if need_download:
        try:
            _download_gfz_solar_activity(url, raw_path, verbose=verbose)
        except Exception as exc:
            if raw_path.exists():
                print(f'Solar-activity download failed '
                      f'({type(exc).__name__}: {exc}); reusing existing cache at {raw_path}.')
            else:
                raise RuntimeError(
                    f'Could not download solar-activity archive from {url} '
                    f'({type(exc).__name__}: {exc}); no local cache. '
                    f'Provide the file manually at {raw_path} to proceed.') from exc

    parsed = _parse_gfz_solar_activity_raw(raw_path)
    kp_order = np.argsort(parsed['kp_mjd_start'])
    kp_mjd_start = parsed['kp_mjd_start'][kp_order].astype(np.float64)
    kp_value = parsed['kp_value'][kp_order].astype(np.float32)
    f107_order = np.argsort(parsed['f107_mjd'])
    f107_mjd = parsed['f107_mjd'][f107_order].astype(np.float64)
    f107_obs = parsed['f107_obs'][f107_order].astype(np.float32)
    f107_81d = _f107_running_mean(f107_mjd, f107_obs, window_days=81.0).astype(np.float32)

    table = {
        'kp_mjd_start': kp_mjd_start,
        'kp_value': kp_value,
        'f107_mjd': f107_mjd,
        'f107_obs': f107_obs,
        'f107_81d': f107_81d,
    }
    npz_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(npz_path, **table)
    _SOLAR_ACTIVITY_TABLE = table
    if verbose:
        print(f'Solar-activity table ready: '
              f'{kp_mjd_start.size} Kp windows '
              f'(MJD {kp_mjd_start.min():.1f} .. {kp_mjd_start.max():.1f}), '
              f'{f107_mjd.size} daily F10.7 samples; cached to {npz_path}.')
    return table


def _solar_activity_lookup(mjd_query, feature):
    """Evaluate one solar-activity feature at each observation MJD."""
    table = load_solar_activity_table()
    mjd_query = np.asarray(mjd_query, dtype=np.float64)
    if feature == 'kp':
        # nearest 3-hour window that starts at or before the observation
        idx = np.searchsorted(table['kp_mjd_start'], mjd_query, side='right') - 1
        n_kp = table['kp_value'].size
        oob = (idx < 0) | (idx >= n_kp)
        idx = np.clip(idx, 0, n_kp - 1)
        val = table['kp_value'][idx].astype(np.float32).copy()
        val[oob] = np.nan
        return val
    if feature in ('f107', 'f107_81d'):
        src = table['f107_obs'] if feature == 'f107' else table['f107_81d']
        val = np.interp(mjd_query, table['f107_mjd'], src.astype(np.float64),
                        left=np.nan, right=np.nan)
        return val.astype(np.float32)
    raise KeyError(f'unknown solar-activity feature: {feature!r}')


def _decode_cyclic_context(ctx_names, ctx_matrix):
    """Fold `<name>_sin`/`<name>_cos` pairs back into a single `<name>` in degrees [0, 360).

    Purely for display: the model still consumes the sin/cos pair, but plots
    (histograms, coefficient-vs-context relationship panels) are much more
    legible with a single 0-360 axis than with a pair of [-1, 1] projections.
    """
    names = list(ctx_names)
    mat = np.asarray(ctx_matrix, dtype=np.float64)
    out_names = []
    out_cols = []
    handled = set()
    for i, name in enumerate(names):
        if name in handled:
            continue
        if name.endswith('_sin'):
            stem = name[:-4]
            cos_name = stem + '_cos'
            if cos_name in names:
                cos_idx = names.index(cos_name)
                rad = np.arctan2(mat[:, i], mat[:, cos_idx])
                out_names.append(stem)
                out_cols.append((np.rad2deg(rad) % 360.0).astype(np.float32))
                handled.add(name)
                handled.add(cos_name)
                continue
        if name.endswith('_cos') and name in handled:
            continue
        out_names.append(name)
        out_cols.append(mat[:, i].astype(np.float32))
    return out_names, (np.column_stack(out_cols).astype(np.float32)
                       if out_cols else np.empty((mat.shape[0], 0), dtype=np.float32))


def _resolve_base_context_array(meta, meta_upper, labels, kind, base_name):
    key = base_name.upper()

    if kind == 'sci':
        sci_key = f'SCI_{key}'
        if sci_key in meta_upper:
            arr = _as_array(meta[meta_upper[sci_key]])
            if arr is None:
                raise ValueError(f'Context column {sci_key} is non-numeric.')
            return arr

    if key in meta_upper:
        arr = _as_array(meta[meta_upper[key]])
        if arr is None:
            raise ValueError(f'Context column {base_name} is non-numeric.')
        return arr

    if labels is not None:
        arr = _select_context_from_labels(meta, meta_upper, labels, base_name)
        if arr is not None:
            return arr

    return None


def _resolve_context_feature(meta, meta_upper, labels, kind, feature_name, cache):
    if feature_name in cache:
        return cache[feature_name]

    if feature_name in ('obstime_mjd', 'obstime_mjd_z'):
        raise ValueError('Direct time features are excluded from model context; use only periodic year/day/lunation sine and cosine terms.')

    if feature_name in ASTROPY_GEOMETRY_FEATURES:
        # Compute all astropy features in one pass the first time any is asked for.
        if '_astropy_computed' not in cache:
            for _name, _arr in _compute_astropy_geometry(meta, meta_upper, kind).items():
                cache[_name] = np.asarray(_arr, dtype=np.float32)
            cache['_astropy_computed'] = True
        return cache[feature_name]

    if feature_name in SOLAR_ACTIVITY_FEATURES:
        if '_solar_activity_computed' not in cache:
            _sa_mjd = _extract_obstime_mjd(meta, meta_upper)
            for _sa_name in SOLAR_ACTIVITY_FEATURES:
                cache[_sa_name] = np.asarray(
                    _solar_activity_lookup(_sa_mjd, _sa_name), dtype=np.float32)
            cache['_solar_activity_computed'] = True
        return cache[feature_name]

    if feature_name == 'ew':
        if kind == 'sci':
            arr = np.zeros(len(meta), dtype=np.float32)
        else:
            arr = np.where(labels == 'SKYE', 1.0,
                           np.where(labels == 'SKYW', -1.0, 0.0)).astype(np.float32)
        cache[feature_name] = arr
        return arr

    if feature_name in TIME_FEATURE_NAMES:
        arr = _build_obstime_feature(meta, meta_upper, feature_name)
    elif feature_name == 'zenith_deg':
        alt = _resolve_context_feature(meta, meta_upper, labels, kind, 'alt', cache)
        arr = _altitude_to_zenith_deg(alt).astype(np.float32)
    elif (feature_name.endswith('_sin') or feature_name.endswith('_cos')) \
            and feature_name[:-4] in CYCLIC_META_DEGREE_FEATURES:
        raw = _resolve_base_context_array(meta, meta_upper, labels, kind, feature_name[:-4])
        if raw is None:
            raise KeyError(feature_name)
        rad = np.deg2rad(np.asarray(raw, dtype=np.float64))
        arr = (np.sin(rad) if feature_name.endswith('_sin') else np.cos(rad)).astype(np.float32)
    elif feature_name in VAN_RHIJN_FEATURES:
        alt = _resolve_context_feature(meta, meta_upper, labels, kind, 'alt', cache)
        arr = _van_rhijn_factor(alt, VAN_RHIJN_FEATURES[feature_name])
    else:
        arr = _resolve_base_context_array(meta, meta_upper, labels, kind, feature_name)
        if arr is None:
            raise KeyError(feature_name)

    cache[feature_name] = np.asarray(arr, dtype=np.float32)
    return cache[feature_name]


def _build_context_matrix(meta, context_columns, kind):
    meta_upper = {c.upper(): c for c in meta.colnames}
    labels = None

    if kind in ('sky1', 'sky2'):
        label_col = 'SKY_NEAR_LABEL' if kind == 'sky1' else 'SKY_FAR_LABEL'
        if label_col not in meta_upper:
            raise KeyError(f'Missing required META label column: {label_col}')
        labels = np.char.upper(np.char.strip(np.asarray(meta[meta_upper[label_col]]).astype(str)))

    ctx_names = []
    ctx_cols = []
    missing_cols = []
    cache = {}

    for cname in context_columns:
        try:
            arr = _resolve_context_feature(meta, meta_upper, labels, kind, cname, cache)
        except KeyError:
            missing_cols.append(cname)
            continue
        ctx_names.append(cname)
        ctx_cols.append(arr)

    if missing_cols:
        raise KeyError(f'Missing requested context columns: {missing_cols}')
    if len(ctx_cols) == 0:
        raise ValueError('No usable context columns were assembled.')

    return np.column_stack(ctx_cols).astype(np.float32), ctx_names


def _find_chi2_column(meta_tbl):
    names = {c.upper(): c for c in meta_tbl.colnames}
    for cand in ('REDUCED_CHI2', 'CHI2_REDUCED', 'CHI2', 'RCHI2'):
        if cand in names:
            return names[cand]
    raise KeyError('No chi2-like column found in decomposition META table')


def read_decomp_dataset(decomp_fits_path, input_fits_path, context_columns,
                        decomp_kind='sky1', return_chi2=False, return_err=False):
    if context_columns is None or len(context_columns) == 0:
        raise ValueError('context_columns must be a non-empty list.')

    kind = decomp_kind.lower()
    if kind not in ('sky1', 'sky2', 'sci'):
        raise ValueError("decomp_kind must be one of: 'sky1', 'sky2', 'sci'")

    with fits.open(decomp_fits_path) as hdul_dec, fits.open(input_fits_path) as hdul_in:
        coef_tbl = _coerce_coef_hdu_to_table(hdul_dec['COEF'])
        coef_mat, coef_names = _table_to_float32_matrix(coef_tbl, 'coefficient')

        # COEF_ERR is the active-set 1-sigma uncertainty on each coefficient
        # (see fit.SkyDecomp._coef_err_active_set); older FITS files omit it,
        # so we fall back to NaN and callers can decide whether to weight.
        coef_err_mat = None
        if return_err:
            if 'COEF_ERR' in hdul_dec:
                coef_err_tbl = _coerce_coef_hdu_to_table(hdul_dec['COEF_ERR'])
                coef_err_mat, coef_err_names = _table_to_float32_matrix(coef_err_tbl, 'coefficient error')
                if coef_err_names != coef_names:
                    raise ValueError('COEF_ERR column ordering does not match COEF')
            else:
                coef_err_mat = np.full_like(coef_mat, np.nan, dtype=np.float32)

        meta = Table(hdul_in['META'].data)
        ctx_mat, ctx_names = _build_context_matrix(meta, context_columns, kind)

        if coef_mat.shape[0] != ctx_mat.shape[0]:
            raise ValueError(
                f'Row count mismatch: COEF has {coef_mat.shape[0]} rows, META has {ctx_mat.shape[0]} rows'
            )

        good = np.isfinite(coef_mat).all(axis=1) & np.isfinite(ctx_mat).all(axis=1)
        coef_mat = coef_mat[good]
        ctx_mat = ctx_mat[good]
        if coef_err_mat is not None:
            coef_err_mat = coef_err_mat[good]

        if not return_chi2 and not return_err:
            return coef_mat, ctx_mat, coef_names, ctx_names

        chi2_used = None
        if return_chi2:
            dec_meta = Table(hdul_dec['META'].data)
            chi2_col = _find_chi2_column(dec_meta)
            chi2_full = np.asarray(dec_meta[chi2_col], dtype=np.float64)
            chi2_used = chi2_full[good]
            if chi2_used.shape[0] != coef_mat.shape[0]:
                raise ValueError(
                    f'Aligned chi2 rows ({chi2_used.shape[0]}) do not match coef rows ({coef_mat.shape[0]})'
                )

        result = [coef_mat, ctx_mat, coef_names, ctx_names]
        if return_chi2:
            result.append(chi2_used)
        if return_err:
            result.append(coef_err_mat)
        return tuple(result)


# ---------------------------------------------------------------------------
# Coefficient grouping by physical origin / emission layer.
# Single definition, used by the wavelength resolver, the structure-function
# diagnostics and the model.  Coefficients are consumed in order so each one
# lands in exactly one group.
# ---------------------------------------------------------------------------

def _contains_any(name, needles):
    return any(token in name for token in needles)


# Coefficient-name schema. One-to-one with the design_names produced by
# SkyDecomp._build_static_basis() in skysub/sky_decomp/fit.py. Every entry
# is (regex, group, wavelength_a, description). Any coefficient that does
# not match a pattern raises in _build_group_indices() -- silently
# absorbing unknown names into an 'other' group previously hid that
# HO2/FeO/O2Ac were routed into 'mesospheric' and that OI 6300/6364 +
# OI 7774/8446 (ATOM_Or, ATOM_Orc_*) were routed into 'atomic' (~95 km)
# instead of 'ionospheric' (~285 km).
#
# Group layers (see GROUP_HEIGHT_FEATURE):
#   moon         no thin-shell height (scattered moonlight + spline)
#   continuum    ~87 km  HO2 / FeO / O2Ac mesopause chemiluminescence
#   mesospheric  ~87 km  OH Meinel bands + O2 atmospheric band
#   atomic       ~95 km  K, Na, N I, OI 5577 (mesopause metal / metastable)
#   ionospheric  ~285 km OI 6300/6364 red + OI 7774/8446 F-region recomb
COEF_SCHEMA = [
    (re.compile(r'^OH_\d{3}$'),         'mesospheric', None,   'OH Meinel band group (wavelength from basis)'),
    (re.compile(r'^O2_b\d+$'),          'mesospheric', 8645.0, 'O2 (b1Sigma) atmospheric band'),
    (re.compile(r'^Moon_bs\d+$'),       'moon',        None,   'Moon scattered-light spline knot'),
    (re.compile(r'^MoonZodi_bs\d+$'),   'moon',        None,   'Moon+Zodi physical predictor spline correction'),
    (re.compile(r'^HO2$'),              'continuum',   None,   'HO2 diffuse continuum'),
    (re.compile(r'^FeO$'),              'continuum',   None,   'FeO orange-arc diffuse continuum'),
    (re.compile(r'^O2Ac$'),             'continuum',   None,   'O2 Chamberlain diffuse continuum'),
    (re.compile(r'^ATOM_K$'),           'atomic',      7698.96, 'K I 7699'),
    (re.compile(r'^ATOM_N$'),           'ionospheric', 5199.0,  'N I 5199 [NI] 2D->4S F-region metastable'),
    (re.compile(r'^ATOM_Na$'),          'atomic',      5892.9,  'Na D'),
    (re.compile(r'^ATOM_Og$'),          'atomic',      5577.34, 'OI 5577 green line'),
    (re.compile(r'^ATOM_Or$'),          'ionospheric', 6300.3,  'OI 6300/6364 red doublet'),
    (re.compile(r'^ATOM_Orc_OI0777$'),  'ionospheric', 7774.2,  'OI 7774 F-region recombination'),
    (re.compile(r'^ATOM_Orc_OI0845$'),  'ionospheric', 8446.4,  'OI 8446 F-region recombination'),
]

# Canonical output order for group dicts.
_COEF_GROUP_ORDER = ('moon', 'continuum', 'mesospheric', 'ionospheric', 'atomic')


def _lookup_coef_schema(name):
    """Return (group, wavelength_a, description) for one coefficient, else None."""
    for pat, group, lam, desc in COEF_SCHEMA:
        if pat.match(str(name)):
            return group, lam, desc
    return None


def _build_group_indices(coef_names):
    """Route each coefficient name to exactly one group via COEF_SCHEMA.

    Raises RuntimeError on any unmatched name so a new coefficient family
    added to fit.py becomes a loud error here rather than a silent 'other'.
    """
    groups = {g: [] for g in _COEF_GROUP_ORDER}
    unmatched = []
    for j, name in enumerate(coef_names):
        hit = _lookup_coef_schema(name)
        if hit is None:
            unmatched.append((j, str(name)))
        else:
            groups[hit[0]].append(j)
    if unmatched:
        raise RuntimeError(
            'Unrecognised coefficient names (no COEF_SCHEMA match): '
            f'{unmatched[:8]}{"..." if len(unmatched) > 8 else ""}. '
            'Extend COEF_SCHEMA in the loaders cell and cross-check '
            'against SkyDecomp._build_static_basis() in '
            'skysub/sky_decomp/fit.py.'
        )
    return {g: np.asarray(idx, dtype=int) for g, idx in groups.items() if idx}


# Self-test: verify the schema on a canonical name list. Runs when the
# cell is executed, so a regression in COEF_SCHEMA fails immediately.
_SCHEMA_TEST_NAMES = (
    [f'OH_{i:03d}' for i in (0, 42, 401)]
    + [f'Moon_bs{i:02d}' for i in (0, 15, 28)]
    + [f'MoonZodi_bs{i:03d}' for i in (0, 12, 28)]
    + ['HO2', 'FeO', 'O2Ac']
    + ['ATOM_K', 'ATOM_N', 'ATOM_Na', 'ATOM_Og', 'ATOM_Or',
       'ATOM_Orc_OI0777', 'ATOM_Orc_OI0845']
    + ['O2_b01']
)
_SCHEMA_TEST_EXPECTED = {
    'moon': 6,           # 3x Moon_bs + 3x MoonZodi_bs
    'continuum': 3,
    'mesospheric': 4,    # 3x OH_### + O2_b01
    'atomic': 3,         # K, Na, Og
    'ionospheric': 4,    # N, Or, Orc_OI0777, Orc_OI0845
}
_schema_test = _build_group_indices(_SCHEMA_TEST_NAMES)
_schema_test_sizes = {g: int(idx.size) for g, idx in _schema_test.items()}
if _schema_test_sizes != _SCHEMA_TEST_EXPECTED:
    raise RuntimeError(
        f'COEF_SCHEMA self-test failed: got {_schema_test_sizes}, '
        f'expected {_SCHEMA_TEST_EXPECTED}'
    )
try:
    _build_group_indices(['this_is_not_a_valid_coef'])
except RuntimeError:
    pass
else:
    raise RuntimeError('COEF_SCHEMA self-test: unknown name did not raise')

# ---------------------------------------------------------------------------
# Geometric normalisation of airglow coefficients  (PHYSICAL units)
#
# Airglow amplitudes scale as
#       A_obs = A_intrinsic * V(z; h) * 10**(-0.4 * k(lambda) * (X - 1))
# where V is the van Rhijn slant-path enhancement through a thin emitting
# shell at height h, and the power of ten is the extinction suffered by that
# emission on the way down.  Dividing an observed amplitude by this factor
# recovers the intrinsic layer emissivity, which is the only quantity
# comparable between two different lines of sight.
#
# The normalisation therefore has to happen HERE, on physical coefficients,
# before the sqrt transform and before any robust scaling.  Doing it on
# scaled or sqrt-transformed values is not equivalent: the scaler subtracts a
# median, so (sqrt(c) - m) / V is not sqrt(c / V) - m, and the sqrt means the
# correct divisor would be sqrt(V) rather than V.
# ---------------------------------------------------------------------------

# HO2 (H+O2+M chemiluminescence, ~87 km), FeO (Fe+O3 orange arc, ~85-90 km)
# and O2Ac (O+O+M -> O2 Chamberlain bands, ~90-95 km) are mesopause
# thin-shell emitters, not aerosol/Rayleigh continuum -- their sky-to-sci
# transfer needs the same van Rhijn factor as the OH bands. They are kept
# in a separate coefficient GROUP (broadband basis vs OH line groups, so
# their compression pipeline stays sqrt-identity with no PCA) but the
# geometric normalisation is now shared with mesospheric at 87 km.
AIRGLOW_GROUPS = {'mesospheric', 'atomic', 'ionospheric', 'continuum'}

GROUP_HEIGHT_FEATURE = {
    'mesospheric': 'vanrhijn_87km',
    'atomic': 'vanrhijn_95km',
    'ionospheric': 'vanrhijn_285km',
    'continuum': 'vanrhijn_87km',
}

# Only used as a last-resort fallback when a per-coefficient wavelength cannot
# be determined; see resolve_coef_wavelengths_a().
GROUP_EFFECTIVE_WAVELENGTH_A = {
    'mesospheric': 8500.0,
    'continuum': 7500.0,   # HO2/FeO/O2Ac mid-band; basis route usually resolves per-coefficient
    'atomic': 5750.0,
    'ionospheric': 6300.0,
}


def _group_height_feature_name(group_name):
    """van Rhijn context feature (hence effective layer height) for a group.

    mesospheric -> 87 km   (OH Meinel bands + O2 atmospheric band)
    continuum   -> 87 km   (HO2 / FeO / O2Ac mesopause chemiluminescence)
    atomic      -> 95 km   (K / Na / N I / OI 5577 mesopause metal / metastable)
    ionospheric -> 285 km  (OI 6300/6364 red + OI 7774/8446 F-region recomb)
    moon / other -> None (not thin-shell emission)
    """
    return GROUP_HEIGHT_FEATURE.get(group_name)


def _load_lco_extinction_curve():
    lvmcore_dir = os.environ.get('LVMCORE_DIR')
    if not lvmcore_dir:
        raise EnvironmentError('LVMCORE_DIR is not set; cannot load lco_extinction.txt')
    curve_path = Path(lvmcore_dir) / 'etc' / 'lco_extinction.txt'
    curve = np.loadtxt(curve_path, dtype=np.float64)
    if curve.ndim != 2 or curve.shape[1] < 2:
        raise ValueError(f'Unexpected extinction-curve shape: {curve.shape}')
    wave_a = np.asarray(curve[:, 0], dtype=np.float64)
    ext_k = np.asarray(curve[:, 1], dtype=np.float64)
    order = np.argsort(wave_a)
    return wave_a[order], ext_k[order]


LCO_EXTINCTION_WAVE_A, LCO_EXTINCTION_K = _load_lco_extinction_curve()


def _lco_extinction_k(wavelength_a):
    """Extinction coefficient (mag/airmass) at one or many wavelengths (Angstrom)."""
    lam = np.asarray(wavelength_a, dtype=np.float64)
    k = np.interp(
        lam,
        LCO_EXTINCTION_WAVE_A,
        LCO_EXTINCTION_K,
        left=float(LCO_EXTINCTION_K[0]),
        right=float(LCO_EXTINCTION_K[-1]),
    )
    return float(k) if np.ndim(wavelength_a) == 0 else np.asarray(k, dtype=np.float64)


# --- per-coefficient effective wavelength -----------------------------------
#
# Extinction varies strongly across the LVM range (k ~ 0.11 at 5577 A vs
# ~0.02 in the z band), so a single wavelength per group is not good enough:
# the 'mesospheric' group alone spans OI 5577, Na D and the OH/O2 bands.
#
# The primary wavelength source is coef_wavelengths_from_basis (reads the
# reconstructed basis directly). The name-based fallback below consults
# COEF_SCHEMA only, so new coefficient names cannot silently pick up a
# default guessed from digit tokens or species substrings.


def _wavelength_from_name(name):
    """Wavelength (Angstrom) from COEF_SCHEMA, else NaN."""
    hit = _lookup_coef_schema(name)
    if hit is None:
        return float('nan')
    _group, lam, _desc = hit
    return float(lam) if lam is not None and np.isfinite(lam) else float('nan')


def coef_wavelengths_from_basis(
    coef_names,
    wave,
    lsf_sigma,
    base_dir=None,
    n_spline_knots=25,
    cache_path=None,
    only_indices=None,
    verbose=True,
    return_k_eff=False,
):
    """Per-coefficient wavelength centroid and B^2-weighted effective extinction.

    Exploits linearity of the decomposition: reconstructing with coef = e_j
    returns basis component j on its own, so we read two scalars per
    coefficient in one reconstruction pass:

      1. INTENSITY-WEIGHTED CENTROID (reporting / binning):
             lambda_eff(j) = sum(lambda |f_j|) / sum(|f_j|)

      2. B^2-WEIGHTED EFFECTIVE EXTINCTION under the LCO stellar curve:
             k_eff(j) = sum(|f_j|^2 k(lambda)) / sum(|f_j|^2)
         For a broadband basis component the least-squares decomposition
         amplitude picks up the B^2-weighted average of any wavelength-dependent
         factor multiplying the true emissivity, so <k>_{B^2} is the correct
         effective extinction, not k(centroid). For narrow-line components
         |f_j| is sharply peaked and both weightings agree; the correction only
         matters for HO2 / FeO / O2Ac (continuum group) and any other broadband
         basis functions.

    Costs one reconstruction call per coefficient. Both scalars are cached to
    `cache_path` (npz keys `wavelengths_a`, `k_eff_a`); an older cache without
    `k_eff_a` is auto-invalidated and recomputed. Components that reconstruct
    to zero (outside the wavelength range) leave NaN in both arrays and fall
    back to name parsing / group defaults downstream.

    Returns `wavelengths_a` by default, or `(wavelengths_a, k_eff_a)` when
    `return_k_eff=True`.
    """
    coef_names = [str(n) for n in coef_names]
    n_coef = len(coef_names)
    wave = np.asarray(wave, dtype=np.float64)

    # The cache key must include the wavelength grid: the same coefficient
    # names evaluated on a different grid give different centroids, so keying
    # on names alone would silently return stale values after a change of input
    # product.
    grid_key = float(np.nansum(wave.astype(np.float64) * np.arange(1, wave.size + 1)))
    if cache_path is not None:
        cache_path = Path(cache_path)
        if cache_path.exists():
            cached = np.load(cache_path, allow_pickle=True)
            same_names = [str(x) for x in cached['coef_names']] == coef_names
            same_grid = ('grid_key' in cached.files
                         and np.isclose(float(cached['grid_key']), grid_key, rtol=1e-12))
            has_k_eff = 'k_eff_a' in cached.files
            if same_names and same_grid and has_k_eff:
                if verbose:
                    print(f'Basis wavelengths + k_eff loaded from cache: {cache_path}')
                lam_cached = np.asarray(cached['wavelengths_a'], dtype=np.float64)
                k_cached = np.asarray(cached['k_eff_a'], dtype=np.float64)
                return (lam_cached, k_cached) if return_k_eff else lam_cached
            if verbose:
                if not same_names:
                    reason = 'coefficient names'
                elif not same_grid:
                    reason = 'wavelength grid'
                else:
                    reason = 'missing k_eff_a (older cache format)'
                print(f'Cache {cache_path} does not match on {reason}; recomputing.')

    if base_dir is None:
        if '_infer_base_dir_for_reconstruction' not in globals():
            raise RuntimeError(
                'base_dir not given and _infer_base_dir_for_reconstruction() is '
                'not defined yet; run the reconstruction-helpers cell first or '
                'pass base_dir= explicitly.'
            )
        base_dir = _infer_base_dir_for_reconstruction()

    if only_indices is None:
        targets = np.arange(n_coef)
    else:
        targets = np.unique(np.asarray(only_indices, dtype=int))
        if verbose:
            print(f'Basis wavelengths: evaluating {targets.size} of {n_coef} '
                  f'coefficients (one reconstruction call each).')

    k_wave = np.asarray(_lco_extinction_k(wave), dtype=np.float64)
    lam_eff = np.full(n_coef, np.nan, dtype=np.float64)
    k_eff = np.full(n_coef, np.nan, dtype=np.float64)
    for j in targets:
        unit = np.zeros(n_coef, dtype=np.float64)
        unit[j] = 1.0
        comps = reconstruct_component_spectra(
            wave=wave,
            coef=unit,
            lsf_sigma=lsf_sigma,
            n_spline_knots=n_spline_knots,
            base_dir=base_dir,
        )
        f = np.abs(np.asarray(comps['total'], dtype=np.float64))
        tot = float(np.nansum(f))
        if np.isfinite(tot) and tot > 0.0:
            lam_eff[j] = float(np.nansum(wave * f) / tot)
            f2 = f * f
            f2_sum = float(np.nansum(f2))
            if np.isfinite(f2_sum) and f2_sum > 0.0:
                k_eff[j] = float(np.nansum(f2 * k_wave) / f2_sum)

    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez(cache_path, coef_names=np.asarray(coef_names),
                 wavelengths_a=lam_eff, k_eff_a=k_eff,
                 grid_key=np.float64(grid_key))
        if verbose:
            print(f'Basis wavelengths + k_eff cached to: {cache_path}')

    if verbose:
        n_ok = int(np.isfinite(lam_eff).sum())
        n_ok_k = int(np.isfinite(k_eff).sum())
        print(f'Basis wavelengths derived for {n_ok}/{n_coef} coefficients; '
              f'basis B^2-weighted extinction k for {n_ok_k}/{n_coef}.')
    return (lam_eff, k_eff) if return_k_eff else lam_eff


def resolve_coef_wavelengths_a(
    coef_names,
    group_indices=None,
    basis_wavelengths_a=None,
    verbose=True,
):
    """Per-coefficient effective wavelength (Angstrom) plus its provenance.

    Priority, per coefficient:
      1. basis centroid from coef_wavelengths_from_basis(), where finite --
         reads the reconstructed basis directly, so it already carries the
         PMD-driven populations (including the mesopause rotational Boltzmann
         factor for OH), the LSF, and the blend structure across overlapping
         bands. This is the definitionally-correct weighting.
      2. explicit wavelength token or species lookup baked into COEF_SCHEMA.
      3. group default from GROUP_EFFECTIVE_WAVELENGTH_A.

    Returns (wavelengths_a, sources).  The printed table is deliberately
    verbose for the airglow groups: a silently wrong wavelength turns into a
    silently wrong extinction correction.
    """
    coef_names = [str(n) for n in coef_names]
    n_coef = len(coef_names)
    lam = np.full(n_coef, np.nan, dtype=np.float64)
    source = ['unset'] * n_coef

    if basis_wavelengths_a is not None:
        basis = np.asarray(basis_wavelengths_a, dtype=np.float64)
        if basis.size != n_coef:
            raise ValueError(
                f'basis_wavelengths_a has {basis.size} entries, expected {n_coef}'
            )
        ok = np.isfinite(basis) & (basis > 0.0)
        lam[ok] = basis[ok]
        for j in np.flatnonzero(ok):
            source[j] = 'basis'

    for j in range(n_coef):
        if np.isfinite(lam[j]):
            continue
        cand = _wavelength_from_name(coef_names[j])
        if np.isfinite(cand):
            lam[j] = cand
            source[j] = 'name'

    group_of = {}
    if group_indices is not None:
        for g, idx in group_indices.items():
            for j in np.asarray(idx, dtype=int):
                group_of[int(j)] = g
        for g, idx in group_indices.items():
            default = GROUP_EFFECTIVE_WAVELENGTH_A.get(g)
            if default is None:
                continue
            for j in np.asarray(idx, dtype=int):
                if not np.isfinite(lam[int(j)]):
                    lam[int(j)] = float(default)
                    source[int(j)] = f'group:{g}'

    if verbose:
        airglow_idx = [j for j in range(n_coef)
                       if group_of.get(j) in AIRGLOW_GROUPS]
        print(f'Per-coefficient wavelengths resolved for {n_coef} coefficients '
              f'({len(airglow_idx)} in airglow groups, which are the ones that '
              f'receive a geometry correction).')
        if airglow_idx:
            print(f"  {'coefficient':<28s} {'group':<13s} {'lambda[A]':>10s} "
                  f"{'k[mag/X]':>9s}  source")
            for j in airglow_idx:
                kj = _lco_extinction_k(lam[j]) if np.isfinite(lam[j]) else np.nan
                print(f'  {coef_names[j]:<28s} {group_of.get(j, "-"):<13s} '
                      f'{lam[j]:>10.1f} {kj:>9.4f}  {source[j]}')
            from collections import Counter
            print('  wavelength provenance:',
                  dict(Counter(source[j] for j in airglow_idx)))

    return lam, source


def assert_context_is_physical(ctx, ctx_names, rtol=1e-4, atol=1e-4):
    """Fail loudly if scaler-normalised context reaches the geometry code.

    Two independent checks:
      1. physical ranges (alt within +/-90 deg, airmass >= 1)
      2. any vanrhijn_* column must agree with recomputing van Rhijn from the
         'alt' column.  Those two are computed independently upstream, so
         disagreement means the context has been transformed on the way in.

    This is the regression test for the class of bug where geometry is
    evaluated on RobustScaler output: a scaled altitude of ~0.5 becomes a
    zenith angle of ~89.5 deg and every row gets a near-horizon van Rhijn
    factor of ~6.
    """
    ctx = np.asarray(ctx, dtype=np.float64)
    names = [str(n).strip().lower() for n in ctx_names]

    if ctx.ndim != 2:
        raise ValueError(f'context must be 2-D, got shape {ctx.shape}')
    if ctx.shape[1] != len(names):
        raise ValueError(
            f'context has {ctx.shape[1]} columns but {len(names)} names given'
        )
    for required in ('alt', 'airmass'):
        if required not in names:
            raise KeyError(
                f"geometry normalisation needs '{required}' among the context columns"
            )

    alt = ctx[:, names.index('alt')]
    airmass = ctx[:, names.index('airmass')]

    if np.nanmin(alt) < -90.0 or np.nanmax(alt) > 90.0:
        raise ValueError(
            f'alt outside [-90, 90] deg (min={np.nanmin(alt):.4g}, '
            f'max={np.nanmax(alt):.4g}); context looks scaled, not physical'
        )
    if np.nanmin(airmass) < 0.99:
        raise ValueError(
            f'airmass below 1 (min={np.nanmin(airmass):.4g}); '
            f'context looks scaled, not physical'
        )

    for feat, height_km in VAN_RHIJN_FEATURES.items():
        if feat not in names:
            continue
        stored = ctx[:, names.index(feat)]
        recomputed = np.asarray(_van_rhijn_factor(alt, float(height_km)), dtype=np.float64)
        if not np.allclose(stored, recomputed, rtol=rtol, atol=atol, equal_nan=True):
            worst = float(np.nanmax(np.abs(stored - recomputed)))
            raise ValueError(
                f"context column '{feat}' disagrees with van Rhijn recomputed "
                f"from 'alt' (max abs diff {worst:.4g}); context is not physical"
            )

    return True


def _airglow_ctx_columns(ctx_phys, ctx_names, check_physical=True):
    """Validate context and return (ctx, names, alt, airmass) in physical units."""
    ctx_phys = np.asarray(ctx_phys, dtype=np.float64)
    if ctx_phys.ndim == 1:
        ctx_phys = ctx_phys[None, :]
    names = [str(n).strip().lower() for n in ctx_names]
    if check_physical:
        assert_context_is_physical(ctx_phys, names)
    return (ctx_phys, names,
            ctx_phys[:, names.index('alt')],
            ctx_phys[:, names.index('airmass')])


def _coef_wavelengths_with_fallback(coef_wavelengths_a, group_indices, n_coef):
    """Per-coefficient wavelength array with group defaults filled in."""
    n_coef = int(n_coef)
    if coef_wavelengths_a is None:
        lam = np.full(n_coef, np.nan, dtype=np.float64)
    else:
        lam = np.asarray(coef_wavelengths_a, dtype=np.float64).astype(np.float64).copy()
        if lam.size != n_coef:
            raise ValueError(
                f'coef_wavelengths_a has {lam.size} entries, expected {n_coef}'
            )
    for group_name, idx in group_indices.items():
        fallback = GROUP_EFFECTIVE_WAVELENGTH_A.get(group_name)
        if fallback is None:
            continue
        idx = np.asarray(idx, dtype=int)
        if idx.size == 0:
            continue
        bad = ~(np.isfinite(lam[idx]) & (lam[idx] > 0.0))
        lam[idx[bad]] = float(fallback)
    return lam


def airglow_coef_extinction_k(coef_wavelengths_a, group_indices, n_coef,
                              coef_extinction_k=None):
    """Per-coefficient extinction coefficient (mag/airmass).

    If `coef_extinction_k` is supplied it is used verbatim -- that is the hook
    for the empirically fitted effective extinction (see
    fit_effective_extinction).  Otherwise the generic LCO stellar curve is
    interpolated at each coefficient's effective wavelength.
    """
    n_coef = int(n_coef)
    if coef_extinction_k is not None:
        k = np.asarray(coef_extinction_k, dtype=np.float64)
        if k.size != n_coef:
            raise ValueError(
                f'coef_extinction_k has {k.size} entries, expected {n_coef}'
            )
        return np.where(np.isfinite(k), k, 0.0)

    lam = _coef_wavelengths_with_fallback(coef_wavelengths_a, group_indices, n_coef)
    k = np.zeros(n_coef, dtype=np.float64)
    ok = np.isfinite(lam) & (lam > 0.0)
    if ok.any():
        k[ok] = np.atleast_1d(_lco_extinction_k(lam[ok])).astype(np.float64)
    return k


def airglow_van_rhijn_matrix(ctx_phys, ctx_names, group_indices, n_coef,
                             check_physical=True):
    """V(z; h) per row and coefficient; exactly 1.0 for non-airglow coefficients."""
    ctx_phys, _names, alt, _airmass = _airglow_ctx_columns(
        ctx_phys, ctx_names, check_physical)
    n_coef = int(n_coef)
    van_rhijn = np.ones((ctx_phys.shape[0], n_coef), dtype=np.float64)

    for group_name, idx in group_indices.items():
        if group_name not in AIRGLOW_GROUPS:
            continue
        feature_name = _group_height_feature_name(group_name)
        if feature_name is None or feature_name not in VAN_RHIJN_FEATURES:
            continue
        idx = np.asarray(idx, dtype=int)
        if idx.size == 0:
            continue
        height_km = float(VAN_RHIJN_FEATURES[feature_name])
        van_rhijn[:, idx] = np.asarray(
            _van_rhijn_factor(alt, height_km), dtype=np.float64)[:, None]

    return van_rhijn


def airglow_extinction_matrix(ctx_phys, ctx_names, group_indices, n_coef,
                              coef_wavelengths_a=None, coef_extinction_k=None,
                              check_physical=True):
    """10**(-0.4 k (X - 1)) per row and coefficient; 1.0 for non-airglow.

    Normalised to X = 1 rather than X = 0, so what this removes is the
    extinction *relative to zenith*.  The recovered quantity is therefore a
    zenith-equivalent amplitude, not an absolute layer emissivity.  That is
    self-consistent and cancels in the sky-to-science transfer.
    """
    ctx_phys, _names, _alt, airmass = _airglow_ctx_columns(
        ctx_phys, ctx_names, check_physical)
    n_coef = int(n_coef)
    k_all = airglow_coef_extinction_k(
        coef_wavelengths_a, group_indices, n_coef, coef_extinction_k)

    extinction = np.ones((ctx_phys.shape[0], n_coef), dtype=np.float64)
    for group_name, idx in group_indices.items():
        if group_name not in AIRGLOW_GROUPS:
            continue
        idx = np.asarray(idx, dtype=int)
        if idx.size == 0:
            continue
        extinction[:, idx] = 10.0 ** (
            -0.4 * k_all[idx][None, :] * (airmass[:, None] - 1.0))

    return extinction


def airglow_geometry_scale(
    ctx_phys,
    ctx_names,
    group_indices,
    n_coef,
    coef_wavelengths_a=None,
    coef_extinction_k=None,
    check_physical=True,
):
    """Geometry factor V(z; h) * 10**(-0.4 k (X - 1)) per row and coefficient.

    Returns an (n_rows, n_coef) array that is exactly 1.0 for every
    non-airglow coefficient (moon, other), since scattered moonlight needs
    scattering geometry rather than van Rhijn. The continuum group (HO2,
    FeO, O2Ac) IS airglow -- it's mesopause chemiluminescence, not aerosol
    or Rayleigh scattering -- and is treated at 87 km, same as OH.

    ctx_phys must be in physical units.  Divide physical coefficients by this
    to get zenith-equivalent amplitudes; multiply a prediction by it to get
    back to observed amplitudes at the target line of sight.

    Pass `coef_extinction_k` to use an empirically fitted effective extinction
    instead of the generic stellar curve.
    """
    van_rhijn = airglow_van_rhijn_matrix(
        ctx_phys, ctx_names, group_indices, n_coef, check_physical=check_physical)
    extinction = airglow_extinction_matrix(
        ctx_phys, ctx_names, group_indices, n_coef,
        coef_wavelengths_a=coef_wavelengths_a,
        coef_extinction_k=coef_extinction_k,
        check_physical=False)
    return np.clip(van_rhijn * extinction, 1e-6, None)


# ---------------------------------------------------------------------------
# Empirical effective airglow extinction
#
# The generic LCO curve is a *stellar* extinction curve.  Applying it to
# airglow over-corrects, because airglow is a quasi-uniform extended source:
# photons scattered out of the beam are largely replaced by photons scattered
# in from adjacent lines of sight, and scattering (Rayleigh + aerosol) is
# 70-100% of the total k everywhere airglow matters.  The effective airglow
# extinction is therefore well below the stellar value.
#
# Rather than model multiple scattering, fit the effective coefficient from
# the data.  The two sky pointings are simultaneous, so for coefficient j
#
#   ln(A_near/A_far) - ln(V_near/V_far) = -0.4 ln10 k_eff (X_near - X_far) + eps
#
# with eps the gravity-wave fluctuation between the two lines of sight (zero
# mean in log, uncorrelated with the airmass difference).  This is a Bouguer
# fit that uses airglow as its own source, and the recovered k_eff absorbs the
# multiple-scattering correction, the airglow-versus-stellar difference, the
# site aerosol level and any residual error in the assumed layer height.
#
# Two cautions, both documented in the methods section:
#   * h and k are nearly degenerate over the observed zenith range, so the
#     layer height is HELD FIXED here and only k_eff is fitted.  The product
#     V * 10^(-0.4 k (X-1)) is what the data constrain.
#   * a systematic horizontal gradient in layer brightness that correlates
#     with elevation at a fixed site would bias k_eff.  The returned table
#     includes split-half values so that can be checked.
# ---------------------------------------------------------------------------

def _airglow_height_per_coef(group_indices, n_coef):
    """Effective layer height per coefficient; NaN for non-airglow."""
    heights = np.full(int(n_coef), np.nan, dtype=np.float64)
    for group_name, idx in group_indices.items():
        if group_name not in AIRGLOW_GROUPS:
            continue
        feature_name = _group_height_feature_name(group_name)
        if feature_name is None or feature_name not in VAN_RHIJN_FEATURES:
            continue
        heights[np.asarray(idx, dtype=int)] = float(VAN_RHIJN_FEATURES[feature_name])
    return heights


def _masked_ols_rowconst(y, mask, x_row):
    """OLS of y_ij on [1, x_i] over masked entries, cluster-robust on rows.

    The airmass difference x is constant within a row, which collapses every
    sufficient statistic -- and the entire cluster-robust meat matrix -- to
    per-row counts and per-row sums:

        A_c' u_c = [sum_j u_ij, x_i sum_j u_ij] = g_i * [1, x_i]

    so no flattening and no per-cluster Python loop is needed.  The previous
    implementation looped over clusters (one iteration per row, via np.split)
    and was called about six times per wavelength bin, which dominated the
    runtime on large row counts.
    """
    y_masked = np.where(mask, y, 0.0)
    n_row = mask.sum(axis=1).astype(np.float64)
    s_row = y_masked.sum(axis=1)

    s1 = float(n_row.sum())
    sx = float((x_row * n_row).sum())
    sxx = float((x_row * x_row * n_row).sum())
    sy = float(s_row.sum())
    sxy = float((x_row * s_row).sum())

    xtx = np.array([[s1, sx], [sx, sxx]], dtype=np.float64)
    xtx_inv = np.linalg.pinv(xtx)
    beta = xtx_inv @ np.array([sy, sxy], dtype=np.float64)

    g = s_row - beta[0] * n_row - beta[1] * x_row * n_row
    g2 = g * g
    m01 = float((x_row * g2).sum())
    meat = np.array([[float(g2.sum()), m01],
                     [m01, float((x_row * x_row * g2).sum())]], dtype=np.float64)
    cov = xtx_inv @ meat @ xtx_inv
    return beta, cov, n_row


def fit_effective_extinction(
    coef_near,
    coef_far,
    ctx_near,
    ctx_far,
    ctx_names,
    group_indices,
    coef_wavelengths_a,
    n_wavelength_bins=8,
    wavelength_bin_edges=None,
    min_positive_fraction=0.80,
    clip_sigma=3.0,
    clip_iters=3,
    min_rows=100,
    min_pairs=500,
    clip_sample_rows=20000,
    seed=0,
    verbose=True,
):
    """Fit effective airglow extinction per wavelength bin from the sky pairs.

    Returns a DataFrame with one row per wavelength bin:
      lam_lo, lam_hi, lam_mid, n_coef, n_rows, n_pairs, retained_frac
      k_generic   -- generic LCO stellar curve at lam_mid, for comparison
      k_eff, k_eff_err  -- fitted effective extinction (cluster-robust error)
      intercept, intercept_err -- should be ~0; a significant value indicates a
                       relative throughput offset between the two sky channels
      k_eff_half1, k_eff_half2 -- split-half stability check
      ok          -- whether the bin is well enough constrained to be used

    Performance notes: van Rhijn is evaluated once per distinct layer height as
    an (n_rows,) vector rather than as an (n_rows, n_coef) matrix, the design
    statistics are accumulated from per-row sums instead of flattened
    (n_rows * n_coef) arrays, and the cluster-robust covariance is closed-form
    (see _masked_ols_rowconst).  Only the robust scale used for sigma clipping
    touches individual elements, and it is estimated on a row subsample.
    """
    rng = np.random.default_rng(seed)
    coef_near = np.asarray(coef_near, dtype=np.float64)
    coef_far = np.asarray(coef_far, dtype=np.float64)
    n_rows_all, n_coef = coef_near.shape

    ctx_near_arr, names, alt_near, airmass_near = _airglow_ctx_columns(
        ctx_near, ctx_names, check_physical=True)
    ctx_far_arr, _, alt_far, airmass_far = _airglow_ctx_columns(
        ctx_far, ctx_names, check_physical=False)
    delta_airmass = np.ascontiguousarray(airmass_near - airmass_far)

    height_per_coef = _airglow_height_per_coef(group_indices, n_coef)
    distinct_heights = np.unique(height_per_coef[np.isfinite(height_per_coef)])
    log_vr_ratio = {
        float(h): (np.log(_van_rhijn_factor(alt_near, float(h)))
                   - np.log(_van_rhijn_factor(alt_far, float(h))))
        for h in distinct_heights
    }

    lam = _coef_wavelengths_with_fallback(coef_wavelengths_a, group_indices, n_coef)
    airglow_cols = np.flatnonzero(np.isfinite(height_per_coef))
    if airglow_cols.size == 0:
        if verbose:
            print('fit_effective_extinction: no airglow coefficients; skipping.')
        return pd.DataFrame()

    if verbose:
        print(f'Effective-extinction fit: {airglow_cols.size} airglow coefficients, '
              f'{n_rows_all} rows, {len(distinct_heights)} distinct layer heights.')
        print(f'  airmass difference (near - far): '
              f'16-84% = {np.nanpercentile(delta_airmass, 16):+.3f}..'
              f'{np.nanpercentile(delta_airmass, 84):+.3f}, '
              f'|dX| median = {np.nanmedian(np.abs(delta_airmass)):.3f}')

    lam_ag = lam[airglow_cols]
    if wavelength_bin_edges is None:
        finite = np.isfinite(lam_ag)
        if finite.sum() < 2:
            return pd.DataFrame()
        wavelength_bin_edges = np.unique(np.nanpercentile(
            lam_ag[finite], np.linspace(0.0, 100.0, int(n_wavelength_bins) + 1)))
    wavelength_bin_edges = np.asarray(wavelength_bin_edges, dtype=np.float64)
    if wavelength_bin_edges.size < 2:
        return pd.DataFrame()

    half_assignment = rng.integers(0, 2, n_rows_all).astype(bool)
    slope_to_k = -1.0 / (0.4 * np.log(10.0))
    rows_out = []

    for b in range(wavelength_bin_edges.size - 1):
        lo, hi = wavelength_bin_edges[b], wavelength_bin_edges[b + 1]
        last = (b == wavelength_bin_edges.size - 2)
        in_bin = (lam_ag >= lo) & ((lam_ag <= hi) if last else (lam_ag < hi))
        cols = airglow_cols[in_bin]
        if cols.size == 0:
            continue

        near_block = coef_near[:, cols]
        far_block = coef_far[:, cols]

        # column-level selection only -- a per-row amplitude cut would be
        # selection on the outcome and biases the slope, because whichever arm
        # sits at lower airmass has the smaller van Rhijn factor and fails the
        # cut unless it carries a positive gravity-wave fluctuation
        positive = (near_block > 0.0) & (far_block > 0.0)
        keep_col = positive.mean(axis=0) >= float(min_positive_fraction)
        if not keep_col.any():
            continue
        cols = cols[keep_col]
        near_block = near_block[:, keep_col]
        far_block = far_block[:, keep_col]
        mask = positive[:, keep_col]

        # log amplitude ratio with van Rhijn removed, extinction left in.
        # van Rhijn is constant within a layer height, so it is subtracted as a
        # column vector per height rather than as a full matrix.
        with np.errstate(divide='ignore', invalid='ignore'):
            y_mat = np.log(np.where(mask, near_block, 1.0)) - np.log(
                np.where(mask, far_block, 1.0))
        for h in distinct_heights:
            sel = height_per_coef[cols] == h
            if sel.any():
                y_mat[:, sel] -= log_vr_ratio[float(h)][:, None]
        mask &= np.isfinite(y_mat)
        n_possible = int(mask.size)

        finite_x = np.isfinite(delta_airmass)
        if not finite_x.all():
            mask &= finite_x[:, None]

        for _ in range(int(clip_iters)):
            if mask.sum() < 10:
                break
            beta, _cov, _n_row = _masked_ols_rowconst(y_mat, mask, delta_airmass)
            resid = y_mat - (beta[0] + beta[1] * delta_airmass[:, None])
            if n_rows_all > int(clip_sample_rows):
                sample_rows = rng.choice(n_rows_all, int(clip_sample_rows), replace=False)
            else:
                sample_rows = np.arange(n_rows_all)
            vals = resid[sample_rows][mask[sample_rows]]
            if vals.size < 10:
                break
            med = float(np.median(vals))
            mad = 1.4826 * float(np.median(np.abs(vals - med)))
            if not np.isfinite(mad) or mad <= 0.0:
                break
            new_mask = mask & (np.abs(resid - med) < clip_sigma * mad)
            if new_mask.sum() == mask.sum():
                mask = new_mask
                break
            mask = new_mask

        n_pairs = int(mask.sum())
        lam_mid = float(np.nanmedian(lam[cols]))
        k_generic = float(_lco_extinction_k(lam_mid))
        if n_pairs < 10:
            continue

        beta, cov, n_row = _masked_ols_rowconst(y_mat, mask, delta_airmass)
        n_rows = int((n_row > 0).sum())
        retained = n_pairs / max(n_possible, 1)
        k_eff = float(beta[1] * slope_to_k)
        k_err = float(np.sqrt(max(cov[1, 1], 0.0)) * abs(slope_to_k))

        halves = []
        for which in (False, True):
            half_mask = mask & (half_assignment == which)[:, None]
            if half_mask.sum() >= 10:
                bh, _, _ = _masked_ols_rowconst(y_mat, half_mask, delta_airmass)
                halves.append(float(bh[1] * slope_to_k))
            else:
                halves.append(np.nan)

        rel_err = k_err / abs(k_eff) if k_eff != 0 else np.inf
        ok = bool(n_rows >= min_rows and n_pairs >= min_pairs
                  and np.isfinite(k_eff) and np.isfinite(k_err) and rel_err < 0.5)

        rows_out.append({
            'lam_lo': float(lo), 'lam_hi': float(hi), 'lam_mid': lam_mid,
            'n_coef': int(cols.size), 'n_rows': n_rows, 'n_pairs': n_pairs,
            'retained_frac': float(retained),
            'k_generic': k_generic, 'k_eff': k_eff, 'k_eff_err': k_err,
            'k_ratio': k_eff / k_generic if k_generic > 0 else np.nan,
            'intercept': float(beta[0]),
            'intercept_err': float(np.sqrt(max(cov[0, 0], 0.0))),
            'k_eff_half1': halves[0], 'k_eff_half2': halves[1],
            'ok': ok,
        })

    table = pd.DataFrame(rows_out)
    if verbose and not table.empty:
        print('  fitted effective extinction by wavelength bin:')
        cols_show = ['lam_mid', 'n_coef', 'n_rows', 'n_pairs', 'retained_frac',
                     'k_generic', 'k_eff', 'k_eff_err', 'k_ratio', 'intercept',
                     'k_eff_half1', 'k_eff_half2', 'ok']
        print(table[cols_show].to_string(
            index=False, float_format=lambda v: f'{v:.4g}'))
        low_ret = table[table['retained_frac'] < 0.5]
        if len(low_ret):
            print(f'  NOTE: {len(low_ret)} bin(s) retained under 50% of pairs. Heavy '
                  f'row-level loss reintroduces selection on the outcome and biases '
                  f'k_eff upward; consider raising min_positive_fraction so whole '
                  f'columns are dropped instead.')
        bad_int = table[np.abs(table['intercept'])
                        > 3.0 * table['intercept_err'].clip(lower=1e-12)]
        if len(bad_int):
            print(f'  NOTE: {len(bad_int)} bin(s) have an intercept >3 sigma from '
                  f'zero, which points to a relative throughput offset between '
                  f'the two sky channels rather than an extinction effect.')
    elif verbose:
        print('  effective-extinction fit produced no usable bins.')

    return table


def resolve_coef_extinction_k(
    coef_names,
    coef_wavelengths_a,
    group_indices,
    fit_table=None,
    clip_to_generic=True,
    verbose=True,
    coef_basis_k_generic=None,
):
    """Per-coefficient extinction, preferring the fitted k_eff over the generic curve.

    Well-constrained bins (`ok` in the fit table) are interpolated in
    wavelength; everything else falls back to the generic LCO curve.  With
    `clip_to_generic`, fitted values are restricted to [0, k_generic]: the
    multiple-scattering argument makes the effective airglow extinction a lower
    bound problem, so a fitted value above the stellar curve indicates noise or
    an unmodelled gradient rather than real physics.

    `coef_basis_k_generic`, when provided, is the per-coefficient B^2-weighted
    effective k from coef_wavelengths_from_basis(); where finite it overrides
    the point-sample `_lco_extinction_k(lambda_centroid)` as both the generic
    reference and the [0, k_generic] clip. For narrow-line coefficients this is
    a no-op (|f_j|^2 is peaked at the centroid); for broadband basis components
    (HO2, FeO, O2Ac) it correctly reflects the integrated behaviour of the LCO
    stellar curve across the basis function's support.
    """
    coef_names = [str(n) for n in coef_names]
    n_coef = len(coef_names)
    lam = _coef_wavelengths_with_fallback(coef_wavelengths_a, group_indices, n_coef)
    k_generic = airglow_coef_extinction_k(lam, group_indices, n_coef)
    if coef_basis_k_generic is not None:
        basis_k = np.asarray(coef_basis_k_generic, dtype=np.float64)
        if basis_k.size != n_coef:
            raise ValueError(
                f'coef_basis_k_generic has {basis_k.size} entries, expected {n_coef}'
            )
        replace = np.isfinite(basis_k) & (basis_k >= 0.0)
        if replace.any():
            k_generic = np.where(replace, basis_k, k_generic)
    k_out = k_generic.copy()
    source = ['generic'] * n_coef

    airglow_cols = np.concatenate(
        [np.asarray(idx, dtype=int) for g, idx in group_indices.items()
         if g in AIRGLOW_GROUPS and len(idx) > 0]
    ) if any(g in AIRGLOW_GROUPS and len(idx) > 0
             for g, idx in group_indices.items()) else np.zeros(0, dtype=int)

    usable = None
    if fit_table is not None and len(fit_table):
        usable = fit_table[fit_table['ok'].astype(bool)].sort_values('lam_mid')

    n_clipped = 0
    if usable is not None and len(usable) >= 1:
        lam_nodes = np.asarray(usable['lam_mid'], dtype=np.float64)
        k_nodes = np.asarray(usable['k_eff'], dtype=np.float64)
        for j in airglow_cols:
            if not np.isfinite(lam[j]):
                continue
            k_fit = float(np.interp(lam[j], lam_nodes, k_nodes,
                                    left=k_nodes[0], right=k_nodes[-1]))
            if clip_to_generic:
                k_clipped = float(np.clip(k_fit, 0.0, k_generic[j]))
                if k_clipped != k_fit:
                    n_clipped += 1
                k_fit = k_clipped
            k_out[j] = k_fit
            source[j] = 'fitted'

    if verbose:
        n_fit = sum(1 for s in source if s == 'fitted')
        print(f'Extinction resolved for {airglow_cols.size} airglow coefficients: '
              f'{n_fit} fitted, {airglow_cols.size - n_fit} generic.')
        if n_clipped:
            print(f'  {n_clipped} fitted value(s) clipped into [0, k_generic].')
        if airglow_cols.size:
            ratio = np.divide(k_out[airglow_cols], k_generic[airglow_cols],
                              out=np.full(airglow_cols.size, np.nan),
                              where=k_generic[airglow_cols] > 0)
            with np.errstate(invalid='ignore'):
                print(f'  k_eff / k_generic: median {np.nanmedian(ratio):.3f}, '
                      f'range {np.nanmin(ratio):.3f}..{np.nanmax(ratio):.3f}')

    return k_out, source




In [ ]:
# New helper: build simultaneous triplets (near, far -> sci) from decomposition files
def _load_decomp_with_row_index(
    decomp_fits_path,
    input_fits_path,
    context_columns,
    decomp_kind,
    return_chi2=False,
    return_err=False,
):
    """Load one decomp product and keep original source-row indices after finite filtering."""
    out = read_decomp_dataset(
        decomp_fits_path=decomp_fits_path,
        input_fits_path=input_fits_path,
        context_columns=context_columns,
        decomp_kind=decomp_kind,
        return_chi2=return_chi2,
        return_err=return_err,
    )

    # Unpack in the order used by read_decomp_dataset: base, then chi2, then err.
    _it = iter(out)
    coef_mat = next(_it); ctx_mat = next(_it); coef_names = next(_it); ctx_names = next(_it)
    chi2_used = next(_it) if return_chi2 else None
    coef_err_mat = next(_it) if return_err else None

    with fits.open(decomp_fits_path) as hdul_dec, fits.open(input_fits_path) as hdul_in:
        coef_tbl_full = _coerce_coef_hdu_to_table(hdul_dec['COEF'])
        coef_full, _ = _table_to_float32_matrix(coef_tbl_full, 'coefficient')
        meta_full = Table(hdul_in['META'].data)
        ctx_full, _ = _build_context_matrix(meta_full, context_columns, decomp_kind)
        meta_upper_full = {c.upper(): c for c in meta_full.colnames}
        obstime_mjd_full = _extract_obstime_mjd(meta_full, meta_upper_full)

    if coef_full.shape[0] != ctx_full.shape[0]:
        raise ValueError(
            f'Row count mismatch while building row_index: COEF has {coef_full.shape[0]} rows, META has {ctx_full.shape[0]} rows'
        )

    good = np.isfinite(coef_full).all(axis=1) & np.isfinite(ctx_full).all(axis=1)
    row_index = np.flatnonzero(good).astype(np.int64)
    obstime_mjd = np.asarray(obstime_mjd_full[good], dtype=np.float64)

    if row_index.size != coef_mat.shape[0]:
        raise RuntimeError(
            f'Internal row-index mismatch: row_index has {row_index.size}, data has {coef_mat.shape[0]}'
        )

    result = [coef_mat, ctx_mat, coef_names, ctx_names]
    if return_chi2:
        result.append(chi2_used)
    result.extend([row_index, obstime_mjd])
    if return_err:
        result.append(coef_err_mat)
    return tuple(result)


def build_triplet_coef_dataset(
    input_fits_path,
    sky_near_decomp_fits_path,
    sky_far_decomp_fits_path,
    sci_decomp_fits_path,
    context_columns,
    return_chi2=False,
    return_err=True,
):
    """Build aligned triplet arrays for coefficient-transfer experiments.

    ``return_err=True`` (default) also pulls the per-coefficient 1-sigma
    uncertainties from the ``COEF_ERR`` HDU (see fit.SkyDecomp._coef_err_active_set)
    and stores them as ``coef_err_near``, ``coef_err_far`` and ``coef_err_sci``.
    Older decomposition files without ``COEF_ERR`` yield all-NaN arrays.
    """

    def _load(kind, path):
        return _load_decomp_with_row_index(
            decomp_fits_path=path,
            input_fits_path=input_fits_path,
            context_columns=context_columns,
            decomp_kind=kind,
            return_chi2=return_chi2,
            return_err=return_err,
        )

    near = _load('sky1', sky_near_decomp_fits_path)
    far = _load('sky2', sky_far_decomp_fits_path)
    sci = _load('sci', sci_decomp_fits_path)

    def _unpack(out):
        it = iter(out)
        coef = next(it); ctx = next(it); cnames = next(it); xnames = next(it)
        chi2 = next(it) if return_chi2 else None
        row = next(it); mjd = next(it)
        err = next(it) if return_err else None
        return coef, ctx, cnames, xnames, chi2, row, mjd, err

    coef_near, ctx_near, coef_names_n, ctx_names_n, chi2_near, row_near, mjd_near, coef_err_near = _unpack(near)
    coef_far, ctx_far, coef_names_f, ctx_names_f, chi2_far, row_far, mjd_far, coef_err_far = _unpack(far)
    coef_sci, ctx_sci, coef_names_s, ctx_names_s, chi2_sci, row_sci, mjd_sci, coef_err_sci = _unpack(sci)

    if coef_names_n != coef_names_f or coef_names_n != coef_names_s:
        raise ValueError('Coefficient name mismatch across near/far/sci decomposition products.')
    if ctx_names_n != ctx_names_f or ctx_names_n != ctx_names_s:
        raise ValueError('Context name mismatch across near/far/sci products.')

    row_common = np.intersect1d(np.intersect1d(row_near, row_far), row_sci)
    if row_common.size == 0:
        raise ValueError('No shared source rows across near/far/sci after finite filtering.')

    idx_near = np.searchsorted(row_near, row_common)
    idx_far = np.searchsorted(row_far, row_common)
    idx_sci = np.searchsorted(row_sci, row_common)

    if (not np.array_equal(row_near[idx_near], row_common)
            or not np.array_equal(row_far[idx_far], row_common)
            or not np.array_equal(row_sci[idx_sci], row_common)):
        raise RuntimeError('Row-index alignment failure while building triplet dataset.')

    mjd_stack = np.column_stack([mjd_near[idx_near], mjd_far[idx_far], mjd_sci[idx_sci]])
    obstime_mjd = np.nanmedian(mjd_stack, axis=1).astype(np.float64)

    out = {
        'coef_near': coef_near[idx_near],
        'coef_far': coef_far[idx_far],
        'coef_sci': coef_sci[idx_sci],
        'ctx_near': ctx_near[idx_near],
        'ctx_far': ctx_far[idx_far],
        'ctx_sci': ctx_sci[idx_sci],
        'coef_names': coef_names_n,
        'ctx_names': ctx_names_n,
        'row_index': row_common.astype(np.int64),
        'obstime_mjd': obstime_mjd,
        'n_rows': int(row_common.size),
    }

    if return_chi2:
        out['chi2_near'] = chi2_near[idx_near]
        out['chi2_far'] = chi2_far[idx_far]
        out['chi2_sci'] = chi2_sci[idx_sci]

    if return_err:
        out['coef_err_near'] = coef_err_near[idx_near]
        out['coef_err_far'] = coef_err_far[idx_far]
        out['coef_err_sci'] = coef_err_sci[idx_sci]

    # Attach per-arm RA/Dec from META so downstream filters can exclude tiles
    # by sky region (LMC/SMC exclusion in apply_triplet_filters) and so the
    # ecliptic-augment cell can compute per-arm zodi-relevant ecliptic coords.
    # SCI is required (falls back to unprefixed RA/DEC); SKY_NEAR/SKY_FAR are
    # optional (missing -> just no per-arm ecliptic features later).
    try:
        with fits.open(input_fits_path) as _hdul_meta:
            _meta_tbl = Table(_hdul_meta['META'].data)
        _meta_upper = {c.upper(): c for c in _meta_tbl.colnames}
        _ra_col = next((_meta_upper[k] for k in ('SCI_RA', 'RA') if k in _meta_upper), None)
        _dec_col = next((_meta_upper[k] for k in ('SCI_DEC', 'DEC') if k in _meta_upper), None)
        if _ra_col is not None and _dec_col is not None:
            out['sci_ra'] = np.asarray(_meta_tbl[_ra_col], dtype=np.float64)[row_common]
            out['sci_dec'] = np.asarray(_meta_tbl[_dec_col], dtype=np.float64)[row_common]
            out['sci_radec_source'] = (_ra_col, _dec_col)
        else:
            print('  WARNING: sci RA/Dec columns not found in META '
                  '(looked for SCI_RA/SCI_DEC then RA/DEC); '
                  'field-level exclusion in apply_triplet_filters will be skipped.')
        for _arm_prefix, _ra_key, _dec_key in (
            ('near', 'SKY_NEAR_RA', 'SKY_NEAR_DEC'),
            ('far', 'SKY_FAR_RA', 'SKY_FAR_DEC'),
        ):
            _arm_ra = _meta_upper.get(_ra_key)
            _arm_dec = _meta_upper.get(_dec_key)
            if _arm_ra is not None and _arm_dec is not None:
                out[f'{_arm_prefix}_ra'] = np.asarray(
                    _meta_tbl[_arm_ra], dtype=np.float64)[row_common]
                out[f'{_arm_prefix}_dec'] = np.asarray(
                    _meta_tbl[_arm_dec], dtype=np.float64)[row_common]
    except Exception as _exc:
        print(f'  WARNING: could not attach sci_ra/sci_dec '
              f'({type(_exc).__name__}: {_exc}).')

    print(
        f"Triplet dataset built: n_rows={out['n_rows']}, n_coef={out['coef_near'].shape[1]}, n_ctx={out['ctx_near'].shape[1]}"
        + (f" | sci pointing from META columns ({out['sci_radec_source'][0]}, {out['sci_radec_source'][1]})"
           if 'sci_radec_source' in out else "")
    )
    return out

In [ ]:
# Reused reconstruction/prediction helpers for quick visual checks
def _meta_row_to_dict_upper(meta_row):
    names = list(meta_row.colnames) if hasattr(meta_row, 'colnames') else list(meta_row.dtype.names)
    return {str(k).upper(): k for k in names}

def _safe_float(x):
    arr = np.asarray(x)
    if arr.size == 0:
        raise ValueError('Empty value cannot be converted to float')
    if arr.shape != ():
        arr = arr.ravel()[0]
    return float(arr)


def _infer_base_dir_for_reconstruction():
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    if 'PALACE_DIR' in globals():
        try:
            p = Path(PALACE_DIR).resolve()
            candidates.extend([p, p.parent])
        except Exception:
            pass

    for cand in candidates:
        if (cand / 'palace' / 'PMD').exists() and (cand / 'Spectre_HR_LATMOS_Meftah_V1_350_1000nm.txt').exists():
            return cand

    raise FileNotFoundError('Could not infer reconstruction base_dir containing palace/PMD and solar reference file')

def load_lsf_state_if_available(decomp_fits_path, spectrum_index):
    """Return an LSFSurfaceState from a decomposition FITS row, or None if absent.

    The new sky_decomp.lsf_surface_iterative module writes LSF_COEF / LSF_KNOTS /
    LSF_META extensions alongside COEF and META.  Older decomposition outputs do
    not have those extensions; this returns None so callers can fall back to a
    Gaussian LSF from the input FITS.
    """
    path = Path(decomp_fits_path)
    if not path.exists():
        return None
    try:
        with fits.open(str(path)) as hdul:
            ext_names = {h.name for h in hdul}
            if 'LSF_COEF' not in ext_names:
                return None
        return load_lsf_surface_state(str(path), int(spectrum_index))
    except (KeyError, IndexError, ValueError) as exc:
        print(f'  LSF surface state unavailable in {path.name} row {spectrum_index}: '
              f'{type(exc).__name__}: {exc}')
        return None


def load_o2_vector_if_available(decomp_fits_path, spectrum_index):
    """Return the unit-integrated O2 template for one row, or None if absent.

    The reworked decomposition pipeline (fit.py + decompose_parallel.py) writes a
    VECTOR_O2 ImageHDU (n_rows x n_wave) alongside COEF / META. Older
    decomposition products predate the split of the O2 amplitude into the
    coefficient, so their VECTOR_O2 is missing and the O2 basis reconstructs to
    zero (matching the historical behaviour of reconstruct_component_spectra).
    """
    path = Path(decomp_fits_path)
    if not path.exists():
        return None
    try:
        with fits.open(str(path)) as hdul:
            if 'VECTOR_O2' not in {h.name for h in hdul}:
                return None
            data = np.asarray(hdul['VECTOR_O2'].data, dtype=np.float64)
    except (KeyError, IndexError, ValueError) as exc:
        print(f'  VECTOR_O2 unavailable in {path.name} row {spectrum_index}: '
              f'{type(exc).__name__}: {exc}')
        return None
    if data.ndim != 2 or int(spectrum_index) >= data.shape[0]:
        return None
    row = data[int(spectrum_index)]
    if not np.isfinite(row).any() or float(np.nansum(np.abs(row))) == 0.0:
        return None
    return row


def load_moon_zodi_state_if_available(decomp_fits_path, spectrum_index):
    """Return a MoonZodiState from a decomp FITS row, or None if the file is not moon-zodi.

    The moon-zodi variant of the decomposition (``sky_decomp.moon_zodi_lsf_surface_iterative``)
    writes four extra HDUs (``MZ_MODEL`` / ``MZ_ASSETS`` / ``MZ_KNOTS`` / ``MZ_META``)
    that together encode the physical Moon+Zodi predictor state per row.  Older
    ``lsf_surface_iterative`` outputs do not have those extensions; this returns
    ``None`` so callers can fall back to the parent (moon-spline-only) reconstruction.
    """
    path = Path(decomp_fits_path)
    if not path.exists():
        return None
    try:
        with fits.open(str(path)) as hdul:
            ext_names = {h.name for h in hdul}
            if not all(name in ext_names for name in MOON_ZODI_HDU_NAMES):
                return None
        return load_moon_zodi_state(str(path), int(spectrum_index))
    except (KeyError, IndexError, ValueError) as exc:
        print(f'  Moon/Zodi state unavailable in {path.name} row {spectrum_index}: '
              f'{type(exc).__name__}: {exc}')
        return None


def reconstruct_with_lsf(wave, coef, lsf, *, n_spline_knots=25, base_dir=None,
                         o2_vector=None, coef_err=None,
                         moon_zodi_state=None, detector_lsf_fwhm=None,
                         physical_to_fit_flux_scale=None):
    """Reconstruct component spectra, dispatching on the LSF representation.

    ``lsf`` is either an ``LSFSurfaceState`` (uses the wavelength-dependent
    B-spline kernel via ``SkyDecompLSFSurfaceIterative._assemble_refined_matrices``)
    or a per-pixel Gaussian sigma vector / scalar (uses the parent-class
    ``reconstruct_component_spectra``).  Returns the same components dict as
    ``reconstruct_component_spectra``.

    If ``coef_err`` (per-coefficient 1σ, same shape as ``coef``) is
    provided, the returned dict additionally contains ``sigma`` (dict of
    per-component 1σ flux uncertainty per pixel) and ``sigma_total``
    (quadrature sum across independent components).  See
    ``SkyDecompBase._components_sigma_from_coef_err`` for the propagation.

    When ``moon_zodi_state`` is provided (loaded via
    ``load_moon_zodi_state_if_available``), the moon block is re-materialised
    from a per-row physical Moon+Zodi predictor multiplied by a fitted
    B-spline correction.  ``detector_lsf_fwhm`` (per-pixel FWHM in wavelength
    units, float64, same shape as ``wave``) and ``physical_to_fit_flux_scale``
    are then required; ``lsf`` must be an ``LSFSurfaceState``.  Returned
    components include an extra ``zodi`` key and ``total`` sums moon + zodi
    + oh + diffuse + atom + orc + o2.
    """
    if moon_zodi_state is not None:
        if not isinstance(lsf, LSFSurfaceState):
            raise ValueError(
                'moon_zodi_state requires an LSFSurfaceState (the surface '
                'kernel is part of the pipeline used to fit the correction)'
            )
        if detector_lsf_fwhm is None or physical_to_fit_flux_scale is None:
            raise ValueError(
                'moon_zodi_state requires detector_lsf_fwhm and '
                'physical_to_fit_flux_scale (same scale the pipeline used at fit time)'
            )
        wave_arr = np.asarray(wave, dtype=np.float64)
        lsf_fwhm = np.asarray(detector_lsf_fwhm, dtype=np.float64).ravel()
        if lsf_fwhm.shape != wave_arr.shape:
            raise ValueError(
                f'detector_lsf_fwhm shape mismatch: expected {wave_arr.shape}, '
                f'got {lsf_fwhm.shape}'
            )
        coef_arr = np.asarray(coef, float).ravel()
        # n_spline_knots = n_moon_correction_basis - 4 (cubic B-spline with clamped ends).
        n_full_knots = len(moon_zodi_state.correction_knots)
        n_interior = max(1, n_full_knots - 8)
        # Moon/Zodi model bundles its PALACE tables (with the "_joint_v2_updated" suffix)
        # and moon_zodi assets inside its own DEFAULT_DATA_ROOT (sky_decomp/data), so let
        # the constructor use its default data_root; do NOT pass the notebook base_dir.
        model = SkyDecompMoonZodiLSFSurfaceIterative(
            wave=wave_arr,
            lsf_sigma=1.0,
            physical_to_fit_flux_scale=float(physical_to_fit_flux_scale),
            n_spline_knots=int(n_interior),
        )
        try:
            model._install_prediction(moon_zodi_state.observation, lsf_fwhm)
        except Exception as exc:
            raise RuntimeError(f'Moon/Zodi prediction failed: {exc}') from exc
        model._set_lsf_state(lsf)
        mats = model._assemble_refined_matrices()
        if o2_vector is not None:
            o2_vec = np.asarray(o2_vector, float).ravel()
            if o2_vec.shape != model.wave.shape:
                raise ValueError(
                    f'o2_vector shape mismatch: expected {model.wave.shape}, '
                    f'got {o2_vec.shape}'
                )
            mats['o2'] = o2_vec[None, :]
        n_expected = sum(m.shape[0] for m in mats.values())
        if coef_arr.size != n_expected:
            raise ValueError(
                f'Coefficient length mismatch: expected {n_expected}, got {coef_arr.size}'
            )
        comps = model._components_from_coef(coef_arr, mats)
        # comps['moon'] and comps['zodi'] share the same coefficient block, so 
        # their uncertainties are fully correlated; the propagator returns a single 
        # sigma['moon'] built from the combined design matrix.
        comps['total'] = (comps['oh'] + comps['moon'] + comps['zodi']
                          + comps['diffuse'] + comps['atom']
                          + comps['orc'] + comps['o2'])
        if coef_err is not None:
            err_arr = np.asarray(coef_err, float).ravel()
            if err_arr.size != coef_arr.size:
                raise ValueError(
                    f'coef_err length mismatch: expected {coef_arr.size}, '
                    f'got {err_arr.size}'
                )
            sigma_comps = model._components_sigma_from_coef_err(err_arr, mats)
            comps['sigma'] = sigma_comps
            comps['sigma_total'] = np.sqrt(
                sigma_comps['oh'] ** 2
                + sigma_comps['moon'] ** 2
                + sigma_comps['diffuse'] ** 2
                + sigma_comps['atom'] ** 2
                + sigma_comps['orc'] ** 2
                + sigma_comps['o2'] ** 2
            )
        return comps
    if isinstance(lsf, LSFSurfaceState):
        model = SkyDecompLSFSurfaceIterative(
            wave,
            lsf_sigma=1.0,  # dummy; only stick matrices are used with the surface path
            n_spline_knots=n_spline_knots,
            base_dir=base_dir,
        )
        coef_arr = np.asarray(coef, float).ravel()
        model._set_lsf_state(lsf)
        mats = model._assemble_refined_matrices()
        # VECTOR_O2 on disk is already convolved by the fitted LSF surface at fit
        # time; injecting it into matrix_o2_stick and letting _assemble_refined_matrices
        # re-convolve would double-broaden the O2 shape. Override the assembled O2
        # block verbatim instead.
        if o2_vector is not None:
            o2_vec = np.asarray(o2_vector, float).ravel()
            if o2_vec.shape != model.wave.shape:
                raise ValueError(
                    f'o2_vector shape mismatch: expected {model.wave.shape}, got {o2_vec.shape}'
                )
            mats['o2'] = o2_vec[None, :]
        n_expected = sum(m.shape[0] for m in mats.values())
        if coef_arr.size != n_expected:
            raise ValueError(
                f'Coefficient length mismatch: expected {n_expected}, got {coef_arr.size}'
            )
        comps = model._components_from_coef(coef_arr, mats)
        comps['total'] = (comps['oh'] + comps['moon'] + comps['diffuse']
                          + comps['atom'] + comps['orc'] + comps['o2'])
        if coef_err is not None:
            err_arr = np.asarray(coef_err, float).ravel()
            if err_arr.size != coef_arr.size:
                raise ValueError(
                    f'coef_err length mismatch: expected {coef_arr.size}, '
                    f'got {err_arr.size}'
                )
            sigma_comps = model._components_sigma_from_coef_err(err_arr, mats)
            comps['sigma'] = sigma_comps
            comps['sigma_total'] = np.sqrt(
                sigma_comps['oh'] ** 2
                + sigma_comps['moon'] ** 2
                + sigma_comps['diffuse'] ** 2
                + sigma_comps['atom'] ** 2
                + sigma_comps['orc'] ** 2
                + sigma_comps['o2'] ** 2
            )
        return comps
    return reconstruct_component_spectra(
        wave=wave, coef=coef, lsf_sigma=lsf,
        n_spline_knots=n_spline_knots, base_dir=base_dir, o2_vector=o2_vector,
        coef_err=coef_err,
    )

In [ ]:
# Starter data load for coefficient prediction experiments
# Context features (moon+zodi variant, 2026-08-18 v2): moon_sep / moon_alt /
# moon_az_{sin,cos} RESTORED after /tmp/moon_geometry_correlation.py showed
# they correlate strongly (|r|=0.24-0.39) with the moon PCA scores in the
# moon+zodi coefficient basis -- much stronger than in the moon-only variant
# where /tmp/post_moonzodi_analysis.py measured |r|<0.06 and led us to drop
# them. kp / obstime_lunation_sin,cos remain dropped (still universally weak).
context_cols = [
    'alt',
    'az_sin',
    'az_cos',
    'airmass',
    'moon_sep',
    'moon_alt',
    'moon_az_sin',
    'moon_az_cos',
    'moon_phase_sin',
    'moon_phase_cos',
    'sun_sep',
    'sun_alt',
    'sun_az_sin',
    'sun_az_cos',
    'sci_sep',
    'vanrhijn_87km',
    'vanrhijn_95km',
    'vanrhijn_285km',
    'obstime_day_sin',
    'obstime_day_cos',
    'obstime_year_sin',
    'obstime_year_cos',
    'f107',
    'f107_81d',
    'ew',
]

# Toggle the decomposition source: LSF-surface-iterative (wavelength-dependent LSF) or the baseline (Gaussian LSF).
USE_LSF_SURFACE_ITERATIVE = True
USE_MOON_ZODI = True   # 2026-08-17: moon-zodi decompositions available
if USE_MOON_ZODI:
    _DECOMP_SUFFIX = '_moon_zodi_lsf_surface_iterative'
elif USE_LSF_SURFACE_ITERATIVE:
    _DECOMP_SUFFIX = '_lsf_surface_iterative'
else:
    _DECOMP_SUFFIX = ''
_DECOMP_STEM = 'lvmsframe_median_stack_1.2.1_p40_p70'
print(f'Using decomposition products: suffix={_DECOMP_SUFFIX!r}')

triplet = build_triplet_coef_dataset(
    input_fits_path=f'{_DECOMP_STEM}_meta_only.fits',
    sky_near_decomp_fits_path=f'{_DECOMP_STEM}_sky1_meta_coef{_DECOMP_SUFFIX}.fits',
    sky_far_decomp_fits_path=f'{_DECOMP_STEM}_sky2_meta_coef{_DECOMP_SUFFIX}.fits',
    sci_decomp_fits_path=f'{_DECOMP_STEM}_sci_meta_coef{_DECOMP_SUFFIX}.fits',
    context_columns=context_cols,
    return_chi2=True,
)

print('Shapes:')
print('  coef_near', triplet['coef_near'].shape)
print('  coef_far ', triplet['coef_far'].shape)
print('  coef_sci ', triplet['coef_sci'].shape)
print('  ctx_near ', triplet['ctx_near'].shape)
print('  ctx_far  ', triplet['ctx_far'].shape)
print('  ctx_sci  ', triplet['ctx_sci'].shape)
print('  obstime  ', triplet['obstime_mjd'].shape)


In [ ]:
# Filtering borrowed from the original notebook workflow, adapted to triplets
import plotly.express as px


def _angular_separation_deg_vec(ra_deg, dec_deg, ra_c_deg, dec_c_deg):
    """Great-circle angular separation (degrees), vectorised over (ra, dec)."""
    ra = np.deg2rad(np.asarray(ra_deg, dtype=np.float64))
    dec = np.deg2rad(np.asarray(dec_deg, dtype=np.float64))
    ra_c = np.deg2rad(float(ra_c_deg))
    dec_c = np.deg2rad(float(dec_c_deg))
    cos_sep = (np.sin(dec) * np.sin(dec_c)
               + np.cos(dec) * np.cos(dec_c) * np.cos(ra - ra_c))
    return np.rad2deg(np.arccos(np.clip(cos_sep, -1.0, 1.0)))

LMC_EXCLUSION = {'name': 'LMC', 'ra_deg': 81, 'dec_deg': -69.7, 'radius_deg': 10.0}
SMC_EXCLUSION = {'name': 'SMC', 'ra_deg': 14, 'dec_deg': -73, 'radius_deg': 10}

def _kappa_sigma_row_mask(x, kappa=5.0, n_iter=3):
    x = np.asarray(x, dtype=np.float64)
    keep = np.isfinite(x).all(axis=1)
    if not np.any(keep):
        return keep

    for _ in range(n_iter):
        mu = np.nanmean(x[keep], axis=0)
        sig = np.nanstd(x[keep], axis=0)
        sig = np.where(np.isfinite(sig) & (sig > 0), sig, 1.0)
        within = np.all(np.abs(x - mu) <= (kappa * sig), axis=1)
        within &= np.isfinite(x).all(axis=1)
        new_keep = keep & within
        if new_keep.sum() == keep.sum() or new_keep.sum() == 0:
            break
        keep = new_keep

    return keep


def _kappa_mad_row_mask(x, kappa=4.0, n_iter=3):
    """Robust variant of _kappa_sigma_row_mask using per-column median + MAD.

    Median and MAD are not inflated by the outliers themselves, so at kappa=4
    the threshold reflects the actual bulk of the distribution.  Necessary for
    OH coefficients where a small fraction of rows carry decomposition failures
    at ~1e10 to 1e15 that would drag the mean/std of _kappa_sigma_row_mask so
    high that the outliers pass their own gate.  MAD is scaled by 1.4826 so
    kappa is directly comparable to sigma in the Gaussian limit.
    """
    x = np.asarray(x, dtype=np.float64)
    keep = np.isfinite(x).all(axis=1)
    if not np.any(keep):
        return keep

    for _ in range(n_iter):
        med = np.nanmedian(x[keep], axis=0)
        mad = np.nanmedian(np.abs(x[keep] - med), axis=0) * 1.4826
        mad = np.where(np.isfinite(mad) & (mad > 0), mad, 1.0)
        within = np.all(np.abs(x - med) <= (kappa * mad), axis=1)
        within &= np.isfinite(x).all(axis=1)
        new_keep = keep & within
        if new_keep.sum() == keep.sum() or new_keep.sum() == 0:
            break
        keep = new_keep

    return keep


def apply_triplet_filters(
    triplet_data,
    thin_every_n=1,
    chi2_qmax=90.0,
    chi2_min=0.0,
    chi2_max=10.0,
    hard_coef_bounds=None,
    kappa=6.0,
    kappa_iter=3,
    oh_kappa=4.0,
    oh_kappa_iter=3,
    exclude_field_regions=None,
    airmass_max=3.0,
):
    if hard_coef_bounds is None:
        hard_coef_bounds = {'feo': (0.0, 1.0), 'atom_k': (0.0, 1.0)}

    coef_names_local = [str(n) for n in triplet_data['coef_names']]
    coef_name_l = [n.lower() for n in coef_names_local]

    coef_near = np.asarray(triplet_data['coef_near'], dtype=np.float32)
    coef_far = np.asarray(triplet_data['coef_far'], dtype=np.float32)
    coef_sci = np.asarray(triplet_data['coef_sci'], dtype=np.float32)
    ctx_near = np.asarray(triplet_data['ctx_near'], dtype=np.float32)
    ctx_far = np.asarray(triplet_data['ctx_far'], dtype=np.float32)
    ctx_sci = np.asarray(triplet_data['ctx_sci'], dtype=np.float32)

    n0 = coef_near.shape[0]
    keep = np.ones(n0, dtype=bool)

    # Exclude tiles whose science pointing falls within a specified angular
    # radius of a listed sky region.  Defaults to LMC and SMC at 10 deg each --
    # both Clouds combine bright stellar populations, dense H II regions and
    # diffuse ionised gas at velocities close enough to airglow to blend with
    # the sky-component fit of coef_sci (methods sect 11.2).  Pass an empty
    # list to disable.
    if exclude_field_regions is None:
        exclude_field_regions = [LMC_EXCLUSION, SMC_EXCLUSION]
    if exclude_field_regions:
        if 'sci_ra' not in triplet_data or 'sci_dec' not in triplet_data:
            print("Science-field exclusion requested but triplet has no "
                  "sci_ra/sci_dec; skipping.  Re-run build_triplet_coef_dataset "
                  "so it attaches the science pointing coordinates.")
        else:
            sci_ra = np.asarray(triplet_data['sci_ra'], dtype=np.float64)
            sci_dec = np.asarray(triplet_data['sci_dec'], dtype=np.float64)
            field_mask = np.ones(n0, dtype=bool)
            for region in exclude_field_regions:
                reg_name = str(region.get('name', 'unnamed'))
                ra_c = float(region['ra_deg'])
                dec_c = float(region['dec_deg'])
                r_deg = float(region['radius_deg'])
                sep = _angular_separation_deg_vec(sci_ra, sci_dec, ra_c, dec_c)
                inside = np.isfinite(sep) & (sep <= r_deg)
                print(
                    f"Science-field exclusion around {reg_name} "
                    f"(ra={ra_c:.3f} deg, dec={dec_c:.3f} deg, radius={r_deg:.1f} deg): "
                    f"excluded {int(inside.sum())}/{inside.size} "
                    f"({100.0 * inside.mean():.1f}%)"
                )
                field_mask &= ~inside
            keep &= field_mask
            print(
                f"Combined science-field exclusion: kept {int(field_mask.sum())}/"
                f"{len(field_mask)} ({100.0 * field_mask.mean():.1f}%)"
            )

    ctx_names_l = [str(n).strip().lower() for n in triplet_data['ctx_names']]
    alt_idx = ctx_names_l.index('alt') if 'alt' in ctx_names_l else None
    airmass_idx = ctx_names_l.index('airmass') if 'airmass' in ctx_names_l else None
    vr87_idx = ctx_names_l.index('vanrhijn_87km') if 'vanrhijn_87km' in ctx_names_l else None
    vr95_idx = ctx_names_l.index('vanrhijn_95km') if 'vanrhijn_95km' in ctx_names_l else None
    vr285_idx = ctx_names_l.index('vanrhijn_285km') if 'vanrhijn_285km' in ctx_names_l else None

    if alt_idx is None:
        print("Altitude sanity filter: context column 'alt' not found; skipping altitude >= 0 check.")

    physical_mask = np.ones(n0, dtype=bool)
    if alt_idx is not None:
        alt_ok = (
            np.isfinite(ctx_near[:, alt_idx]) & (ctx_near[:, alt_idx] >= 0.0)
            & np.isfinite(ctx_far[:, alt_idx]) & (ctx_far[:, alt_idx] >= 0.0)
            & np.isfinite(ctx_sci[:, alt_idx]) & (ctx_sci[:, alt_idx] >= 0.0)
        )
        physical_mask &= alt_ok
        print(
            f"Altitude sanity filter (alt >= 0 in near/far/sci): kept {alt_ok.sum()}/{alt_ok.size} "
            f"({100.0 * alt_ok.mean():.1f}%)"
        )

    # airmass = sec(z) diverges near the horizon; anything above ~3 is not
    # science-usable and dominates the loss through the (X-1) extinction term.
    if airmass_idx is not None and airmass_max is not None:
        am_max = float(airmass_max)
        am_ok = (
            np.isfinite(ctx_near[:, airmass_idx]) & (ctx_near[:, airmass_idx] <= am_max)
            & np.isfinite(ctx_far[:, airmass_idx]) & (ctx_far[:, airmass_idx] <= am_max)
            & np.isfinite(ctx_sci[:, airmass_idx]) & (ctx_sci[:, airmass_idx] <= am_max)
        )
        physical_mask &= am_ok
        print(
            f"Airmass sanity filter (airmass <= {am_max:g} in near/far/sci): "
            f"kept {am_ok.sum()}/{am_ok.size} ({100.0 * am_ok.mean():.1f}%)"
        )
    elif airmass_idx is None and airmass_max is not None:
        print("Airmass sanity filter: context column 'airmass' not found; skipping.")

    for label, idx in (('vanrhijn_87km', vr87_idx), ('vanrhijn_95km', vr95_idx), ('vanrhijn_285km', vr285_idx)):
        if idx is None:
            continue
        vr_ok = (
            np.isfinite(ctx_near[:, idx]) & (ctx_near[:, idx] >= 1.0)
            & np.isfinite(ctx_far[:, idx]) & (ctx_far[:, idx] >= 1.0)
            & np.isfinite(ctx_sci[:, idx]) & (ctx_sci[:, idx] >= 1.0)
        )
        physical_mask &= vr_ok
        print(
            f"{label} sanity filter (finite and >= 1): kept {vr_ok.sum()}/{vr_ok.size} "
            f"({100.0 * vr_ok.mean():.1f}%)"
        )

    keep &= physical_mask
    print(
        f"Combined physical context filter: kept {physical_mask.sum()}/{len(physical_mask)} "
        f"({100.0 * physical_mask.mean():.1f}%)"
    )

    if all(k in triplet_data for k in ('chi2_near', 'chi2_far', 'chi2_sci')):
        chi2_stack = np.column_stack(
            [
                np.asarray(triplet_data['chi2_near'], dtype=np.float64),
                np.asarray(triplet_data['chi2_far'], dtype=np.float64),
                np.asarray(triplet_data['chi2_sci'], dtype=np.float64),
            ]
        )
        chi2_combined = np.nanmax(chi2_stack, axis=1)
        chi2_finite = chi2_combined[np.isfinite(chi2_combined)]
        chi2_hi = np.nanpercentile(chi2_finite, chi2_qmax)
        chi2_upper = min(float(chi2_max), float(chi2_hi)) if chi2_max is not None else float(chi2_hi)
        chi2_mask = np.isfinite(chi2_combined) & (chi2_combined >= chi2_min) & (chi2_combined <= chi2_upper)
        keep &= chi2_mask
        print(
            f"Triplet chi2 filter: min={chi2_min:.3g}, qmax={chi2_qmax:.1f}%=>{chi2_hi:.3g}, "
            f"upper={chi2_upper:.3g} | keep={chi2_mask.sum()}/{len(chi2_mask)} ({100.0*chi2_mask.mean():.1f}%)"
        )
        fig_chi2_triplet = px.histogram(
            x=chi2_combined[keep],
            nbins=80,
            title='Triplet combined reduced chi2 distribution (rows used for training)',
            labels={'x': 'max(reduced chi2 near/far/sci)', 'y': 'count'},
        )
        fig_chi2_triplet.update_layout(template='plotly_white', bargap=0.03)
        fig_chi2_triplet.show()
    else:
        print('Triplet chi2 columns not present; chi2 filtering skipped.')

    for cname, (lo, hi) in hard_coef_bounds.items():
        idxs = np.where(np.array(coef_name_l) == str(cname).lower())[0]
        if idxs.size == 0:
            print(f"Manual hard clip: coefficient {cname} not found; skipping.")
            continue

        j = int(idxs[0])
        within = (
            np.isfinite(coef_near[:, j]) & (coef_near[:, j] >= lo) & (coef_near[:, j] <= hi)
            & np.isfinite(coef_far[:, j]) & (coef_far[:, j] >= lo) & (coef_far[:, j] <= hi)
            & np.isfinite(coef_sci[:, j]) & (coef_sci[:, j] >= lo) & (coef_sci[:, j] <= hi)
        )
        keep &= within
        print(
            f"Manual hard clip {cname}: [{lo:.3g}, {hi:.3g}] | kept {within.sum()}/{within.size} ({100.0 * within.mean():.1f}%)"
        )

    # OH per-row-mean MAD filter.  Per-column filtering on OH doesn't work:
    # many OH columns have near-zero medians so their column-MAD is tiny and
    # legitimate rows look like outliers.  Instead we filter on the same
    # scalar the diagnostic histogram shows -- per-row mean(|OH|) taken as
    # the max across the three arms so any arm with a runaway decomposition
    # marks the row.  Robust median+MAD on this scalar puts pathological
    # rows (mean |OH| ~ 1e6 to 1e15) trillions of MADs above the bulk while
    # leaving rows with real airglow variability untouched.
    if oh_kappa is not None and float(oh_kappa) > 0:
        oh_idx_local = np.array(
            [j for j, n in enumerate(coef_name_l) if n.startswith('oh_')],
            dtype=int)
        if oh_idx_local.size > 0:
            _oh_row_scalar = np.maximum.reduce([
                np.nanmean(np.abs(coef_near[:, oh_idx_local]), axis=1),
                np.nanmean(np.abs(coef_far[:, oh_idx_local]), axis=1),
                np.nanmean(np.abs(coef_sci[:, oh_idx_local]), axis=1),
            ]).astype(np.float64)
            _finite = np.isfinite(_oh_row_scalar)
            _med = float(np.nanmedian(_oh_row_scalar[_finite]))
            _mad = float(np.nanmedian(np.abs(_oh_row_scalar[_finite] - _med))) * 1.4826
            _mad = max(_mad, 1e-30)
            _threshold = _med + float(oh_kappa) * _mad
            oh_mask = _finite & (_oh_row_scalar <= _threshold)
            keep &= oh_mask
            print(
                f"OH per-row-mean MAD filter (kappa={oh_kappa:.1f}, "
                f"n_oh_coefs={oh_idx_local.size} per arm; "
                f"median={_med:.3g}, MAD={_mad:.3g}, threshold={_threshold:.3g}): "
                f"kept {int(oh_mask.sum())}/{len(oh_mask)} "
                f"({100.0 * oh_mask.mean():.1f}%)"
            )

    coef_concat = np.hstack([coef_near, coef_far, coef_sci]).astype(np.float32)
    kappa_mask = _kappa_sigma_row_mask(coef_concat, kappa=float(kappa), n_iter=int(kappa_iter))
    keep &= kappa_mask
    print(
        f"Kappa-sigma filter (kappa={kappa:.1f}): kept {kappa_mask.sum()}/{len(kappa_mask)} ({100.0 * kappa_mask.mean():.1f}%)"
    )

    # Thinning is applied LAST so the filter-fraction prints above reflect
    # counts against the full pre-thinning dataset.  It selects every N-th row
    # in the ORIGINAL row indexing; combined with the accumulated filter mask
    # via boolean AND, the result is the intersection.
    if int(thin_every_n) > 1:
        thin_mask = np.zeros(n0, dtype=bool)
        thin_mask[:: int(thin_every_n)] = True
        keep &= thin_mask
        print(f"Final thinning: every {int(thin_every_n)}-th of {n0} original rows; "
              f"combined with earlier filters, kept {int(keep.sum())} rows")
    else:
        print(f"Final thinning disabled: kept {int(keep.sum())}/{n0} rows after filters")

    if keep.sum() == 0:
        raise RuntimeError('Filtering removed all rows; relax thresholds.')

    out = {
        'coef_near': coef_near[keep],
        'coef_far': coef_far[keep],
        'coef_sci': coef_sci[keep],
        'ctx_near': ctx_near[keep],
        'ctx_far': ctx_far[keep],
        'ctx_sci': ctx_sci[keep],
        'coef_names': coef_names_local,
        'ctx_names': list(triplet_data['ctx_names']),
        'mask': keep,
        'row_index': np.asarray(triplet_data['row_index'])[keep],
    }
    if 'obstime_mjd' in triplet_data:
        out['obstime_mjd'] = np.asarray(triplet_data['obstime_mjd'], dtype=np.float64)[keep]
    for k in ('chi2_near', 'chi2_far', 'chi2_sci',
              'coef_err_near', 'coef_err_far', 'coef_err_sci',
              'sci_ra', 'sci_dec',
              'near_ra', 'near_dec', 'far_ra', 'far_dec'):
        if k in triplet_data:
            out[k] = np.asarray(triplet_data[k])[keep]
    if 'sci_radec_source' in triplet_data:
        out['sci_radec_source'] = triplet_data['sci_radec_source']

    print(
        f"Filtered triplet shapes: near={out['coef_near'].shape}, far={out['coef_far'].shape}, "
        f"sci={out['coef_sci'].shape}"
    )

    # Fold cyclic sin/cos pairs (az, moon_az, sun_az, moon_phase) back to
    # a single 0-360 degree axis per feature so histograms are readable.
    _display_names, _display_near = _decode_cyclic_context(out['ctx_names'], out['ctx_near'])
    _, _display_far = _decode_cyclic_context(out['ctx_names'], out['ctx_far'])
    _, _display_sci = _decode_cyclic_context(out['ctx_names'], out['ctx_sci'])

    # Distribution plot: one panel per context feature, three fields shown as
    # step lines (no fill).  Bin edges are computed per panel from that
    # feature's own min/max, so each panel has an independent x-range and
    # bin width -- which matters because the features span very different
    # ranges (degrees, airmass, sin/cos, van Rhijn factors).
    _hist_field_arrays = {
        'sky_near': _display_near,
        'sky_far': _display_far,
        'science': _display_sci,
    }
    _hist_field_colors = {
        'sky_near': '#1f77b4',
        'sky_far': '#9467bd',
        'science': '#2ca02c',
    }
    _hist_field_order = ['sky_near', 'sky_far', 'science']
    _hist_n_bins = 60

    context_order = list(_display_names)
    n_context = len(context_order)
    n_facet_cols = min(5, max(1, n_context))
    n_facet_rows = int(np.ceil(n_context / n_facet_cols))
    _n_panels_total = n_facet_rows * n_facet_cols

    fig_ctx_hist = make_subplots(
        rows=n_facet_rows,
        cols=n_facet_cols,
        subplot_titles=[c.replace('_', ' ') for c in context_order]
            + [''] * (_n_panels_total - n_context),
        horizontal_spacing=0.05,
        vertical_spacing=0.09,
    )

    for k, cname in enumerate(context_order):
        row = k // n_facet_cols + 1
        col = k % n_facet_cols + 1
        # sci_sep is 0 by construction for the science pointing; skip its delta at 0.
        panel_field_order = [f for f in _hist_field_order
                             if not (cname == 'sci_sep' and f == 'science')]
        per_field_vals = {}
        for f in panel_field_order:
            v = np.asarray(_hist_field_arrays[f][:, k], dtype=np.float64)
            per_field_vals[f] = v[np.isfinite(v)]
        combined = np.concatenate([per_field_vals[f] for f in panel_field_order])
        if combined.size < 2:
            continue
        lo = float(np.nanmin(combined))
        hi = float(np.nanmax(combined))
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            continue
        edges = np.linspace(lo, hi, _hist_n_bins + 1)
        step_x = np.repeat(edges, 2)[1:-1]
        for f in panel_field_order:
            v = per_field_vals[f]
            if v.size == 0:
                continue
            counts, _ = np.histogram(v, bins=edges)
            step_y = np.repeat(counts, 2)
            fig_ctx_hist.add_trace(
                go.Scattergl(
                    x=step_x,
                    y=step_y,
                    mode='lines',
                    line=dict(color=_hist_field_colors[f], width=1.4),
                    name=f,
                    legendgroup=f,
                    showlegend=(k == 0),
                    hovertemplate=f + ': %{y}<extra></extra>',
                ),
                row=row,
                col=col,
            )

    fig_ctx_hist.for_each_annotation(lambda a: a.update(font=dict(size=11)))
    fig_ctx_hist.update_xaxes(showline=True, mirror=True, ticks='outside',
                              ticklen=4, showticklabels=True)
    fig_ctx_hist.update_yaxes(showline=True, mirror=True, ticks='outside',
                              ticklen=4, showticklabels=True)
    fig_ctx_hist.update_layout(
        template='plotly_white',
        title='Context parameter distributions by field (rows used after all filters)',
        height=n_facet_rows * 220 + 200,
        width=min(2000, n_facet_cols * 340 + 140),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0),
        margin=dict(l=60, r=20, t=90, b=50),
    )
    fig_ctx_hist.show()

    return out


filtered_triplet = apply_triplet_filters(
    triplet,
    thin_every_n=1,
    chi2_qmax=90.0,
    chi2_min=0.0,
    chi2_max=10.0,
    hard_coef_bounds={'feo': (0.0, 10.01), 'atom_k': (0.0, 10.01)},
    kappa=6.0,
    kappa_iter=3,
    oh_kappa=4.0,
    oh_kappa_iter=3,
    exclude_field_regions=[LMC_EXCLUSION, SMC_EXCLUSION],
)

In [ ]:
# Coefficient distributions by field, pre- vs post-filtering.
# Row 1: raw triplet (all rows returned by the loader).
# Row 2: filtered_triplet (rows surviving field / chi2 / hard-clip /
# kappa-sigma). Companion to the context histogram above; three step-line
# traces per panel (sky_near / sky_far / science) so per-arm amplitude
# shifts stay visible in the same physical units the decomposition writes
# to disk. x-range is shared per column (computed from the pre-filter
# combined range) so the filtering effect is directly comparable.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_coef_names_hist = [str(n) for n in filtered_triplet['coef_names']]
_names_l_hist = [n.lower() for n in _coef_names_hist]
_moonzodi_hist_idx = np.array(
    [i for i, n in enumerate(_names_l_hist)
     if n.startswith(('moon_bs', 'moonzodi_bs'))], dtype=int)
_oh_hist_idx = np.array(
    [i for i, n in enumerate(_names_l_hist) if n.startswith('oh_')], dtype=int)


def _find_coef_col_hist(name):
    lname = name.lower()
    hits = [i for i, n in enumerate(_names_l_hist) if n == lname]
    return int(hits[0]) if hits else None


_individual_coef_names_hist = ['HO2', 'FeO', 'O2Ac']
_individual_coef_idx_hist = {n: _find_coef_col_hist(n)
                             for n in _individual_coef_names_hist}


def _per_row_scalar_hist(coef_mat, kind):
    _mat = np.asarray(coef_mat, dtype=np.float64)
    if kind == 'mean_moon_zodi':
        if _moonzodi_hist_idx.size == 0:
            return None
        return np.nanmean(_mat[:, _moonzodi_hist_idx], axis=1)
    if kind == 'mean_oh':
        if _oh_hist_idx.size == 0:
            return None
        return np.nanmean(_mat[:, _oh_hist_idx], axis=1)
    j = _individual_coef_idx_hist.get(kind)
    return _mat[:, j] if j is not None else None


_coef_panels_hist = [
    (f"mean moon+zodi (n={_moonzodi_hist_idx.size})", 'mean_moon_zodi'),
    (f"mean OH (n={_oh_hist_idx.size})", 'mean_oh'),
]
for _n in _individual_coef_names_hist:
    if _individual_coef_idx_hist.get(_n) is not None:
        _coef_panels_hist.append((_n, _n))

_hist_field_colors_coef = {
    'sky_near': '#1f77b4',
    'sky_far': '#9467bd',
    'science': '#2ca02c',
}
_hist_field_order_coef = ['sky_near', 'sky_far', 'science']
_hist_n_bins_coef = 60

_hist_source_arrays = {
    'pre-filter': {
        'sky_near': triplet['coef_near'],
        'sky_far': triplet['coef_far'],
        'science': triplet['coef_sci'],
        'n_rows': int(np.asarray(triplet['coef_near']).shape[0]),
    },
    'post-filter': {
        'sky_near': filtered_triplet['coef_near'],
        'sky_far': filtered_triplet['coef_far'],
        'science': filtered_triplet['coef_sci'],
        'n_rows': int(np.asarray(filtered_triplet['coef_near']).shape[0]),
    },
}
_hist_source_order = ['pre-filter', 'post-filter']

_n_panels_coef = len(_coef_panels_hist)
_n_rows_coef = len(_hist_source_order)
_n_cols_coef = _n_panels_coef

# Column headers only on the top row so the two rows read as a single grid.
_subplot_titles_coef = []
for _r in range(_n_rows_coef):
    for _label, _ in _coef_panels_hist:
        _subplot_titles_coef.append(_label if _r == 0 else '')

fig_coef_hist = make_subplots(
    rows=_n_rows_coef,
    cols=_n_cols_coef,
    subplot_titles=_subplot_titles_coef,
    horizontal_spacing=0.06,
    vertical_spacing=0.14,
)

# Per-row per-column full range: cover every finite value in that row's data
# so the pre-filter tail (including outlier decompositions) and the post-filter
# training range are each shown in full without percentile clipping.
for _r_i, _src in enumerate(_hist_source_order, start=1):
    _bundle = _hist_source_arrays[_src]
    _row_label = f"{_src} (n={_bundle['n_rows']})"
    for _c_i, (_label, _kind) in enumerate(_coef_panels_hist, start=1):
        _per_field_vals = {}
        for _f in _hist_field_order_coef:
            _v = _per_row_scalar_hist(_bundle[_f], _kind)
            if _v is None:
                continue
            _v = np.asarray(_v, dtype=np.float64)
            _per_field_vals[_f] = _v[np.isfinite(_v)]
        if not _per_field_vals:
            continue
        _combined = np.concatenate(list(_per_field_vals.values()))
        if _combined.size < 2:
            continue
        _lo = float(np.min(_combined))
        _hi = float(np.max(_combined))
        if not np.isfinite(_lo) or not np.isfinite(_hi) or _hi <= _lo:
            continue
        _edges = np.linspace(_lo, _hi, _hist_n_bins_coef + 1)
        _step_x = np.repeat(_edges, 2)[1:-1]
        _first_legend_slot = (_r_i == 1 and _c_i == 1)
        for _f in _hist_field_order_coef:
            _v = _per_field_vals.get(_f)
            if _v is None or _v.size == 0:
                continue
            _counts, _ = np.histogram(_v, bins=_edges)
            _step_y = np.repeat(_counts, 2)
            fig_coef_hist.add_trace(
                go.Scattergl(
                    x=_step_x,
                    y=_step_y,
                    mode='lines',
                    line=dict(color=_hist_field_colors_coef[_f], width=1.4),
                    name=_f,
                    legendgroup=_f,
                    showlegend=_first_legend_slot,
                    hovertemplate=_f + ': %{y}<extra></extra>',
                ),
                row=_r_i,
                col=_c_i,
            )
    fig_coef_hist.update_yaxes(
        title_text=_row_label, row=_r_i, col=1,
        title_font=dict(size=10))

fig_coef_hist.for_each_annotation(lambda a: a.update(font=dict(size=11)))
# exponentformat='e' + separatethousands=False: force 1e+06 style tick labels
# instead of plotly's default SI suffix (which prints 1M / 1B for the outlier
# pre-filter tail and hides the actual magnitude).
fig_coef_hist.update_xaxes(showline=True, mirror=True, ticks='outside',
                            ticklen=4, showticklabels=True,
                            exponentformat='e', separatethousands=False)
fig_coef_hist.update_yaxes(showline=True, mirror=True, ticks='outside',
                            ticklen=4, showticklabels=True,
                            exponentformat='e', separatethousands=False)
fig_coef_hist.update_layout(
    template='plotly_white',
    title=('Sky-decomposition coefficient distributions by field, '
           'pre- (top) vs post- (bottom) filtering. '
           'x-range per row: full min-max of each row\'s finite values '
           '(pre-filter shows the outlier tail; post-filter shows only rows '
           'that enter training).'),
    height=_n_rows_coef * 260 + 200,
    width=min(2000, _n_cols_coef * 380 + 140),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0.0),
    margin=dict(l=90, r=20, t=110, b=50),
)
fig_coef_hist.show()


In [ ]:
# ECLIPTIC-CTX-V3: augment triplet ctx with PER-ARM ecliptic coordinates.
# Adds three features to each arm's ctx (following the same convention as
# `alt`, `az_sin`, `az_cos`: shared feature names, arm-specific values):
#   ecl_beta_deg -- raw ecliptic latitude in degrees (-90..+90). Zodi
#     amplitude peaks at beta=0 (ecliptic plane), so cos(beta) drives the
#     scaling; the MLP learns that from the raw latitude directly.
#   ecl_lon_sin/cos -- cyclic embedding of ecliptic longitude.
# If the triplet builder didn't attach near/far RA/Dec, this cell reads
# the meta FITS directly and fills them in, so per-arm ecliptic works
# even when the builder / filter cells were left with stale kernel state.
import numpy as np
from astropy.io import fits
from astropy.coordinates import SkyCoord, BarycentricMeanEcliptic
from astropy.table import Table
import astropy.units as u

ECLIPTIC_FEATURE_NAMES = ['ecl_beta_deg', 'ecl_lon_sin', 'ecl_lon_cos']
# Older feature names we strip on sight so re-running is idempotent
# (V1 = sci-broadcast sin/cos-lat, V2 = per-arm sin/cos-lat).
_LEGACY_ECLIPTIC_NAMES = [
    'sci_ecl_lat_sin', 'sci_ecl_lat_cos',
    'sci_ecl_lon_sin', 'sci_ecl_lon_cos',
    'ecl_lat_sin', 'ecl_lat_cos',
]


def _ecliptic_features(ra_deg, dec_deg):
    _coords = SkyCoord(ra=np.asarray(ra_deg) * u.deg,
                        dec=np.asarray(dec_deg) * u.deg, frame='icrs')
    _ecl = _coords.transform_to(BarycentricMeanEcliptic())
    _lat_deg = np.asarray(_ecl.lat.degree, dtype=np.float32)
    _lon_rad = np.asarray(_ecl.lon.radian, dtype=np.float64)
    return np.stack([_lat_deg,
                     np.sin(_lon_rad).astype(np.float32),
                     np.cos(_lon_rad).astype(np.float32)],
                    axis=1).astype(np.float32)


def _fill_arm_radec_from_meta_fits(triplet, stem):
    """Attach near/far RA/Dec to triplet by reading the meta FITS directly.
    Bypass path for a stale triplet-builder buffer."""
    _row_idx = np.asarray(triplet['row_index'], dtype=int)
    with fits.open(f'{stem}_meta_only.fits') as _hdul:
        _meta = Table(_hdul['META'].data)
    _meta_up = {c.upper(): c for c in _meta.colnames}
    for _pref, _rakey, _deckey in (
        ('near', 'SKY_NEAR_RA', 'SKY_NEAR_DEC'),
        ('far', 'SKY_FAR_RA', 'SKY_FAR_DEC'),
    ):
        if f'{_pref}_ra' in triplet and f'{_pref}_dec' in triplet:
            continue
        if _rakey in _meta_up and _deckey in _meta_up:
            triplet[f'{_pref}_ra'] = np.asarray(
                _meta[_meta_up[_rakey]], dtype=np.float64)[_row_idx]
            triplet[f'{_pref}_dec'] = np.asarray(
                _meta[_meta_up[_deckey]], dtype=np.float64)[_row_idx]
            print(f'  filled {_pref}_ra/{_pref}_dec from meta FITS')


def _augment_triplet_with_ecliptic(triplet, force=True):
    """Append per-arm ecliptic features to ctx_near/ctx_far/ctx_sci in place.
    force=True (default): always strip pre-existing ecliptic features and
    recompute from RA/Dec, so re-running never leaves stale values in place."""
    _names = list(triplet.get('ctx_names', []))
    _to_strip = set(_LEGACY_ECLIPTIC_NAMES)
    if force:
        _to_strip.update(ECLIPTIC_FEATURE_NAMES)
    _strip_idx = [i for i, _n in enumerate(_names) if _n in _to_strip]
    if _strip_idx:
        _keep = [i for i in range(len(_names)) if i not in _strip_idx]
        for _arm in ('ctx_near', 'ctx_far', 'ctx_sci'):
            triplet[_arm] = np.asarray(triplet[_arm],
                                       dtype=np.float32)[:, _keep]
        _names = [_names[i] for i in _keep]
        triplet['ctx_names'] = _names
        print(f'  stripped {len(_strip_idx)} pre-existing ecliptic feature(s) '
              f'so per-arm augment recomputes cleanly')
    if 'sci_ra' not in triplet or 'sci_dec' not in triplet:
        raise RuntimeError('triplet missing sci_ra/sci_dec; cannot compute '
                            'ecliptic features')
    if 'near_ra' not in triplet or 'far_ra' not in triplet:
        _stem = globals().get('_DECOMP_STEM',
                              'lvmsframe_median_stack_1.2.1_p40_p70')
        try:
            _fill_arm_radec_from_meta_fits(triplet, _stem)
        except Exception as _e:
            print(f'  meta FITS fallback failed: {type(_e).__name__}: {_e}')
    _sci_feats = _ecliptic_features(triplet['sci_ra'], triplet['sci_dec'])
    for _arm_prefix, _ctx_key in (('near', 'ctx_near'), ('far', 'ctx_far'),
                                    ('sci', 'ctx_sci')):
        if _arm_prefix == 'sci':
            _feats = _sci_feats
        elif f'{_arm_prefix}_ra' in triplet and f'{_arm_prefix}_dec' in triplet:
            _feats = _ecliptic_features(triplet[f'{_arm_prefix}_ra'],
                                          triplet[f'{_arm_prefix}_dec'])
        else:
            print(f'  WARNING: {_arm_prefix}_ra/{_arm_prefix}_dec unavailable; '
                  f'falling back to sci ecliptic for ctx_{_arm_prefix}')
            _feats = _sci_feats
        _prev = np.asarray(triplet[_ctx_key], dtype=np.float32)
        triplet[_ctx_key] = np.concatenate([_prev, _feats],
                                            axis=1).astype(np.float32)
    triplet['ctx_names'] = _names + ECLIPTIC_FEATURE_NAMES


_augment_triplet_with_ecliptic(filtered_triplet, force=True)
print(f'filtered_triplet ctx augmented: n_ctx={len(filtered_triplet["ctx_names"])} '
      f'(added {ECLIPTIC_FEATURE_NAMES}, per-arm ecliptic)')


In [ ]:
# Per-coefficient effective wavelength and effective extinction.
#
# Both are needed because extinction varies strongly across the LVM range: the
# 'mesospheric' group alone spans OI 5577, Na D and the OH/O2 bands, and even
# within the OH Meinel system k varies by nearly a factor of two.
#
INPUT_FITS_FOR_BASIS = 'lvmsframe_median_stack_1.2.1_p40_p70_every10.fits'
WAVELENGTH_CACHE = Path('coef_wavelengths_basis_v3.npz')  # p40_p70 decomposition (2026-08-15); uses v3 cache based on p70 improved continuumnuum basis
USE_FITTED_EXTINCTION = True   # False -> generic LCO stellar curve only

import time as _time
_t_start = _time.perf_counter()


def _lap(label):
    global _t_start
    now = _time.perf_counter()
    print(f'    [{label}: {now - _t_start:.1f} s]')
    _t_start = now


# --- 1. basis-derived wavelengths + B^2-weighted effective extinction -----
# One reconstruction call per coefficient. Reads the actual basis (built from
# the same PMD populations, LSF, and wavelength grid the decomposition uses at
# fit time), so the returned centroid already carries the mesopause rotational
# Boltzmann factor for OH and the correct blend structure across overlapping
# bands. There is no auxiliary population-model route to cross-check against;
# it was retired 2026-08-04 (closes methods sect 11.5).
_group_indices_preview = _build_group_indices(filtered_triplet['coef_names'])
coef_wavelengths_basis = None
coef_k_eff_basis = None
try:
    with fits.open(INPUT_FITS_FOR_BASIS) as _hdul:
        _ext_names = [h.name for h in _hdul]
        if 'WAVE' not in _ext_names:
            raise KeyError(f'{INPUT_FITS_FOR_BASIS} has no WAVE extension')
        _wave_ref = np.asarray(_hdul['WAVE'].data, dtype=np.float64)
        _lsf_name = 'LSF_SCI' if 'LSF_SCI' in _ext_names else (
            'LSF' if 'LSF' in _ext_names else None)
        if _lsf_name is None:
            raise KeyError(f'{INPUT_FITS_FOR_BASIS} has no LSF_SCI or LSF extension')
        _lsf_ref = np.asarray(_hdul[_lsf_name].data, dtype=np.float64)
    if _wave_ref.ndim > 1:
        _wave_ref = _wave_ref[0]
    if _lsf_ref.ndim > 1:
        _lsf_ref = _lsf_ref[0]

    coef_wavelengths_basis, coef_k_eff_basis = coef_wavelengths_from_basis(
        coef_names=filtered_triplet['coef_names'],
        wave=_wave_ref,
        lsf_sigma=_lsf_ref / 2.35,
        n_spline_knots=25,
        cache_path=WAVELENGTH_CACHE,
        only_indices=None,   # every coefficient; disk cache absorbs the one-shot cost
        return_k_eff=True,
        verbose=True,
    )
except Exception as exc:
    print(f'Basis wavelengths unavailable ({type(exc).__name__}: {exc}); '
          f'falling back to name-token and group defaults.')

_lap('basis wavelengths + B^2 k_eff')

# --- 2. resolve one wavelength per coefficient ---------------------------
_grp_sizes = {g: int(idx.size) for g, idx in _group_indices_preview.items()}
_expected_sizes = {'moon': 29, 'continuum': 3, 'mesospheric': 403,
                   'atomic': 3, 'ionospheric': 4}
print(f'Coefficient group sizes: {_grp_sizes} (total {sum(_grp_sizes.values())})')
if _grp_sizes != _expected_sizes:
    print(f'  NOTE: expected {_expected_sizes} for the lvmsframe_median_stack_1.2.1_p70 '
          f'product; a different input product can legitimately shift the '
          f'OH_### count (basis coverage of the OH bands), the Moon_bs## / '
          f'MoonZodi_bs### count (n_spline_knots + 4), or the moon variant '
          f'(scattered-moon spline vs Moon+Zodi correction). If the diff '
          f'is elsewhere, extend COEF_SCHEMA to cover the new family.')
coef_wavelengths_a, coef_wavelength_source = resolve_coef_wavelengths_a(
    filtered_triplet['coef_names'],
    group_indices=_group_indices_preview,
    basis_wavelengths_a=coef_wavelengths_basis,
    verbose=True,
)
filtered_triplet['coef_wavelengths_a'] = coef_wavelengths_a
_lap('wavelength resolution')

# --- 5. contexts must be physical before any geometry is evaluated -------
for _label, _ctx in (('near', filtered_triplet['ctx_near']),
                     ('far', filtered_triplet['ctx_far']),
                     ('sci', filtered_triplet['ctx_sci'])):
    assert_context_is_physical(_ctx, filtered_triplet['ctx_names'])
    print(f'context[{_label}] verified physical (van Rhijn columns consistent with alt)')

# --- 6. fit the EFFECTIVE extinction from the simultaneous sky pairs ------
# The tabulated LCO curve is a stellar curve. Airglow is a quasi-uniform
# extended source, so photons scattered out of the beam are largely replaced by
# photons scattered in from adjacent lines of sight, and the effective
# attenuation is well below the stellar value. Fitting it from the near/far
# pairs sidesteps having to model that, and simultaneously absorbs the
# airglow-versus-stellar difference, the site aerosol level and any residual
# error in the assumed layer height.
extinction_fit_table = pd.DataFrame()
if USE_FITTED_EXTINCTION:
    extinction_fit_table = fit_effective_extinction(
        coef_near=filtered_triplet['coef_near'],
        coef_far=filtered_triplet['coef_far'],
        ctx_near=filtered_triplet['ctx_near'],
        ctx_far=filtered_triplet['ctx_far'],
        ctx_names=filtered_triplet['ctx_names'],
        group_indices=_group_indices_preview,
        coef_wavelengths_a=coef_wavelengths_a,
        n_wavelength_bins=8,
        verbose=True,
    )

coef_extinction_k, coef_extinction_source = resolve_coef_extinction_k(
    filtered_triplet['coef_names'],
    coef_wavelengths_a,
    _group_indices_preview,
    fit_table=extinction_fit_table if USE_FITTED_EXTINCTION else None,
    clip_to_generic=True,
    coef_basis_k_generic=coef_k_eff_basis,
    verbose=True,
)
filtered_triplet['coef_extinction_k'] = coef_extinction_k
_lap('effective-extinction fit')


In [ ]:
# Shared ML utilities for coefficient prediction models
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

class RobustScaler:
    def fit(self, x):
        x = np.asarray(x, dtype=np.float32)
        self.med_ = np.nanmedian(x, axis=0)
        q25 = np.nanpercentile(x, 25, axis=0)
        q75 = np.nanpercentile(x, 75, axis=0)
        iqr = q75 - q25
        self.scale_ = np.where(iqr > 1e-8, iqr, 1.0).astype(np.float32)
        return self

    def transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return (x - self.med_) / self.scale_

    def inverse_transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return x * self.scale_ + self.med_

def _set_reproducibility(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def split_indices(n, train_frac=0.8, val_frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]
    return train_idx, val_idx, test_idx

def split_indices_by_night(obstime_mjd, train_frac=0.8, val_frac=0.1, seed=42):
    t = np.asarray(obstime_mjd, dtype=np.float64).reshape(-1)
    if t.size == 0:
        raise ValueError('obstime_mjd is empty')
    if not np.isfinite(t).all():
        raise ValueError('obstime_mjd contains non-finite values')

    night_id = np.floor(t - 0.5).astype(int)
    unique_nights = np.unique(night_id)
    rng = np.random.default_rng(seed)
    shuffled_nights = unique_nights.copy()
    rng.shuffle(shuffled_nights)

    n_nights = shuffled_nights.size
    n_train = int(train_frac * n_nights)
    n_val = int(val_frac * n_nights)
    train_nights = shuffled_nights[:n_train]
    val_nights = shuffled_nights[n_train:n_train + n_val]
    test_nights = shuffled_nights[n_train + n_val:]

    train_idx = np.flatnonzero(np.isin(night_id, train_nights))
    val_idx = np.flatnonzero(np.isin(night_id, val_nights))
    test_idx = np.flatnonzero(np.isin(night_id, test_nights))
    return train_idx, val_idx, test_idx

def _moon_phase_deg_from_ctx(filtered, arm='sci'):
    """Per-row moon phase in degrees [0, 360), reading whichever encoding is present."""
    names = [str(n) for n in filtered['ctx_names']]
    ctx = np.asarray(filtered[f'ctx_{arm}'], dtype=np.float64)
    if 'moon_phase' in names:
        return ctx[:, names.index('moon_phase')]
    s = names.index('moon_phase_sin')
    c = names.index('moon_phase_cos')
    return np.rad2deg(np.arctan2(ctx[:, s], ctx[:, c])) % 360.0


def split_indices_by_moon_phase(
    obstime_mjd,
    moon_phase,
    train_frac=0.8,
    val_frac=0.1,
    seed=42,
    n_bins=10,
):
    """Night-level split stratified by moon phase.

    Rows are grouped by night (floor(mjd - 0.5)). Each night is assigned a
    representative moon phase (median across its exposures). Nights are sorted
    by that phase and cut into ``n_bins`` equal-count quantiles. Within each
    quantile the nights are shuffled and split into train / val / test with
    the requested fractions rounded per-bin, so val and test each cover the
    full moon-phase range roughly uniformly. Whole nights stay together, so
    within-night airglow autocorrelation is not leaked across splits.
    """
    t = np.asarray(obstime_mjd, dtype=np.float64).reshape(-1)
    phase = np.asarray(moon_phase, dtype=np.float64).reshape(-1)
    if t.size == 0:
        raise ValueError('obstime_mjd is empty')
    if t.shape != phase.shape:
        raise ValueError('obstime_mjd and moon_phase must have the same shape')
    if not np.isfinite(t).all():
        raise ValueError('obstime_mjd contains non-finite values')
    if not np.isfinite(phase).all():
        raise ValueError('moon_phase contains non-finite values')

    night_id = np.floor(t - 0.5).astype(int)
    unique_nights = np.unique(night_id)
    n_nights = unique_nights.size

    # Per-night representative moon phase (median across the night's exposures).
    night_phase = np.empty(n_nights, dtype=np.float64)
    for k, nid in enumerate(unique_nights):
        night_phase[k] = np.median(phase[night_id == nid])

    rng = np.random.default_rng(seed)
    # Random tie-break so nights sharing a phase aren't ordered by their id.
    tie_break = rng.random(n_nights)
    order = np.lexsort((tie_break, night_phase))
    sorted_nights = unique_nights[order]

    n_bins_eff = max(1, min(int(n_bins), n_nights))
    bin_edges = np.linspace(0, n_nights, n_bins_eff + 1, dtype=int)
    test_frac = max(0.0, 1.0 - float(train_frac) - float(val_frac))

    train_list, val_list, test_list = [], [], []
    for k in range(n_bins_eff):
        lo, hi = int(bin_edges[k]), int(bin_edges[k + 1])
        block = sorted_nights[lo:hi].copy()
        b = block.size
        if b == 0:
            continue
        # Random assignment within the phase bin.
        perm = rng.permutation(b)
        block = block[perm]
        if b >= 3:
            n_val = max(1, int(round(float(val_frac) * b)))
            n_test = max(1, int(round(test_frac * b)))
            if n_val + n_test >= b:
                # Leave at least one train per bin.
                n_val = max(1, min(n_val, b - 2))
                n_test = max(1, min(n_test, b - 1 - n_val))
        elif b == 2:
            n_val, n_test = 1, 0
        else:
            n_val, n_test = 0, 0
        val_list.extend(block[:n_val].tolist())
        test_list.extend(block[n_val:n_val + n_test].tolist())
        train_list.extend(block[n_val + n_test:].tolist())

    train_nights = np.asarray(train_list, dtype=int)
    val_nights = np.asarray(val_list, dtype=int)
    test_nights = np.asarray(test_list, dtype=int)

    train_idx = np.flatnonzero(np.isin(night_id, train_nights))
    val_idx = np.flatnonzero(np.isin(night_id, val_nights))
    test_idx = np.flatnonzero(np.isin(night_id, test_nights))
    return train_idx, val_idx, test_idx


def _moon_bs_indices_from_names(coef_names_local):
    """Column indices of the moon spline coefficient block within a coef-name list.

    Matches both the historical ``Moon_bs##`` naming (scattered-moon spline
    amplitude, ``lsf_surface_iterative`` and earlier variants) and the
    ``MoonZodi_bs###``  naming introduced by the moon-zodi decomposition
    (physical Moon+Zodi predictor multiplied by a spline correction).
    """
    return np.asarray([i for i, n in enumerate(coef_names_local)
                       if str(n).lower().startswith(("moon_bs", "moonzodi_bs"))], dtype=int)
def _row_spline_roughness(vals):
    """Root-mean-second-difference of a coefficient vector; a spline-noise diagnostic."""
    vals = np.asarray(vals, dtype=np.float64)
    if vals.size < 3:
        return np.nan
    d2 = vals[2:] - 2.0 * vals[1:-1] + vals[:-2]
    return float(np.sqrt(np.nanmean(d2 * d2)))


def _metric_row(y_true, y_pred, model_name, *,
                sigma=None, group_indices=None, floor_by_group=None):
    """Distributional summary of per-coefficient error (eRMSE), MAE, and Pearson r.

    Terminology (also used in §9 of the top notebook doc):
      * eRMSE  -- ensemble RMSE, i.e. per-coefficient RMSE aggregated
        over the ensemble of rows (axis=0).  ``mean_eRMSE`` /
        ``median_eRMSE`` are the mean / median across the per-coefficient
        RMSE vector.  Literature name: per-target RMSE.
      * sRMSE  -- spectral RMSE, per-row RMSE aggregated over the
        coefficient axis (axis=1).  Emitted per row by
        ``weighted_rmse_per_row``.  Literature name: per-sample RMSE.
      * pRMSE  -- pixel RMSE, per-row RMSE aggregated over the pixel
        axis of the reconstructed spectrum.  Emitted per row by
        ``pixel_wrmse_per_row``.  Distinct from sRMSE because it lives
        in flux space rather than coefficient space.
      * eWRMSE / sWRMSE / pWRMSE -- weighted counterparts (mirror the
        trainer's coef_err-weighted loss so the coefficient-space numbers
        stay comparable to the training objective; pWRMSE uses per-pixel
        sigma from the FLUX_SIGMA_TOTAL HDU or the on-the-fly propagator).

    When ``sigma`` (per-element decomposition COEF_ERR, same shape as
    ``y_true``) is provided along with ``group_indices`` and
    ``floor_by_group``, weighted counterparts are also reported:
    ``mean_eWRMSE`` / ``median_eWRMSE`` (per-column WRMSE with per-group
    sigma floor, aggregated), and ``total_eWRMSE`` (row + column pooled).
    """
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    mae = np.mean(np.abs(y_pred - y_true), axis=0)
    corr = []
    for j in range(y_true.shape[1]):
        x = y_true[:, j]
        y = y_pred[:, j]
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            corr.append(np.nan)
        else:
            corr.append(float(np.corrcoef(x, y)[0, 1]))
    corr = np.asarray(corr)
    out = {
        'model': model_name,
        'mean_eRMSE': float(np.nanmean(rmse)),
        'median_eRMSE': float(np.nanmedian(rmse)),
        'mean_eMAE': float(np.nanmean(mae)),
        'mean_corr': float(np.nanmean(corr)),
        'median_corr': float(np.nanmedian(corr)),
    }
    if sigma is not None and group_indices is not None and floor_by_group is not None:
        wrmse_col, total_wrmse = _per_column_wrmse(
            y_true, y_pred, sigma, group_indices, floor_by_group,
        )
        out['mean_eWRMSE'] = float(np.nanmean(wrmse_col))
        out['median_eWRMSE'] = float(np.nanmedian(wrmse_col))
        out['total_eWRMSE'] = float(total_wrmse)
    return out


def _per_column_wrmse(y_true, y_pred, sigma, group_indices, floor_by_group):
    """Per-column WRMSE with per-group sigma floor + total WRMSE.

    Weights mirror the trainer: sigma is floored per group at
    ``floor_by_group[g] * median_col(sigma)``, per-column normalised to
    E[w]=1 so the mean/median WRMSE stay on the same scale as RMSE.
    Returns (per_col_wrmse, total_wrmse).
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    sigma = np.asarray(sigma, dtype=np.float64)
    resid = y_pred - y_true
    finite = np.where(np.isfinite(sigma) & (sigma > 0.0), sigma, np.nan)
    median_col = np.nanmedian(finite, axis=0)
    median_col = np.where(np.isfinite(median_col) & (median_col > 0.0),
                          median_col, 1.0)
    floor_rel_col = np.full(sigma.shape[1], 0.05, dtype=np.float64)
    for g, idx in group_indices.items():
        idx = np.asarray(idx, dtype=int)
        if idx.size:
            floor_rel_col[idx] = float(floor_by_group.get(g, 0.05))
    floor_col = floor_rel_col * median_col
    sigma_eff = np.where(np.isfinite(sigma) & (sigma > 0.0),
                         sigma, floor_col[None, :])
    sigma_eff = np.maximum(sigma_eff, floor_col[None, :])
    w = 1.0 / (sigma_eff ** 2)
    wcm = np.mean(w, axis=0)
    wcm = np.where(wcm > 0.0, wcm, 1.0)
    w = w / wcm[None, :]
    f = np.isfinite(w) & np.isfinite(resid)
    num_col = np.sum(w * resid ** 2 * f, axis=0)
    den_col = np.sum(w * f, axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        per_col = np.sqrt(np.where(den_col > 0.0, num_col / den_col, np.nan))
    tot_num = float(np.sum(w[f] * resid[f] ** 2))
    tot_den = float(np.sum(w[f]))
    total = float(np.sqrt(tot_num / max(tot_den, 1e-30)))
    return per_col, total


In [ ]:
# --- WRMSE per-row helper (used by R^2 -> WRMSE map/plot cells) --------
# Companion to cell 12's _per_column_wrmse: computes per-row weighted RMSE
# in coefficient space, using the trainer's per-group sigma floor and
# per-column weight normalisation.  Small stand-alone cell so downstream
# reporting/plotting cells never depend on the compressor/trainer being
# loaded first.
import numpy as np


def weighted_rmse_per_row(y_true, y_pred, sigma, group_indices, floor_by_group):
    """Return per-row WRMSE across all coefficients.

    ``sigma`` is native COEF_ERR (same shape as y_true/y_pred).
    Weights = 1 / max(sigma, floor_g * median_col(sigma))**2 with per-column
    mean normalisation over all rows (matches the aggregate reporting).
    Rows with no finite weighted residuals return NaN.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    sigma = np.asarray(sigma, dtype=np.float64)
    resid = y_pred - y_true
    finite = np.where(np.isfinite(sigma) & (sigma > 0.0), sigma, np.nan)
    median_col = np.nanmedian(finite, axis=0)
    median_col = np.where(np.isfinite(median_col) & (median_col > 0.0),
                          median_col, 1.0)
    floor_rel_col = np.full(sigma.shape[1], 0.05, dtype=np.float64)
    for g, idx in group_indices.items():
        idx = np.asarray(idx, dtype=int)
        if idx.size:
            floor_rel_col[idx] = float(floor_by_group.get(g, 0.05))
    floor_col = floor_rel_col * median_col
    sigma_eff = np.where(np.isfinite(sigma) & (sigma > 0.0),
                         sigma, floor_col[None, :])
    sigma_eff = np.maximum(sigma_eff, floor_col[None, :])
    w = 1.0 / (sigma_eff ** 2)
    wcm = np.mean(w, axis=0)
    wcm = np.where(wcm > 0.0, wcm, 1.0)
    w = w / wcm[None, :]
    f = np.isfinite(w) & np.isfinite(resid)
    num = np.sum(w * resid ** 2 * f, axis=1)
    den = np.sum(w * f, axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.sqrt(np.where(den > 0.0, num / den, np.nan))


In [ ]:
# --- Pixel-space per-row WRMSE helpers ---------------------------------
# Companion to `weighted_rmse_per_row` (cell 13) but in flux/pixel space,
# used by every "spectrum-space RMSE" reporting site downstream (§9).
#
# Sigma sourcing (documented in §9 for future decompositions):
#   1. If the decomposition FITS carries a `FLUX_SIGMA_TOTAL` HDU (planned
#      output of the LSF-aware sigma propagator in `sky_decomp/fit.py`
#      -- see `SkyDecompBase._components_sigma_from_coef_err`), the loader
#      picks it up directly.  Shape must match FLUX_* (n_row, n_pix).
#   2. If only `COEF_ERR` is present (current schema), callers derive
#      sigma on-the-fly via `reconstruct_component_spectra(coef=..., coef_err=...)`
#      which returns the same LSF-propagated per-pixel sigma; both routes
#      produce identical arrays.
#   3. If neither is present, pass sigma=None and the WRMSE reduces to
#      plain per-pixel RMSE with a per-row median floor (no error).
import numpy as np


PIXEL_SIGMA_FLOOR_REL = 0.05
"""Relative floor on the per-pixel sigma used by `pixel_wrmse_per_row`.

Floor per row = `PIXEL_SIGMA_FLOOR_REL * median(|flux_true|)`; guards low-noise
regions where the propagator can return near-zero sigma from making WRMSE
diverge.  0.05 matches the moon/continuum/ionospheric floor used in
coefficient-space WRMSE (§7)."""


PIXEL_SIGMA_HDU_NAMES = ("FLUX_SIGMA_TOTAL", "FLUX_SIGMA")
"""HDU names the loader accepts, in preference order.

`FLUX_SIGMA_TOTAL` is the canonical name for the future LSF-aware sigma
propagator output; `FLUX_SIGMA` is a shorter alias for external files."""


def load_pixel_sigma_if_available(fits_path, row_indices=None):
    """Return per-pixel sigma array from *fits_path* or `None`.

    Returns None (does NOT raise) when neither HDU name in
    `PIXEL_SIGMA_HDU_NAMES` is present, so upstream reporting can degrade
    gracefully to plain RMSE.  When ``row_indices`` is given, only those rows
    are returned in that order.
    """
    from astropy.io import fits
    try:
        with fits.open(str(fits_path)) as hdul:
            _names = [str(getattr(h, "name", "")).upper() for h in hdul]
            for _wanted in PIXEL_SIGMA_HDU_NAMES:
                if _wanted in _names:
                    _data = np.asarray(hdul[_wanted].data, dtype=np.float64)
                    if row_indices is not None:
                        _data = _data[np.asarray(row_indices, dtype=int)]
                    return _data
    except FileNotFoundError:
        pass
    return None


def pixel_wrmse_per_row(flux_pred, flux_true, sigma_pix=None,
                        floor_rel=PIXEL_SIGMA_FLOOR_REL):
    """Per-row pixel-space WRMSE.

    Parameters
    ----------
    flux_pred, flux_true : (n_row, n_pix) or (n_pix,) arrays
        Predicted and observed spectra, same shape.  A 1-D input is
        broadcast to (1, n_pix).
    sigma_pix : (n_row, n_pix) array or None
        Per-pixel 1sigma.  None triggers floor-only weighting (uniform
        weights per row --> plain per-row RMSE with median-based floor).
    floor_rel : float
        Relative floor factor -- floor per row = ``floor_rel *
        median(|flux_true|)`` per row.

    Returns
    -------
    (n_row,) array of WRMSE values (NaN where a row has no usable pixels).
    """
    flux_pred = np.atleast_2d(np.asarray(flux_pred, dtype=np.float64))
    flux_true = np.atleast_2d(np.asarray(flux_true, dtype=np.float64))
    if flux_pred.shape != flux_true.shape:
        raise ValueError(f"shape mismatch: pred {flux_pred.shape} vs true {flux_true.shape}")
    resid = flux_pred - flux_true
    true_abs = np.abs(flux_true)
    median_row = np.nanmedian(true_abs, axis=1, keepdims=True)
    median_row = np.where(np.isfinite(median_row) & (median_row > 0.0),
                          median_row, 1.0)
    floor_row = float(floor_rel) * median_row
    if sigma_pix is None:
        sigma_eff = np.broadcast_to(floor_row, resid.shape).copy()
    else:
        s = np.atleast_2d(np.asarray(sigma_pix, dtype=np.float64))
        if s.shape != resid.shape:
            raise ValueError(f"sigma_pix shape {s.shape} != flux shape {resid.shape}")
        sigma_eff = np.where(np.isfinite(s) & (s > 0.0), s, floor_row)
        sigma_eff = np.maximum(sigma_eff, floor_row)
    w = 1.0 / (sigma_eff ** 2)
    # Row-wise normalisation: mean(w) = 1 per row so WRMSE stays on the
    # same scale as unweighted per-row RMSE.
    wrm = np.nanmean(w, axis=1, keepdims=True)
    wrm = np.where(wrm > 0.0, wrm, 1.0)
    w = w / wrm
    f = np.isfinite(w) & np.isfinite(resid)
    num = np.sum(w * resid ** 2 * f, axis=1)
    den = np.sum(w * f, axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.sqrt(np.where(den > 0.0, num / den, np.nan))


def pixel_wrmse_pointwise(flux_pred, flux_true, sigma_pix=None,
                          floor_rel=PIXEL_SIGMA_FLOOR_REL):
    """Per-pixel weighted squared-residual (before per-row reduction).

    Returns ``(w * resid**2)`` (n_row, n_pix), useful for the residual-band
    plots in the batch-RMSE cell.
    """
    flux_pred = np.atleast_2d(np.asarray(flux_pred, dtype=np.float64))
    flux_true = np.atleast_2d(np.asarray(flux_true, dtype=np.float64))
    resid = flux_pred - flux_true
    true_abs = np.abs(flux_true)
    median_row = np.nanmedian(true_abs, axis=1, keepdims=True)
    median_row = np.where(np.isfinite(median_row) & (median_row > 0.0),
                          median_row, 1.0)
    floor_row = float(floor_rel) * median_row
    if sigma_pix is None:
        sigma_eff = np.broadcast_to(floor_row, resid.shape).copy()
    else:
        s = np.atleast_2d(np.asarray(sigma_pix, dtype=np.float64))
        sigma_eff = np.where(np.isfinite(s) & (s > 0.0), s, floor_row)
        sigma_eff = np.maximum(sigma_eff, floor_row)
    w = 1.0 / (sigma_eff ** 2)
    wrm = np.nanmean(w, axis=1, keepdims=True)
    wrm = np.where(wrm > 0.0, wrm, 1.0)
    return w / wrm * resid ** 2


## Training the Residual Emissivity Transfer Network

In [ ]:
# =============================================================================
# Precompute per-row per-band per-coefficient basis flux integrals -- 2026-08-18
# v3 rewrite: fix (b) + optimizations 1a (persistent FITS I/O), 1b (shared-LSF
# line-species hoist), 2 (matmul band-mean), 4 (uncompressed npz), n_workers=8.
#
# For each row i, band b, coefficient j:
#     I[i, b, j] = mean over lambda in b of B_ij(lambda)
# where B_ij is the physical basis function j evaluated at wavelength lambda
# for row i's geometry.  For MoonZodi_bs coefficients the k-th spline weight
# maps to column k of mats['moon'] which the model returns as the combined
# moon+zodi design block (moon_matrix + zodi_matrix).
# =============================================================================

import hashlib
import re
import time
from pathlib import Path
from joblib import Parallel, delayed
from astropy.io import fits

BASIS_INTEGRAL_BANDS = [
    ("B_cont",   4000.0, 4500.0),
    ("B_OH",     5000.0, 5400.0),
    ("G_cont",   5450.0, 5700.0),
    ("G_OH",     5800.0, 6300.0),
    ("R_cont",   7000.0, 7300.0),
    ("R_OH",     7300.0, 9000.0),
    ("NIR_cont", 9600.0, 9900.0),
]
BASIS_INTEGRAL_VERSION = "v4"  # v4 (2026-08-18): pass physical_to_fit_flux_scale so mats['moon'] is in the same units as sci COEF

# mats['moon'] is the combined moon+zodi design block; there is no 'zodi' key.
BASIS_COMPONENT_GROUPS = {
    "moon+zodi": ("moon",),
    "diffuse":   ("diffuse",),
    "lines":     ("oh", "atom", "orc", "o2"),
}


def _basis_cache_hash(coef_names):
    """Cache invalidation key: schema-sensitive, geometry-insensitive."""
    payload = "|".join([
        ",".join(str(n) for n in coef_names),
        ",".join(f"{b[0]}:{b[1]:.1f}:{b[2]:.1f}" for b in BASIS_INTEGRAL_BANDS),
        BASIS_INTEGRAL_VERSION,
    ])
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def _hardcode_moon_zodi_mapping(coef_names):
    """MoonZodi_bs[k] / Moon_bs[k] -> [('moon', k)].  In the MoonZodi model
    mats['moon'] is the combined (moon+zodi) design block (fit.py::_matrix_bundle,
    moon_zodi_lsf_surface_iterative.py::_assemble_refined_matrices), so a single
    ('moon', k) entry captures both contributions.  The unit-vector probe used
    for the other coefs fails here because comps['moon'] = moon_matrix.T @ coef
    is zero when moon_predictor = 0 (dark-time rows).
    """
    mapping = [None] * len(coef_names)
    n_hard = 0
    for j, name in enumerate(coef_names):
        m = re.match(r'^(moonzodi_bs|moon_bs)(\d+)$', str(name).lower())
        if m is None:
            continue
        k = int(m.group(2))
        mapping[j] = [('moon', k)]
        n_hard += 1
    return mapping, n_hard


def _probe_one_row(model, mats, n_coef, mapping):
    """Fill unmapped entries of `mapping` using unit-vector probes on this row."""
    _mat_keys = [k for k in mats.keys() if k not in ('sigma', 'sigma_total', 'total')]
    newly_mapped = 0
    for j in range(n_coef):
        if mapping[j] is not None and len(mapping[j]) > 0:
            continue
        unit = np.zeros(n_coef, dtype=np.float64)
        unit[j] = 1.0
        comps = model._components_from_coef(unit, mats)
        entries = []
        for c in _mat_keys:
            if c not in comps:
                continue
            arr = np.asarray(comps[c], dtype=np.float64)
            if not np.any(np.abs(arr) > 1e-30):
                continue
            dots = np.abs(mats[c] @ arr)
            k = int(np.argmax(dots))
            entries.append((c, k))
        if entries:
            mapping[j] = entries
            newly_mapped += 1
    return newly_mapped


def _discover_coef_to_mat_mapping(filtered, sci_fits_path, wave_ref,
                                    n_spline_knots, detector_lsf_per_row,
                                    row_indices, physical_to_fit_flux_scale,
                                    max_probe_rows=20):
    """Probe multiple rows if needed until every coef has an entry.
    Moon+Zodi coefs are hard-coded first (probe fails for them in dark time).
    """
    n_coef_local = int(filtered['coef_near'].shape[1])
    coef_names_local = list(filtered['coef_names'])
    mapping, n_hard = _hardcode_moon_zodi_mapping(coef_names_local)
    print(f"[basis integrals] hard-coded moon+zodi mapping for {n_hard} coefs "
          f"(MoonZodi_bs / Moon_bs -> mats['moon'])")
    _probe_model = SkyDecompMoonZodiLSFSurfaceIterative(
        wave=wave_ref, lsf_sigma=1.0,
        physical_to_fit_flux_scale=float(physical_to_fit_flux_scale),
        n_spline_knots=int(n_spline_knots),
    )
    for probe_i in range(min(max_probe_rows, len(row_indices))):
        row = int(row_indices[probe_i])
        try:
            _mz = load_moon_zodi_state(str(sci_fits_path), row)
            _lsf = load_lsf_surface_state(str(sci_fits_path), row)
            _probe_model._install_prediction(
                _mz.observation,
                np.asarray(detector_lsf_per_row[probe_i], dtype=np.float64),
            )
            _probe_model._set_lsf_state(_lsf)
            _mats = _probe_model._assemble_refined_matrices()
        except Exception:
            continue
        newly = _probe_one_row(_probe_model, _mats, n_coef_local, mapping)
        n_mapped = sum(1 for m in mapping if m)
        print(f"[basis integrals] probe row {row}: +{newly} newly mapped, "
              f"{n_mapped}/{n_coef_local} total")
        if n_mapped == n_coef_local:
            break
    for j in range(n_coef_local):
        if mapping[j] is None:
            mapping[j] = []
    return mapping


# -----------------------------------------------------------------------------
# Opt 2: band-mean via one BLAS matmul per component matrix.
# -----------------------------------------------------------------------------
def _build_band_weight_matrix(wave_ref):
    """W (n_bands, n_pix) float64; band_means = M @ W.T gives (n_g, n_bands)."""
    n_pix = int(np.asarray(wave_ref).size)
    n_bands = len(BASIS_INTEGRAL_BANDS)
    W = np.zeros((n_bands, n_pix), dtype=np.float64)
    for bi, (_, lo, hi) in enumerate(BASIS_INTEGRAL_BANDS):
        mask = (wave_ref >= lo) & (wave_ref < hi)
        n_in = int(mask.sum())
        if n_in > 0:
            W[bi, mask] = 1.0 / n_in
    return W


def _integrate_row_matmul(mats, coef_mapping, W_bands, shared_bands=None):
    """Given per-row mats, coef mapping, and W_bands, return (n_bands, n_coef).
    per_mat_bands[c] has shape (n_g_c, n_bands).  If `shared_bands` is given,
    reuse those entries instead of recomputing (opt 1b for LSF-shared rows).
    """
    n_bands = int(W_bands.shape[0])
    n_coef = len(coef_mapping)
    out = np.zeros((n_bands, n_coef), dtype=np.float32)
    per_mat_bands = {}
    for c, M in mats.items():
        if c in ('sigma', 'sigma_total', 'total'):
            continue
        if shared_bands is not None and c in shared_bands:
            per_mat_bands[c] = shared_bands[c]  # (n_g_c, n_bands) float32
            continue
        per_mat_bands[c] = (np.asarray(M, dtype=np.float64) @ W_bands.T).astype(np.float32)
    for j, entries in enumerate(coef_mapping):
        if not entries:
            continue
        acc = np.zeros(n_bands, dtype=np.float32)
        for (c, k) in entries:
            if c in per_mat_bands and k < per_mat_bands[c].shape[0]:
                acc += per_mat_bands[c][k, :]
        out[:, j] = acc
    return out


# -----------------------------------------------------------------------------
# Opt 1b: precompute shared line-species band-means when LSF is row-shared.
# -----------------------------------------------------------------------------
def _lsf_state_signature(lsf_state):
    """SHA of concatenated per-channel coefficient + knot arrays."""
    h = hashlib.sha256()
    for _ch in sorted(lsf_state.coefficients.keys()):
        h.update(np.ascontiguousarray(lsf_state.coefficients[_ch]).tobytes())
        h.update(np.ascontiguousarray(lsf_state.knot_vectors[_ch]).tobytes())
    return h.hexdigest()[:16]


def _build_shared_lsf_line_bands(sci_fits_path, wave_ref, n_spline_knots,
                                   detector_lsf_per_row, row_indices, W_bands,
                                   physical_to_fit_flux_scale, n_check=5):
    """If LSF surface states agree across a sample of rows AND detector LSF is
    row-shared, precompute per-band means of the row-independent line species
    (oh, atom, orc, diffuse).  Returns dict[c -> (n_g_c, n_bands) float32] or None.
    """
    # Detector LSF row-share check.
    _dlsf = np.asarray(detector_lsf_per_row)
    if _dlsf.ndim != 2:
        print("[basis integrals] opt 1b: unexpected detector_lsf shape; disabling.")
        return None
    _dlsf_row0 = _dlsf[0]
    for _pi in range(1, min(n_check, _dlsf.shape[0])):
        if not np.allclose(_dlsf[_pi], _dlsf_row0, rtol=1e-12, atol=1e-12):
            print(f"[basis integrals] opt 1b: detector LSF differs at probe row "
                  f"{_pi}; falling back to per-row assembly.")
            return None

    # LSF-surface-state row-share check.
    try:
        _lsf0 = load_lsf_surface_state(str(sci_fits_path), int(row_indices[0]))
    except Exception as _e:
        print(f"[basis integrals] opt 1b: cannot load LSF state at row 0 ({_e}); disabling.")
        return None
    _sig0 = _lsf_state_signature(_lsf0)
    for _pi in range(1, min(n_check, len(row_indices))):
        try:
            _lsf_i = load_lsf_surface_state(str(sci_fits_path), int(row_indices[_pi]))
        except Exception as _e:
            print(f"[basis integrals] opt 1b: LSF load failed at probe row {_pi} ({_e}); disabling.")
            return None
        if _lsf_state_signature(_lsf_i) != _sig0:
            print(f"[basis integrals] opt 1b: LSF surface differs at probe row "
                  f"{row_indices[_pi]}; falling back to per-row assembly.")
            return None

    # LSF and detector LSF are shared -> assemble mats once at row 0 and cache
    # the row-independent line-species band means.
    _model = SkyDecompMoonZodiLSFSurfaceIterative(
        wave=wave_ref, lsf_sigma=1.0,
        physical_to_fit_flux_scale=float(physical_to_fit_flux_scale),
        n_spline_knots=int(n_spline_knots),
    )
    try:
        _mz0 = load_moon_zodi_state(str(sci_fits_path), int(row_indices[0]))
    except Exception as _e:
        print(f"[basis integrals] opt 1b: cannot load MZ state at row 0 ({_e}); disabling.")
        return None
    _model._install_prediction(_mz0.observation, np.asarray(_dlsf_row0, dtype=np.float64))
    _model._set_lsf_state(_lsf0)
    _mats0 = _model._assemble_refined_matrices()
    _shared = {}
    for _c in ('oh', 'atom', 'orc', 'diffuse'):
        if _c in _mats0:
            _shared[_c] = (np.asarray(_mats0[_c], dtype=np.float64) @ W_bands.T).astype(np.float32)
    print(f"[basis integrals] opt 1b enabled: LSF row-shared across {n_check} "
          f"probe rows -> hoisted {list(_shared.keys())} per-band means "
          f"(saves {len(_shared)}/6 mat convolutions per row)")
    return _shared


# -----------------------------------------------------------------------------
# Opt 1a: persistent per-worker FITS I/O via monkey-patched fits.open.
# -----------------------------------------------------------------------------
class _NoCloseHDULCtx:
    """Return this instead of a fresh HDUList so the load functions' `with`
    blocks don't close our persistent handle."""
    def __init__(self, hdul):
        self.hdul = hdul
    def __enter__(self):
        return self.hdul
    def __exit__(self, *args):
        return False


def _basis_worker(payload):
    """Worker: open the sci FITS once with memmap, monkey-patch fits.open so
    the library state loaders reuse it, and integrate a batch of rows."""
    (row_indices, sci_fits_path, wave_ref, n_spline_knots,
     detector_lsf_batch, coef_mapping, W_bands, n_coef,
     shared_line_bands, physical_to_fit_flux_scale) = payload

    import numpy as _np
    from astropy.io import fits as _fits

    _model = SkyDecompMoonZodiLSFSurfaceIterative(
        wave=wave_ref, lsf_sigma=1.0,
        physical_to_fit_flux_scale=float(physical_to_fit_flux_scale),
        n_spline_knots=int(n_spline_knots),
    )

    _sci_path_str = str(sci_fits_path)
    _persistent_hdul = _fits.open(_sci_path_str, memmap=True)
    _original_open = _fits.open

    def _patched_open(name, *args, **kwargs):
        if str(name) == _sci_path_str:
            return _NoCloseHDULCtx(_persistent_hdul)
        return _original_open(name, *args, **kwargs)

    _fits.open = _patched_open  # only patched inside this worker process

    try:
        n_bands = int(W_bands.shape[0])
        out = _np.zeros((len(row_indices), n_bands, n_coef), dtype=_np.float32)

        _o2_cube = None
        if 'VECTOR_O2' in _persistent_hdul:
            _o2_raw = _persistent_hdul['VECTOR_O2'].data
            if _o2_raw is not None and getattr(_o2_raw, 'ndim', 0) == 2:
                _o2_cube = _o2_raw  # memmap view

        for local_i, row_idx in enumerate(row_indices):
            try:
                _mz_state = load_moon_zodi_state(_sci_path_str, int(row_idx))
                _lsf_state = load_lsf_surface_state(_sci_path_str, int(row_idx))
                _o2_vec = None
                if _o2_cube is not None and int(row_idx) < _o2_cube.shape[0]:
                    _row = _np.asarray(_o2_cube[int(row_idx)], dtype=_np.float64)
                    if _np.isfinite(_row).any() and float(_np.nansum(_np.abs(_row))) > 0:
                        _o2_vec = _row
                _model._install_prediction(
                    _mz_state.observation,
                    _np.asarray(detector_lsf_batch[local_i], dtype=_np.float64),
                )
                _model._set_lsf_state(_lsf_state)
                _mats = _model._assemble_refined_matrices()
                if _o2_vec is not None:
                    _mats['o2'] = _o2_vec[None, :]
                out[local_i] = _integrate_row_matmul(
                    _mats, coef_mapping, W_bands, shared_bands=shared_line_bands,
                )
            except Exception:
                out[local_i] = _np.nan
        return out
    finally:
        _fits.open = _original_open
        try:
            _persistent_hdul.close()
        except Exception:
            pass


# -----------------------------------------------------------------------------
# Main entry point.
# -----------------------------------------------------------------------------
def precompute_basis_integrals(filtered, sci_fits_path, wave_ref, n_spline_knots,
                                 detector_lsf_per_row, physical_to_fit_flux_scale,
                                 n_workers=1, force=False):
    """Compute (n_rows, n_bands, n_coef) basis-integral cube for all filtered rows.

    Caches to <sci_fits_path>.basis_integrals.npz keyed by SHA256(schema).
    Uncompressed .npz for fast write (~330 MB for 11k rows; small vs compute cost).
    """
    coef_names = list(filtered['coef_names'])
    row_indices = np.asarray(filtered['row_index'], dtype=int)
    n_rows = int(row_indices.size)
    n_bands = len(BASIS_INTEGRAL_BANDS)
    n_coef = len(coef_names)
    schema_hash = _basis_cache_hash(coef_names)

    cache_path = Path(sci_fits_path).with_suffix('.basis_integrals.npz')

    if cache_path.exists() and not force:
        try:
            cached = np.load(cache_path, allow_pickle=True)
            if (str(cached['schema_hash']) == schema_hash
                    and int(cached['n_rows']) == n_rows):
                print(f"[basis integrals] loaded cache: {cache_path.name}  "
                      f"shape={tuple(cached['integrals'].shape)}  hash={schema_hash}")
                return dict(
                    integrals=np.asarray(cached['integrals'], dtype=np.float32),
                    bands=list(BASIS_INTEGRAL_BANDS),
                    coef_names=list(coef_names),
                    schema_hash=schema_hash,
                    row_indices=np.asarray(cached['row_indices'], dtype=int),
                )
        except Exception as _e:
            print(f"[basis integrals] cache load failed ({_e}); recomputing.")

    W_bands = _build_band_weight_matrix(wave_ref)

    print("[basis integrals] discovering coef -> component mapping...")
    print(f"[basis integrals] using physical_to_fit_flux_scale={physical_to_fit_flux_scale:.3e}")
    _t0 = time.perf_counter()
    coef_mapping = _discover_coef_to_mat_mapping(
        filtered, sci_fits_path, wave_ref, n_spline_knots, detector_lsf_per_row,
        row_indices, physical_to_fit_flux_scale=physical_to_fit_flux_scale,
    )
    _t1 = time.perf_counter()
    _n_mapped = sum(1 for m in coef_mapping if m)
    print(f"[basis integrals] mapping discovered in {_t1 - _t0:.1f}s "
          f"({_n_mapped}/{n_coef} coefs mapped to component mats)")

    print("[basis integrals] probing LSF row-sharing for opt 1b hoist...")
    shared_line_bands = _build_shared_lsf_line_bands(
        sci_fits_path, wave_ref, n_spline_knots, detector_lsf_per_row,
        row_indices, W_bands,
        physical_to_fit_flux_scale=physical_to_fit_flux_scale,
    )

    print(f"[basis integrals] computing over {n_rows} rows with n_workers={n_workers}...")
    _t2 = time.perf_counter()

    if n_workers <= 1:
        integrals = np.zeros((n_rows, n_bands, n_coef), dtype=np.float32)
        _serial_model = SkyDecompMoonZodiLSFSurfaceIterative(
            wave=wave_ref, lsf_sigma=1.0,
            physical_to_fit_flux_scale=float(physical_to_fit_flux_scale),
            n_spline_knots=int(n_spline_knots),
        )
        _persistent = fits.open(str(sci_fits_path), memmap=True)
        _o2_cube = None
        if 'VECTOR_O2' in _persistent:
            _o2_raw = _persistent['VECTOR_O2'].data
            if _o2_raw is not None and getattr(_o2_raw, 'ndim', 0) == 2:
                _o2_cube = _o2_raw
        _original_open = fits.open

        def _patched_open(name, *args, **kwargs):
            if str(name) == str(sci_fits_path):
                return _NoCloseHDULCtx(_persistent)
            return _original_open(name, *args, **kwargs)

        fits.open = _patched_open
        try:
            _last_report = _t2
            for i, row_idx in enumerate(row_indices):
                row_idx = int(row_idx)
                try:
                    _mz = load_moon_zodi_state(str(sci_fits_path), row_idx)
                    _lsf = load_lsf_surface_state(str(sci_fits_path), row_idx)
                    _o2 = None
                    if _o2_cube is not None and row_idx < _o2_cube.shape[0]:
                        _row = np.asarray(_o2_cube[row_idx], dtype=np.float64)
                        if np.isfinite(_row).any() and float(np.nansum(np.abs(_row))) > 0:
                            _o2 = _row
                    _serial_model._install_prediction(
                        _mz.observation,
                        np.asarray(detector_lsf_per_row[i], dtype=np.float64),
                    )
                    _serial_model._set_lsf_state(_lsf)
                    _mats = _serial_model._assemble_refined_matrices()
                    if _o2 is not None:
                        _mats['o2'] = _o2[None, :]
                    integrals[i] = _integrate_row_matmul(
                        _mats, coef_mapping, W_bands, shared_bands=shared_line_bands,
                    )
                except Exception:
                    integrals[i] = np.nan
                _now = time.perf_counter()
                if _now - _last_report >= 15.0:
                    print(f"[basis integrals]   {i+1}/{n_rows} rows  "
                          f"({(i+1) / n_rows * 100:.1f}%,  "
                          f"eta {(n_rows - i - 1) * (_now - _t2) / max(i+1, 1):.0f}s)")
                    _last_report = _now
        finally:
            fits.open = _original_open
            _persistent.close()
    else:
        # joblib loky backend uses cloudpickle to ship the kernel-defined worker.
        row_batches = np.array_split(np.arange(n_rows), int(n_workers))
        payloads = []
        for _batch_rel_indices in row_batches:
            _batch_row_idx = row_indices[_batch_rel_indices]
            _batch_lsf = detector_lsf_per_row[_batch_rel_indices]
            payloads.append((
                [int(r) for r in _batch_row_idx],
                str(sci_fits_path),
                wave_ref,
                int(n_spline_knots),
                _batch_lsf,
                coef_mapping,
                W_bands,
                int(n_coef),
                shared_line_bands,
                float(physical_to_fit_flux_scale),
            ))
        results = Parallel(n_jobs=int(n_workers), backend='loky', verbose=1)(
            delayed(_basis_worker)(_p) for _p in payloads
        )
        integrals = np.concatenate(results, axis=0).astype(np.float32)

    _t3 = time.perf_counter()
    print(f"[basis integrals] compute done in {_t3 - _t2:.1f}s "
          f"({(_t3 - _t2) / max(n_rows, 1) * 1000:.1f} ms/row)")

    _nan_rows = int(np.sum(~np.isfinite(integrals).all(axis=(1, 2))))
    if _nan_rows:
        print(f"[basis integrals] WARNING: {_nan_rows}/{n_rows} rows have NaN "
              f"integrals (reconstruction failed).")

    # Opt 4: uncompressed savez -- ~330 MB but ~5-10 s faster to write.
    np.savez(
        cache_path,
        integrals=integrals,
        row_indices=row_indices,
        schema_hash=schema_hash,
        n_rows=n_rows,
    )
    print(f"[basis integrals] cached to {cache_path.name}")

    return dict(
        integrals=integrals,
        bands=list(BASIS_INTEGRAL_BANDS),
        coef_names=coef_names,
        schema_hash=schema_hash,
        row_indices=row_indices,
    )


# -----------------------------------------------------------------------------
# Run precompute for filtered_triplet using the sci-arm decomposition FITS.
# The bands, detector-LSF grid, wavelength grid, and moon-zodi geometry are all
# consumed by the RETN training loop via `basis_integrals_cache`.
# -----------------------------------------------------------------------------
RUN_BASIS_INTEGRALS = True
BASIS_INTEGRALS_N_WORKERS = 8  # v3: bumped from 4

if RUN_BASIS_INTEGRALS:
    _every10_stem = "lvmsframe_median_stack_1.2.1_p40_p70"
    _every10_input = f"{_every10_stem}.fits"
    _every10_sci   = f"{_every10_stem}_decomp_sci{_DECOMP_SUFFIX}.fits"

    with fits.open(_every10_input, memmap=False) as _hd:
        _wave_full = np.asarray(_hd['WAVE'].data, dtype=np.float64)
        _lsf_full  = np.asarray(_hd['LSF_SCI'].data, dtype=np.float64)

    _row_indices = np.asarray(filtered_triplet['row_index'], dtype=int)
    if _wave_full.ndim == 1:
        _wave_ref = _wave_full
    else:
        _wave_ref = np.asarray(_wave_full[int(_row_indices[0])], dtype=np.float64)
    if _lsf_full.ndim == 1:
        _detector_lsf_per_row = np.broadcast_to(_lsf_full, (len(_row_indices), _lsf_full.size)).copy()
    else:
        _detector_lsf_per_row = np.asarray(_lsf_full[_row_indices], dtype=np.float64)

    _first_mz = load_moon_zodi_state(_every10_sci, int(_row_indices[0]))
    _mz_n_interior = max(1, len(_first_mz.correction_knots) - 8)
    _mz_scale = float(_first_mz.physical_to_fit_flux_scale)

    basis_integrals_cache = precompute_basis_integrals(
        filtered=filtered_triplet,
        sci_fits_path=_every10_sci,
        wave_ref=_wave_ref,
        n_spline_knots=_mz_n_interior,
        detector_lsf_per_row=_detector_lsf_per_row,
        physical_to_fit_flux_scale=_mz_scale,
        n_workers=BASIS_INTEGRALS_N_WORKERS,
        force=False,
    )

    print()
    print(f"basis_integrals_cache['integrals'] shape: {basis_integrals_cache['integrals'].shape}")
    print(f"bands: {[b[0] for b in basis_integrals_cache['bands']]}")
    print(f"schema_hash: {basis_integrals_cache['schema_hash']}")

    _row0 = basis_integrals_cache['integrals'][0]  # (n_bands, n_coef)
    for _bi, (_name, _lo, _hi) in enumerate(basis_integrals_cache['bands']):
        print(f"  band {_name:10s} [{_lo:.0f}-{_hi:.0f}]: "
              f"row0 sum={_row0[_bi].sum():.3e}  max={_row0[_bi].max():.3e}  "
              f"n_nonzero={int((np.abs(_row0[_bi]) > 0).sum())}/{_row0.shape[1]}")



In [ ]:
# ============================================================================
# Residual Emissivity Transfer Network (RETN) -- 2026-08-18
#
# First-principles redesign for the moon+zodi era where coefficients encode
# residuals on top of geometry-aware physical predictions. The old
# DualEncoderGroupHeadMLPCompressed was optimized when moon coefficients had
# 5-10x larger variance (moon-only model absorbed geometry). Post moon+zodi,
# em_near ~= em_sci in intrinsic emissivity space, and a "copy-near-normalized"
# baseline beats the current network by ~11% at flux level. RETN exploits this:
#
#   em_pred = em_near + residual_head(em_near, em_far, ctx_near, ctx_far, ctx_sci)
#   coef_pred = ReLU(em_pred * scale_sci)
#
# Residual head is zero-initialized on the FINAL layer, so an untrained RETN
# already produces the copy-near baseline. Training only learns non-zero
# corrections when they demonstrably help.
#
# See "post-moonzodi-architecture-plan" session memory for the full derivation.
# ============================================================================

import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset


class ResidualEmissivityTransferNet(nn.Module):
    """Residual emissivity transfer network. See top-of-cell comment."""

    def __init__(self, n_coef, n_ctx, group_indices,
                 arm_encoder_dims=(512, 256),
                 ctx_dims=(64,),
                 trunk_dims=(256, 128),
                 group_head_hidden=128):
        super().__init__()
        self.n_coef = int(n_coef)
        self.n_ctx = int(n_ctx)
        self.group_indices = group_indices

        # Arm encoder (shared for near and far): em+ctx -> latent
        self.arm_encoder = _make_mlp(
            in_dim=self.n_coef + self.n_ctx,
            hidden_dims=arm_encoder_dims,
            out_dim=arm_encoder_dims[-1],
        )

        # Ctx encoder for sci pointing
        self.ctx_encoder = _make_mlp(
            in_dim=self.n_ctx,
            hidden_dims=ctx_dims,
            out_dim=ctx_dims[-1],
        )

        # Trunk combines: h_near + h_far + h_ctx_sci + ctx_diff(sci-near) + ctx_diff(sci-far)
        # The ctx-difference features give the continuum residual head a direct handle
        # on geometry-driven residuals (van Rhijn / airmass differences between arms).
        arm_dim = arm_encoder_dims[-1]
        ctx_dim = ctx_dims[-1]
        trunk_in = 2 * arm_dim + ctx_dim + 2 * self.n_ctx
        self.trunk = _make_mlp(
            in_dim=trunk_in,
            hidden_dims=trunk_dims,
            out_dim=trunk_dims[-1],
        )

        # Per-group residual heads, all with zero-init final layer.
        self.group_heads = nn.ModuleDict()
        for g, idx in group_indices.items():
            n_g = int(len(idx))
            head = nn.Sequential(
                nn.Linear(trunk_dims[-1], group_head_hidden),
                nn.GELU(),
                nn.Linear(group_head_hidden, n_g),
            )
            with torch.no_grad():
                head[-1].weight.zero_()
                head[-1].bias.zero_()
            self.group_heads[str(g)] = head

        # Register group indices as buffers so .to(device) picks them up.
        for g, idx in group_indices.items():
            self.register_buffer(f"gidx_{g}", torch.as_tensor(idx, dtype=torch.long))

    def forward(self, em_near_std, em_far_std, ctx_near, ctx_far, ctx_sci):
        # All inputs are pre-standardized (em by em_scaler, ctx by ctx_scaler).
        h_near = self.arm_encoder(torch.cat([em_near_std, ctx_near], dim=1))
        h_far = self.arm_encoder(torch.cat([em_far_std, ctx_far], dim=1))
        h_ctx = self.ctx_encoder(ctx_sci)
        ctx_diff_near = ctx_sci - ctx_near
        ctx_diff_far = ctx_sci - ctx_far
        h = torch.cat([h_near, h_far, h_ctx, ctx_diff_near, ctx_diff_far], dim=1)
        h = self.trunk(h)

        # Assemble the standardized residual by writing per-group outputs into slots.
        residual_std = torch.zeros_like(em_near_std)
        for g, head in self.group_heads.items():
            r_g = head(h)
            gidx = getattr(self, f"gidx_{g}")
            residual_std.index_copy_(1, gidx, r_g)

        # Baseline is copy-near in the SAME standardized space so addition is meaningful.
        em_pred_std = em_near_std + residual_std
        return em_pred_std, residual_std


def _make_mlp(in_dim, hidden_dims, out_dim, activation=nn.GELU):
    layers = []
    prev = int(in_dim)
    for h in hidden_dims:
        layers.append(nn.Linear(prev, int(h)))
        layers.append(activation())
        prev = int(h)
    if int(out_dim) != prev:
        layers.append(nn.Linear(prev, int(out_dim)))
    return nn.Sequential(*layers)


def _bias_component_membership(coef_names):
    """Route each coefficient to a bias-loss component group.

    Groups mirror the SSFR cell:
      moon+zodi -> MoonZodi_bs / Moon_bs (drive both moon and zodi flux)
      diffuse   -> HO2 / FeO / O2Ac
      lines     -> OH_### / O2_b## / ATOM_*
    Returns dict[group] = np.ndarray[int] of coefficient indices.
    """
    membership = {"moon+zodi": [], "diffuse": [], "lines": []}
    for j, name in enumerate(coef_names):
        lname = str(name).lower()
        if lname.startswith(("moonzodi_bs", "moon_bs")):
            membership["moon+zodi"].append(j)
        elif lname in ("ho2", "feo", "o2ac"):
            membership["diffuse"].append(j)
        else:
            membership["lines"].append(j)
    return {g: np.asarray(idx, dtype=int) for g, idx in membership.items() if idx}


# -----------------------------------------------------------------------------
# Training
# -----------------------------------------------------------------------------

def train_retn(filtered, group_indices, geom_kwargs,
               split_indices=None,
               n_epochs=60, batch_size=256, lr=7e-4,
               arm_encoder_dims=(512, 256), ctx_dims=(64,),
               trunk_dims=(256, 128), group_head_hidden=128,
               weight_decay=3.3e-5, grad_clip=1.0, patience=10,
               seed=42, positivity_weight=0.1,
               use_coef_err_weights=True, coef_err_sigma_floor_rel=None,
               basis_integrals_cache=None, bias_loss_weight=0.0,
               curv_loss_weight=0.0,
               curv_mode_weights=(1.0, 1.0, 1.0),
               smooth_loss_weight=0.0,
               device='auto'):
    """Train one RETN model.  Returns (model, artifacts_dict_for_this_seed)."""
    _set_reproducibility(seed)

    ctx_near = np.asarray(filtered['ctx_near'], dtype=np.float64)
    ctx_far = np.asarray(filtered['ctx_far'], dtype=np.float64)
    ctx_sci = np.asarray(filtered['ctx_sci'], dtype=np.float64)
    coef_near = np.asarray(filtered['coef_near'], dtype=np.float64)
    coef_far = np.asarray(filtered['coef_far'], dtype=np.float64)
    coef_sci = np.asarray(filtered['coef_sci'], dtype=np.float64)
    n_coef = coef_sci.shape[1]

    if split_indices is not None:
        train_idx, val_idx, test_idx = split_indices
    else:
        _moon_phase_col = _moon_phase_deg_from_ctx(filtered)
        train_idx, val_idx, test_idx = split_indices_by_moon_phase(
            filtered['obstime_mjd'], _moon_phase_col, seed=seed)
    train_idx = np.asarray(train_idx, dtype=int)
    val_idx = np.asarray(val_idx, dtype=int)

    # Compute emissivity = coef / airglow_geometry_scale.  For non-airglow
    # coefficients (moon+zodi) the scale is 1, so em == coef.
    scale_near = airglow_geometry_scale(ctx_near, **geom_kwargs)
    scale_far = airglow_geometry_scale(ctx_far, **geom_kwargs)
    scale_sci = airglow_geometry_scale(ctx_sci, **geom_kwargs)
    em_near = coef_near / scale_near
    em_far = coef_far / scale_far
    em_sci = coef_sci / scale_sci


    # Per-coef scaler on emissivity, fit on pooled near/far/sci train rows.
    em_scaler = RobustScaler().fit(np.vstack([
        em_near[train_idx], em_far[train_idx], em_sci[train_idx],
    ]).astype(np.float32))
    em_near_std = np.clip(em_scaler.transform(em_near.astype(np.float32)), -25.0, 25.0)
    em_far_std = np.clip(em_scaler.transform(em_far.astype(np.float32)), -25.0, 25.0)
    em_sci_std = np.clip(em_scaler.transform(em_sci.astype(np.float32)), -25.0, 25.0)

    ctx_scaler = RobustScaler().fit(np.vstack([
        ctx_near[train_idx], ctx_far[train_idx], ctx_sci[train_idx],
    ]).astype(np.float32))
    ctx_near_n = np.clip(ctx_scaler.transform(ctx_near), -25.0, 25.0).astype(np.float32)
    ctx_far_n = np.clip(ctx_scaler.transform(ctx_far), -25.0, 25.0).astype(np.float32)
    ctx_sci_n = np.clip(ctx_scaler.transform(ctx_sci), -25.0, 25.0).astype(np.float32)

    # Per-element weights from decomposition COEF_ERR translated to em space.
    if use_coef_err_weights and 'coef_err_sci' in filtered:
        _cerr = np.asarray(filtered['coef_err_sci'], dtype=np.float64)
        _cerr = np.where(np.isfinite(_cerr) & (_cerr > 0.0), _cerr, np.nan)
        sigma_em = _cerr / scale_sci
        # Standardize sigma the same way em was (RobustScaler multiplies by scale_)
        _em_scale = np.asarray(em_scaler.scale_, dtype=np.float64)
        sigma_em_std = sigma_em / np.where(_em_scale > 0, _em_scale, 1.0)[None, :]
        # Fill NaN by per-column median (in std space).
        _col_median = np.nanmedian(sigma_em_std, axis=0)
        _col_median = np.where(np.isfinite(_col_median) & (_col_median > 0.0),
                               _col_median, 1.0)
        sigma_em_std = np.where(np.isfinite(sigma_em_std) & (sigma_em_std > 0.0),
                                sigma_em_std, _col_median[None, :])
        # Per-column floor 5% of median finite sigma
        sigma_em_std = np.maximum(sigma_em_std, 0.05 * _col_median[None, :])
        w_pe_np = 1.0 / (sigma_em_std ** 2)
        # Normalize per column so E_train[w] = 1
        _wcm = np.mean(w_pe_np[train_idx], axis=0)
        _wcm = np.where(_wcm > 0.0, _wcm, 1.0)
        w_pe_np = (w_pe_np / _wcm[None, :]).astype(np.float32)
    else:
        w_pe_np = np.ones_like(em_sci, dtype=np.float32)

    # Basis-integrals bias-loss setup (see cell 14 for the precompute).
    _use_bias = (basis_integrals_cache is not None and float(bias_loss_weight) > 0.0)
    if _use_bias:
        _basis_int_np = np.asarray(basis_integrals_cache['integrals'], dtype=np.float32)
        if _basis_int_np.shape[0] != coef_sci.shape[0]:
            raise RuntimeError(
                f"basis_integrals row count {_basis_int_np.shape[0]} does not match "
                f"filtered rows {coef_sci.shape[0]}. Rebuild the cache.")
        _bias_membership = _bias_component_membership(list(filtered['coef_names']))
        _scale_sci_np = np.asarray(scale_sci, dtype=np.float32)
        _coef_sci_np = coef_sci.astype(np.float32)
        print(f"[RETN bias loss] enabled: weight={bias_loss_weight}, "
              f"n_bands={_basis_int_np.shape[1]}, "
              f"groups={ {g: int(idx.size) for g, idx in _bias_membership.items()} }")
    else:
        _basis_int_np = None
        _bias_membership = None
        _scale_sci_np = None
        _coef_sci_np = None

    _train_tensors = [
        torch.from_numpy(em_near_std[train_idx].astype(np.float32)),
        torch.from_numpy(em_far_std[train_idx].astype(np.float32)),
        torch.from_numpy(ctx_near_n[train_idx]),
        torch.from_numpy(ctx_far_n[train_idx]),
        torch.from_numpy(ctx_sci_n[train_idx]),
        torch.from_numpy(w_pe_np[train_idx]),
        torch.from_numpy(em_sci_std[train_idx].astype(np.float32)),
    ]
    _val_tensors = [
        torch.from_numpy(em_near_std[val_idx].astype(np.float32)),
        torch.from_numpy(em_far_std[val_idx].astype(np.float32)),
        torch.from_numpy(ctx_near_n[val_idx]),
        torch.from_numpy(ctx_far_n[val_idx]),
        torch.from_numpy(ctx_sci_n[val_idx]),
        torch.from_numpy(w_pe_np[val_idx]),
        torch.from_numpy(em_sci_std[val_idx].astype(np.float32)),
    ]
    if _use_bias:
        _train_tensors += [
            torch.from_numpy(_basis_int_np[train_idx]),
            torch.from_numpy(_scale_sci_np[train_idx]),
            torch.from_numpy(_coef_sci_np[train_idx]),
        ]
        _val_tensors += [
            torch.from_numpy(_basis_int_np[val_idx]),
            torch.from_numpy(_scale_sci_np[val_idx]),
            torch.from_numpy(_coef_sci_np[val_idx]),
        ]
    tr_loader = DataLoader(TensorDataset(*_train_tensors),
                            batch_size=int(batch_size), shuffle=True)
    va_loader = DataLoader(TensorDataset(*_val_tensors),
                            batch_size=512, shuffle=False)

    if device == 'auto':
        if torch.cuda.is_available():
            device = 'cuda'
        elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
            device = 'mps'
        else:
            device = 'cpu'

    model = ResidualEmissivityTransferNet(
        n_coef=n_coef, n_ctx=ctx_near_n.shape[1],
        group_indices=group_indices,
        arm_encoder_dims=tuple(int(v) for v in arm_encoder_dims),
        ctx_dims=tuple(int(v) for v in ctx_dims),
        trunk_dims=tuple(int(v) for v in trunk_dims),
        group_head_hidden=int(group_head_hidden),
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"RETN seed={seed}: n_coef={n_coef}, n_ctx={ctx_near_n.shape[1]}, params={n_params:,d}")

    opt = torch.optim.AdamW(model.parameters(), lr=float(lr),
                             weight_decay=float(weight_decay))

    # Save em_scaler stats as device tensors so we can un-standardize inside the loss
    # for the positivity penalty (needs physical em to check the sign).
    _em_med_t = torch.tensor(em_scaler.med_, dtype=torch.float32, device=device)
    _em_scl_t = torch.tensor(em_scaler.scale_, dtype=torch.float32, device=device)

    # Bias-loss group indices as device tensors (one per component group).
    _bias_gidx_tensors = {}
    if _use_bias:
        for _g_name, _g_idx in _bias_membership.items():
            _bias_gidx_tensors[_g_name] = torch.as_tensor(
                _g_idx, dtype=torch.long, device=device)

    # RETN-CURV-T1-V1: Chebyshev T_1 (linear tilt) + T_2 (bowl) curvature
    # loss on the MoonZodi_bs coefficient residual. Both modes live at
    # wavelengths (~6500 A) not sampled by the 7 bias-loss bands, so the
    # per-row SSFR^2 term cannot see them directly. Each mode is normalised
    # by RMS(a_k) computed on TRUE MoonZodi_bs training coefficients so the
    # single `curv_loss_weight` scales both dimensionlessly.
    _use_curv = (basis_integrals_cache is not None
                 and float(curv_loss_weight) > 0.0
                 and _bias_membership is not None
                 and 'moon+zodi' in _bias_membership
                 and len(_bias_membership['moon+zodi']) >= 5)
    _cheb_T2 = None
    _a2_ref_scale = 1.0
    _moon_zodi_idx_t = None
    _cheb_T1 = None
    _a1_ref_scale = 1.0
    if _use_curv:
        _n_moon = int(len(_bias_membership['moon+zodi']))
        _x_cheb = ((torch.arange(_n_moon, dtype=torch.float32, device=device)
                    - (_n_moon - 1) / 2.0)
                   / ((_n_moon - 1) / 2.0))
        _cheb_T1 = _x_cheb.clone()                     # (n_moon,) linear tilt
        _cheb_T2 = 2.0 * _x_cheb ** 2 - 1.0            # (n_moon,) bowl
        _cheb_T3 = 4.0 * _x_cheb ** 3 - 3.0 * _x_cheb  # (n_moon,) S-shape
        _cheb_T1_sq = float((_cheb_T1 ** 2).sum().item())
        _cheb_T2_sq = float((_cheb_T2 ** 2).sum().item())
        _cheb_T3_sq = float((_cheb_T3 ** 2).sum().item())
        _moon_zodi_idx_t = _bias_gidx_tensors['moon+zodi']
        _moon_true_train = _coef_sci_np[train_idx][:, _bias_membership['moon+zodi']]
        _moon_true_train_t = torch.from_numpy(_moon_true_train).to(device)
        _a1_true_train = (_moon_true_train_t @ _cheb_T1) / _cheb_T1_sq
        _a2_true_train = (_moon_true_train_t @ _cheb_T2) / _cheb_T2_sq
        _a3_true_train = (_moon_true_train_t @ _cheb_T3) / _cheb_T3_sq
        # RETN-CURV-V2 (2026-08-20): RMS(a_k) was contaminated by decomposition
        # outliers (rows with coefs > 1e5 that survive the trainer's kappa=6
        # filter) -- p99 |a3| is 1.5 but MAX is 1e6, so RMS = 8e3 and the T-mode
        # penalties were effectively zero. Robust MAD-based scale tracks the
        # actual per-row Chebyshev-mode spread (~0.1 for T_3).
        def _mad_scale(a_tensor):
            _a_np = a_tensor.detach().cpu().numpy()
            _med = float(np.median(_a_np))
            _mad = float(np.median(np.abs(_a_np - _med)))
            return max(1.4826 * _mad, 1e-6)
        _a1_ref_scale = _mad_scale(_a1_true_train)
        _a2_ref_scale = _mad_scale(_a2_true_train)
        _a3_ref_scale = _mad_scale(_a3_true_train)
        print(f'[RETN curv loss] enabled: weight={curv_loss_weight}, '
              f'n_moon_zodi_knots={_n_moon}, '
              f'a1_ref_scale={_a1_ref_scale:.4e}, '
              f'a2_ref_scale={_a2_ref_scale:.4e}, '
              f'a3_ref_scale={_a3_ref_scale:.4e}')

    # RETN-SMOOTH-V1: discrete integrated-second-derivative penalty on the
    # predicted MoonZodi_bs spline. Normalised by RMS(d2 of TRUE training
    # coefficients) so smooth_loss_weight=1.0 is dimensionless. Complements
    # the T_1/T_2 curvature loss above by suppressing higher-frequency wiggles.
    _use_smooth = (_use_bias
                   and float(smooth_loss_weight) > 0.0
                   and _bias_membership is not None
                   and 'moon+zodi' in _bias_membership
                   and len(_bias_membership['moon+zodi']) >= 3)
    _smooth_ref_scale = 1.0
    if _use_smooth:
        if _moon_zodi_idx_t is None:
            _moon_zodi_idx_t = _bias_gidx_tensors['moon+zodi']
        _moon_true_smooth = _coef_sci_np[train_idx][:, _bias_membership['moon+zodi']]
        _moon_near_smooth = coef_near[train_idx][:, _bias_membership['moon+zodi']].astype(np.float32)
        _true_residual = _moon_true_smooth - _moon_near_smooth
        _d2_true_residual = (_true_residual[:, 2:]
                             - 2.0 * _true_residual[:, 1:-1]
                             + _true_residual[:, :-2])
        _smooth_ref_scale = float(np.sqrt(np.mean(_d2_true_residual ** 2)))
        _smooth_ref_scale = max(_smooth_ref_scale, 1e-6)
        print(f'[RETN smooth loss] Tikhonov-on-residual: weight={smooth_loss_weight}, '
              f'n_moon_zodi_knots={len(_bias_membership["moon+zodi"])}, '
              f'd2(residual)_ref_scale={_smooth_ref_scale:.4e}')

    def _step(em_near_b, em_far_b, ctx_near_b, ctx_far_b, ctx_sci_b, w_pe_b, y_b,
              basis_int_b=None, scale_sci_b=None, coef_sci_b=None):
        em_pred_std, residual_std = model(em_near_b, em_far_b, ctx_near_b, ctx_far_b, ctx_sci_b)
        # Main heteroscedastic loss in standardized em space
        _per_elem = F.smooth_l1_loss(em_pred_std, y_b, reduction='none') * w_pe_b
        L_em = _per_elem.mean()
        # Positivity penalty: un-standardize em_pred and penalize negative values
        em_pred_phys = em_pred_std * _em_scl_t + _em_med_t
        L_pos = F.relu(-em_pred_phys).pow(2).mean()
        # Per-component per-band flux bias loss (relative to RMS true flux).
        _bias_per_group = {}
        if _use_bias and basis_int_b is not None:
            coef_pred_phys = F.relu(em_pred_phys) * scale_sci_b
            _L_bias_val = torch.zeros((), device=em_pred_std.device)
            for _g_name, _g_idx_t in _bias_gidx_tensors.items():
                _diff_g = (coef_pred_phys - coef_sci_b).index_select(1, _g_idx_t)
                _true_g = coef_sci_b.index_select(1, _g_idx_t)
                _int_g = basis_int_b.index_select(2, _g_idx_t)
                _flux_diff = torch.einsum('brc,bc->br', _int_g, _diff_g)
                _flux_true = torch.einsum('brc,bc->br', _int_g, _true_g)
                # RETN-BIAS-PER-ROW-V1: per-row SSFR^2 normalized by that row's
                # own RMS across bands (was per-batch-per-band RMS). This makes
                # bright and faint rows contribute equally, so tail-moon-zodi
                # rows (which dominated the SSFR-max plot) get proportionally
                # more gradient than under the batch-mean formulation. Clamp to
                # 1% of batch-mean per-row RMS to avoid a near-zero row blowing
                # up its own row-band contribution.
                _rms_true_row = torch.sqrt((_flux_true ** 2).mean(dim=1))
                _rms_floor = 0.01 * _rms_true_row.mean().clamp(min=1e-12)
                _rms_true_row = _rms_true_row.clamp(min=_rms_floor)
                _bias_norm_sq = ((_flux_diff / _rms_true_row[:, None]) ** 2).mean()
                _bias_per_group[_g_name] = float(_bias_norm_sq.detach().item())
                _L_bias_val = _L_bias_val + _bias_norm_sq
            L_bias = _L_bias_val
        else:
            L_bias = torch.zeros((), device=em_pred_std.device)
        if _use_curv and basis_int_b is not None:
            _diff_moon = ((coef_pred_phys - coef_sci_b)
                          .index_select(1, _moon_zodi_idx_t))
            _a1_err = (_diff_moon @ _cheb_T1) / _cheb_T1_sq   # (batch,)
            _a2_err = (_diff_moon @ _cheb_T2) / _cheb_T2_sq   # (batch,)
            _a3_err = (_diff_moon @ _cheb_T3) / _cheb_T3_sq   # (batch,)
            # RETN-CURV-V3 (2026-08-20): per-mode weights. All three modes have
            # OOS R^2 ~ 0.4 but T_3 dominates the loss surface -- the head keeps
            # finding S-shape solutions that lift one band at the cost of another
            # (v15/v17 row 830). Weighting T_1 and T_2 while zeroing T_3 keeps
            # the physical tilt signal without the wild per-row S-shape.
            _w_t1, _w_t2, _w_t3 = curv_mode_weights
            L_curv = (float(_w_t1) * ((_a1_err / _a1_ref_scale) ** 2).mean()
                      + float(_w_t2) * ((_a2_err / _a2_ref_scale) ** 2).mean()
                      + float(_w_t3) * ((_a3_err / _a3_ref_scale) ** 2).mean())
        else:
            L_curv = torch.zeros((), device=em_pred_std.device)
        if _use_smooth:
            # Tikhonov prior on the RESIDUAL the head added over em_near:
            # penalise d^2 of (pred - near), capping runaway curvature the
            # network invents that neither near nor far actually support.
            # Ref scale is RMS d^2 of the TRUE residual (sci - near) on
            # training, so weight=1.0 admits training-typical residual
            # curvature and only strong deviations get gradient.
            _residual_phys = residual_std * _em_scl_t
            _residual_moon = _residual_phys.index_select(1, _moon_zodi_idx_t)
            _d2_residual = (_residual_moon[:, 2:]
                            - 2.0 * _residual_moon[:, 1:-1]
                            + _residual_moon[:, :-2])
            L_smooth = ((_d2_residual / _smooth_ref_scale) ** 2).mean()
        else:
            L_smooth = torch.zeros((), device=em_pred_std.device)
        loss = (L_em + float(positivity_weight) * L_pos
                + float(bias_loss_weight) * L_bias
                + float(curv_loss_weight) * L_curv
                + float(smooth_loss_weight) * L_smooth)
        return (loss, L_em.detach().item(), L_pos.detach().item(),
                L_bias.detach().item(), _bias_per_group,
                L_curv.detach().item(), L_smooth.detach().item())

    best_val = np.inf
    best_epoch = -1
    best_state = None
    stale = 0
    history = []
    for ep in range(1, int(n_epochs) + 1):
        model.train()
        tr_loss = 0.0
        tr_em = 0.0
        tr_pos = 0.0
        tr_bias = 0.0
        tr_curv = 0.0
        tr_smooth = 0.0
        tr_bias_per_group = {}
        tr_n = 0
        for _batch in tr_loader:
            em_near_b = _batch[0].to(device); em_far_b = _batch[1].to(device)
            ctx_near_b = _batch[2].to(device); ctx_far_b = _batch[3].to(device)
            ctx_sci_b = _batch[4].to(device); w_pe_b = _batch[5].to(device)
            y_b = _batch[6].to(device)
            if _use_bias:
                basis_int_b = _batch[7].to(device)
                scale_sci_b = _batch[8].to(device)
                coef_sci_b = _batch[9].to(device)
            else:
                basis_int_b = scale_sci_b = coef_sci_b = None
            loss, l_em, l_pos, l_bias, l_bias_pg, l_curv, l_smooth = _step(
                em_near_b, em_far_b, ctx_near_b, ctx_far_b, ctx_sci_b,
                w_pe_b, y_b, basis_int_b, scale_sci_b, coef_sci_b)
            if not torch.isfinite(loss):
                continue
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
            opt.step()
            tr_loss += float(loss.item()); tr_em += l_em; tr_pos += l_pos
            tr_bias += l_bias; tr_curv += l_curv
            tr_smooth += l_smooth; tr_n += 1
            for _g_name, _v in l_bias_pg.items():
                tr_bias_per_group[_g_name] = tr_bias_per_group.get(_g_name, 0.0) + _v

        model.eval()
        va_loss = 0.0
        va_em = 0.0
        va_bias = 0.0
        va_curv = 0.0
        va_smooth = 0.0
        va_bias_per_group = {}
        va_n = 0
        with torch.no_grad():
            for _batch in va_loader:
                em_near_b = _batch[0].to(device); em_far_b = _batch[1].to(device)
                ctx_near_b = _batch[2].to(device); ctx_far_b = _batch[3].to(device)
                ctx_sci_b = _batch[4].to(device); w_pe_b = _batch[5].to(device)
                y_b = _batch[6].to(device)
                if _use_bias:
                    basis_int_b = _batch[7].to(device)
                    scale_sci_b = _batch[8].to(device)
                    coef_sci_b = _batch[9].to(device)
                else:
                    basis_int_b = scale_sci_b = coef_sci_b = None
                loss, l_em, _, l_bias, l_bias_pg, l_curv, l_smooth = _step(
                    em_near_b, em_far_b, ctx_near_b, ctx_far_b, ctx_sci_b,
                    w_pe_b, y_b, basis_int_b, scale_sci_b, coef_sci_b)
                if torch.isfinite(loss):
                    va_loss += float(loss.item()); va_em += l_em
                    va_bias += l_bias; va_curv += l_curv
                    va_smooth += l_smooth; va_n += 1
                    for _g_name, _v in l_bias_pg.items():
                        va_bias_per_group[_g_name] = va_bias_per_group.get(_g_name, 0.0) + _v

        tr_mean = tr_loss / max(tr_n, 1); va_mean = va_loss / max(va_n, 1)
        tr_em_mean = tr_em / max(tr_n, 1); tr_pos_mean = tr_pos / max(tr_n, 1)
        tr_bias_mean = tr_bias / max(tr_n, 1)
        va_bias_mean = va_bias / max(va_n, 1)
        tr_curv_mean = tr_curv / max(tr_n, 1)
        va_curv_mean = va_curv / max(va_n, 1)
        tr_smooth_mean = tr_smooth / max(tr_n, 1)
        va_smooth_mean = va_smooth / max(va_n, 1)
        _tr_pg_mean = {g: v / max(tr_n, 1) for g, v in tr_bias_per_group.items()}
        _va_pg_mean = {g: v / max(va_n, 1) for g, v in va_bias_per_group.items()}
        history.append(dict(epoch=ep, train=tr_mean, val=va_mean,
                            train_em=tr_em_mean, train_pos=tr_pos_mean,
                            train_bias=tr_bias_mean, val_bias=va_bias_mean,
                            train_curv=tr_curv_mean, val_curv=va_curv_mean,
                            train_smooth=tr_smooth_mean, val_smooth=va_smooth_mean,
                            train_bias_per_group=_tr_pg_mean,
                            val_bias_per_group=_va_pg_mean))
        # Print every epoch with per-group bias breakdown (both train and val
        # to spot moon+zodi over-shooting into overfit territory).
        _pg_str = ' '.join(
            f"{g}=t{_tr_pg_mean.get(g, 0.0):.5f}/v{_va_pg_mean.get(g, 0.0):.5f}"
            for g in ('moon+zodi', 'diffuse', 'lines') if g in _tr_pg_mean)
        print(f"  [RETN seed={seed}] ep={ep:3d} train={tr_mean:.5f} "
              f"val={va_mean:.5f} (em={tr_em_mean:.5f} pos={tr_pos_mean:.5f} "
              f"bias_agg={tr_bias_mean:.5f}/{va_bias_mean:.5f} "
              f"curv={tr_curv_mean:.5f}/{va_curv_mean:.5f} "
              f"smooth={tr_smooth_mean:.5f}/{va_smooth_mean:.5f} | {_pg_str})")

        if va_mean < best_val - 1e-6:
            best_val = va_mean; best_epoch = ep; stale = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
            if stale >= int(patience):
                print(f"  [RETN seed={seed}] early stop at ep={ep}, best={best_epoch}, best_val={best_val:.5f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, dict(
        em_scaler=em_scaler, ctx_scaler=ctx_scaler,
        best_val=float(best_val), best_epoch=int(best_epoch),
        history=history, seed=int(seed),
    )


def predict_sci_coefficients_retn(retn_artifacts,
                                    coef_near_phys, coef_far_phys,
                                    ctx_near_phys, ctx_far_phys, ctx_sci_phys):
    """Predict sci coefficients from near/far coefs + contexts via RETN ensemble."""
    coef_near_phys = np.asarray(coef_near_phys, dtype=np.float64)
    coef_far_phys = np.asarray(coef_far_phys, dtype=np.float64)
    ctx_near_phys = np.asarray(ctx_near_phys, dtype=np.float64)
    ctx_far_phys = np.asarray(ctx_far_phys, dtype=np.float64)
    ctx_sci_phys = np.asarray(ctx_sci_phys, dtype=np.float64)

    geom_kwargs = retn_artifacts['geom_kwargs']
    scale_near = airglow_geometry_scale(ctx_near_phys, **geom_kwargs)
    scale_far = airglow_geometry_scale(ctx_far_phys, **geom_kwargs)
    scale_sci = airglow_geometry_scale(ctx_sci_phys, **geom_kwargs)
    em_near = coef_near_phys / scale_near
    em_far = coef_far_phys / scale_far

    em_scaler = retn_artifacts['em_scaler']
    ctx_scaler = retn_artifacts['ctx_scaler']
    em_near_std = np.clip(em_scaler.transform(em_near.astype(np.float32)), -25.0, 25.0)
    em_far_std = np.clip(em_scaler.transform(em_far.astype(np.float32)), -25.0, 25.0)
    ctx_near_n = np.clip(ctx_scaler.transform(ctx_near_phys), -25.0, 25.0).astype(np.float32)
    ctx_far_n = np.clip(ctx_scaler.transform(ctx_far_phys), -25.0, 25.0).astype(np.float32)
    ctx_sci_n = np.clip(ctx_scaler.transform(ctx_sci_phys), -25.0, 25.0).astype(np.float32)

    device = retn_artifacts['device']
    x_en = torch.from_numpy(em_near_std.astype(np.float32)).to(device)
    x_ef = torch.from_numpy(em_far_std.astype(np.float32)).to(device)
    x_cn = torch.from_numpy(ctx_near_n).to(device)
    x_cf = torch.from_numpy(ctx_far_n).to(device)
    x_cs = torch.from_numpy(ctx_sci_n).to(device)

    em_pred_std_stack = []
    for m in retn_artifacts['ensemble_models']:
        m.eval()
        with torch.no_grad():
            em_pred_std, _ = m(x_en, x_ef, x_cn, x_cf, x_cs)
        em_pred_std_stack.append(em_pred_std.cpu().numpy())
    em_pred_std_mean = np.mean(em_pred_std_stack, axis=0)

    # Un-standardize back to em, apply positivity, then to coef via the mixed
    # factor (scale_sci everywhere, amp on moon+zodi block if relative-mz active).
    em_pred = em_pred_std_mean * em_scaler.scale_[None, :] + em_scaler.med_[None, :]
    em_pred = np.clip(em_pred, 0.0, None)
    coef_pred = em_pred * scale_sci
    return coef_pred


# -----------------------------------------------------------------------------
# Config, train ensemble, install as the default predictor
# -----------------------------------------------------------------------------

default_retn_config = {
    'name': 'residual_emissivity_transfer_net',
    'arm_encoder_dims': (512, 256),
    'ctx_dims': (64,),
    'trunk_dims': (256, 128),
    'group_head_hidden': 128,
    'n_epochs': 60,
    'batch_size': 512,
    'lr': 7e-4,
    'weight_decay': 3.3e-5,
    'patience': 10,
    'grad_clip': 1.0,
    'positivity_weight': 0.1,
    'use_coef_err_weights': True,
    # 2026-08-18 (v5): per-ROW SSFR^2 loss driven by basis integrals cache (cell 14).
    # v8 (2026-08-18 late): normalizer switched from per-batch-per-band RMS to
    # per-row RMS across bands (RETN-BIAS-PER-ROW-V1) so bright and faint rows
    # contribute equally, giving proportionally more gradient to the tail rows
    # where SSFR was highest.
    'bias_loss_weight': 1.0,
    # 2026-08-18 (v6): Chebyshev T_2 curvature loss on the MoonZodi_bs coefficient
    # residual. Targets the second-order (bowl / inverted-bowl) residual visible
    # on the worst rows after v5. Normalized by RMS T_2 amplitude of TRUE
    # MoonZodi_bs on the training set, so weight 1.0 is dimensionless.
    # 2026-08-18 (v7): T_1 (linear tilt) mode added under the same weight,
    # normalised analogously by RMS T_1 amplitude of the TRUE training
    # MoonZodi_bs coefficients (see RETN-CURV-T1-V1 marker).
    # 2026-08-20 (v13): T_3 (S-shape, under-over-under) mode added. At
    # smooth_loss_weight=5.0 the residual head compressed all its residual
    # activity into T_3 (row 830: under-predict blue, over-predict 5500-7500 A,
    # under-predict red) because T_3 was the lowest-frequency Chebyshev mode
    # not already regularised. Same truth-RMS normalisation as T_1/T_2 so the
    # network can still express training-typical T_3 amplitude but not more.
        # 2026-08-20 (v16): DISABLED. Per-arm predictability diagnostic
    # (cell 19) shows MoonZodi_bs T_1/T_2/T_3 have OOS R^2 = 0.38-0.43,
    # so ~57% of truth Chebyshev-mode variance is UNPREDICTABLE from
    # (near, far). At weight=1.0 the curvature term chased that
    # unpredictable variance and forced per-row wild solutions
    # (row 830 case). Tikhonov-on-residual (smooth_loss_weight)
    # already protects the copy-near baseline curvature; that is
    # enough given the 0.97 per-row d^2 correlation.
        # 2026-08-20 (v20): revert to T_3=0. v19 (T_3=0.3) restored 830's T_3
    # residual only partially but re-introduced the tilt (T_1) residual on row
    # 1605 that v18 had cleanly fixed. The T_3-vs-T_1 tradeoff couples through
    # the shared trunk representation, so any positive T_3 weight steals
    # capacity from T_1 correction on rows whose truth T_3 direction doesn't
    # align with what T_1 correction wants. Accept 830's T_3 residual as a
    # per-row decomposition floor (unpredictable component of truth's T_3)
    # and keep T_1/T_2 fitting clean everywhere else.
    'curv_loss_weight': 1.0,
    'curv_mode_weights': (1.0, 1.0, 0.0),
    # 2026-08-19 (v10): discrete integrated-2nd-derivative smoothness on the
    # predicted MoonZodi_bs spline; normalised by RMS(d2) of TRUE training coefs.
    # 2026-08-20 (v11..v12): absolute-d^2 form scaled up to 5.0.
    # 2026-08-20 (v14): DELTA form ((d2_pred - d2_true)^2) freed the network to
    # match truth curvature but at test time the network had to GUESS truth's
    # per-row d^2 from only near/far, and guesses were noisy -- row 830 pred
    # went to a wild solution.
    # 2026-08-20 (v15): TIKHONOV-ON-RESIDUAL. Penalise d^2 of (pred - near),
    # not d^2 of pred and not (d^2_pred - d^2_true).  This caps ADDED curvature
    # relative to the copy-near baseline without demanding the network guess
    # truth's specific d^2 pattern.  Ref scale is RMS d^2(sci - near) on train.
    # weight=1.0 -> a residual with training-typical curvature costs O(1).
    'smooth_loss_weight': 1.0,
    'ensemble_seeds': (42, 43, 44, 45, 46),
}

RUN_RETN = True

if RUN_RETN:
    print(f'=== Training Residual Emissivity Transfer Network '
          f'({len(default_retn_config["ensemble_seeds"])}-seed ensemble) ===')
    print(default_retn_config)

    # RETN is self-contained: build group_indices + geom_kwargs directly from
    # the filtered triplet.  Merge the 3-coef 'continuum' block into the 29-coef
    # 'moon' block so the joint continuum-producing coefficients share a single
    # residual head (they both drive continuum flux, and the joint compressor
    # experiment showed this join reduces continuum-decomposition mass shuffle).
    _retn_group_indices = _build_group_indices(filtered_triplet['coef_names'])
    if 'continuum' in _retn_group_indices:
        _moon_idx = np.asarray(_retn_group_indices['moon'], dtype=int)
        _cont_idx = np.asarray(_retn_group_indices['continuum'], dtype=int)
        _retn_group_indices['moon'] = np.sort(np.concatenate([_moon_idx, _cont_idx]))
        del _retn_group_indices['continuum']
        print(f'[join continuum] moon block: {_moon_idx.size} + {_cont_idx.size} '
              f'= {_retn_group_indices["moon"].size} coefs (continuum group removed)')

    compress_geom_kwargs = dict(
        ctx_names=filtered_triplet['ctx_names'],
        group_indices=_retn_group_indices,
        n_coef=int(filtered_triplet['coef_near'].shape[1]),
        coef_wavelengths_a=filtered_triplet.get('coef_wavelengths_a'),
        coef_extinction_k=filtered_triplet.get('coef_extinction_k'),
    )

    # Shared split so eRMSE metrics are directly comparable to the current baseline.
    _retn_split = (
        np.asarray(filtered_triplet.get('compress_train_idx',
                                        split_indices_by_moon_phase(
                                            filtered_triplet['obstime_mjd'],
                                            _moon_phase_deg_from_ctx(filtered_triplet), seed=42)[0]),
                   dtype=int),
        np.asarray(filtered_triplet.get('compress_val_idx',
                                        split_indices_by_moon_phase(
                                            filtered_triplet['obstime_mjd'],
                                            _moon_phase_deg_from_ctx(filtered_triplet), seed=42)[1]),
                   dtype=int),
        np.asarray(filtered_triplet.get('compress_test_idx',
                                        split_indices_by_moon_phase(
                                            filtered_triplet['obstime_mjd'],
                                            _moon_phase_deg_from_ctx(filtered_triplet), seed=42)[2]),
                   dtype=int),
    )

    _retn_models = []
    _retn_per_seed = []
    _basis_int_for_retn = globals().get('basis_integrals_cache', None)
    if _basis_int_for_retn is None:
        print('[RETN] basis_integrals_cache not found in globals; '
              'bias loss will be disabled. Run cell 14 (basis integrals) first to enable it.')
    for _seed in default_retn_config['ensemble_seeds']:
        print(f'\n--- RETN ensemble member seed={_seed} ---')
        _m, _art = train_retn(
            filtered_triplet, _retn_group_indices, compress_geom_kwargs,
            split_indices=_retn_split,
            arm_encoder_dims=default_retn_config['arm_encoder_dims'],
            ctx_dims=default_retn_config['ctx_dims'],
            trunk_dims=default_retn_config['trunk_dims'],
            group_head_hidden=default_retn_config['group_head_hidden'],
            n_epochs=default_retn_config['n_epochs'],
            batch_size=default_retn_config['batch_size'],
            lr=default_retn_config['lr'],
            weight_decay=default_retn_config['weight_decay'],
            patience=default_retn_config['patience'],
            grad_clip=default_retn_config['grad_clip'],
            positivity_weight=default_retn_config['positivity_weight'],
            use_coef_err_weights=default_retn_config['use_coef_err_weights'],
            basis_integrals_cache=_basis_int_for_retn,
            bias_loss_weight=default_retn_config['bias_loss_weight'],
            curv_loss_weight=default_retn_config['curv_loss_weight'],
            curv_mode_weights=default_retn_config['curv_mode_weights'],
            smooth_loss_weight=default_retn_config['smooth_loss_weight'],
            seed=int(_seed),
        )
        _retn_models.append(_m)
        _retn_per_seed.append(_art)

    _first = _retn_per_seed[0]
    retn_artifacts = dict(
        ensemble_models=_retn_models,
        em_scaler=_first['em_scaler'],
        ctx_scaler=_first['ctx_scaler'],
        geom_kwargs=compress_geom_kwargs,
        group_indices=_retn_group_indices,
        device=next(_retn_models[0].parameters()).device,
        per_seed=_retn_per_seed,
        config=dict(default_retn_config),
    )

    # Report per-seed and ensemble test metrics on the same held-out split
    # as the current baseline for direct comparison.
    _test_idx = _retn_split[2]
    _coef_sci_test = np.asarray(filtered_triplet['coef_sci'][_test_idx], dtype=np.float64)

    _per_seed_rows = []
    for i, m in enumerate(_retn_models):
        _pred_i = predict_sci_coefficients_retn(
            {**retn_artifacts, 'ensemble_models': [m]},
            coef_near_phys=filtered_triplet['coef_near'][_test_idx],
            coef_far_phys=filtered_triplet['coef_far'][_test_idx],
            ctx_near_phys=filtered_triplet['ctx_near'][_test_idx],
            ctx_far_phys=filtered_triplet['ctx_far'][_test_idx],
            ctx_sci_phys=filtered_triplet['ctx_sci'][_test_idx],
        )
        _r = _metric_row(_coef_sci_test, _pred_i, f'seed={default_retn_config["ensemble_seeds"][i]}',
                         sigma=filtered_triplet.get('coef_err_sci', np.full_like(_coef_sci_test, np.nan))[_test_idx],
                         group_indices=_retn_group_indices,
                         floor_by_group={'moon': 0.05, 'mesospheric': 0.2,
                                          'ionospheric': 0.05, 'atomic': 0.2})
        _per_seed_rows.append({'variant': f'seed={default_retn_config["ensemble_seeds"][i]}', **_r})

    _pred_ens = predict_sci_coefficients_retn(
        retn_artifacts,
        coef_near_phys=filtered_triplet['coef_near'][_test_idx],
        coef_far_phys=filtered_triplet['coef_far'][_test_idx],
        ctx_near_phys=filtered_triplet['ctx_near'][_test_idx],
        ctx_far_phys=filtered_triplet['ctx_far'][_test_idx],
        ctx_sci_phys=filtered_triplet['ctx_sci'][_test_idx],
    )
    _r_ens = _metric_row(_coef_sci_test, _pred_ens, 'ensemble',
                          sigma=filtered_triplet.get('coef_err_sci', np.full_like(_coef_sci_test, np.nan))[_test_idx],
                          group_indices=_retn_group_indices,
                          floor_by_group={'moon': 0.05, 'mesospheric': 0.2,
                                           'ionospheric': 0.05, 'atomic': 0.2})
    _per_seed_rows.append({'variant': f'{len(_retn_models)}-seed RETN ensemble', **_r_ens})

    import pandas as pd
    print(f'\nPer-seed and ensemble test metrics on the night-held-out split (n_test = {len(_test_idx)} rows):')
    print(pd.DataFrame(_per_seed_rows).to_string(index=False))

    _per_seed_rmse_arr = np.array([r['mean_eRMSE'] for r in _per_seed_rows[:-1]])
    _seed_std_rmse = float(np.std(_per_seed_rmse_arr))
    _ensemble_stderr = _seed_std_rmse / (len(_retn_models) ** 0.5)
    print(f'\nSeed-to-seed test mean_eRMSE std: {_seed_std_rmse:.3f}  '
          f'(ensemble stderr = {_ensemble_stderr:.3f})')

    # -------------------------------------------------------------------------
    # Install RETN as the default predictor for the downstream cells (25, 26).
    # This overwrites `mlp_artifacts` and `predict_sci_coefficients_default`.
    # Rerun cell 20 (old trainer) to switch back to the compressor-based model.
    # -------------------------------------------------------------------------
    mlp_artifacts = dict(retn_artifacts)  # so downstream cells that check keys still work
    mlp_artifacts['group_indices'] = _retn_group_indices
    mlp_artifacts['geom_kwargs'] = compress_geom_kwargs

    def predict_sci_coefficients_default(_artifacts, coef_near_phys, coef_far_phys,
                                          ctx_near_phys, ctx_far_phys, ctx_sci_phys):
        return predict_sci_coefficients_retn(
            _artifacts, coef_near_phys, coef_far_phys,
            ctx_near_phys, ctx_far_phys, ctx_sci_phys,
        )
    # Expose split indices for downstream cells (24 relationship viz, 27 batch,
    # 47 full-sky galactic-coordinates map).
    train_idx, val_idx, test_idx = _retn_split
    print('\n[RETN] installed as `predict_sci_coefficients_default` -- downstream '
          'cells will now evaluate the RETN model.')


## Diagnostics

In [ ]:
# ============================================================================
# Per-arm predictability of the moon+zodi spline correction.
#
# For each MoonZodi_bs knot k, quantify how much of the sci coefficient
# variance is predictable from a linear combination of the near-arm and
# far-arm coefficients at the same knot.  Fit on the training split, report
# out-of-sample R^2 on the validation split.  Same for the T_1, T_2, T_3
# Chebyshev mode amplitudes.  Repeat on OH lines for comparison (OH is
# dominated by gravity waves shared across arms and should show much
# higher predictability if the airglow signal transfers cleanly).
#
# Motivation: on row 830 the sci moon+zodi spline shows a T_3 S-shape that
# the RETN prediction does not reproduce.  If the true predictor from
# (near, far) has R^2 << 1 for T_3, no network with these inputs can match
# that specific per-row curvature -- the sci spline correction carries
# per-decomposition fit noise that near and far each realise independently.
# ============================================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_required_pred = ['filtered_triplet', '_retn_split', '_bias_component_membership']
_missing_pred = [k for k in _required_pred if k not in globals()]
if _missing_pred:
    raise RuntimeError(
        f"Missing globals: {_missing_pred}. Run the RETN trainer cell first "
        "so filtered_triplet, _retn_split and _bias_component_membership "
        "are populated.")

_pred_train_idx, _pred_val_idx, _ = _retn_split
_pred_train_idx = np.asarray(_pred_train_idx, dtype=int)
_pred_val_idx = np.asarray(_pred_val_idx, dtype=int)

_pred_membership = _bias_component_membership(list(filtered_triplet['coef_names']))
_pred_names_all = [str(n) for n in filtered_triplet['coef_names']]
_pred_names_lower_all = [n.lower() for n in _pred_names_all]

_pred_mz_idx = np.asarray(_pred_membership['moon+zodi'], dtype=int)
_pred_oh_idx = np.array(
    [j for j, n in enumerate(_pred_names_lower_all) if n.startswith('oh_')],
    dtype=int)

_pred_near = np.asarray(filtered_triplet['coef_near'], dtype=np.float64)
_pred_far  = np.asarray(filtered_triplet['coef_far'],  dtype=np.float64)
_pred_sci  = np.asarray(filtered_triplet['coef_sci'],  dtype=np.float64)


def _r_pearson(x, y):
    """Pearson correlation with guards against zero variance / non-finite."""
    x = np.asarray(x, dtype=np.float64); y = np.asarray(y, dtype=np.float64)
    m = np.isfinite(x) & np.isfinite(y)
    if int(m.sum()) < 3:
        return np.nan
    x = x[m]; y = y[m]
    sx = x.std(); sy = y.std()
    if sx < 1e-30 or sy < 1e-30:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def _oos_r2_two_input(y_tr, y_va, x1_tr, x2_tr, x1_va, x2_va):
    """OLS y = a + b*x1 + c*x2 on training; out-of-sample R^2 on validation."""
    y_tr = np.asarray(y_tr, dtype=np.float64)
    y_va = np.asarray(y_va, dtype=np.float64)
    X_tr = np.column_stack([np.ones_like(y_tr), x1_tr, x2_tr]).astype(np.float64)
    X_va = np.column_stack([np.ones_like(y_va), x1_va, x2_va]).astype(np.float64)
    m_tr = np.all(np.isfinite(X_tr), axis=1) & np.isfinite(y_tr)
    m_va = np.all(np.isfinite(X_va), axis=1) & np.isfinite(y_va)
    if int(m_tr.sum()) < 10 or int(m_va.sum()) < 10:
        return np.nan
    beta, *_ = np.linalg.lstsq(X_tr[m_tr], y_tr[m_tr], rcond=None)
    y_hat = X_va[m_va] @ beta
    ss_res = float(np.sum((y_va[m_va] - y_hat) ** 2))
    ss_tot = float(np.sum((y_va[m_va] - y_va[m_va].mean()) ** 2))
    if ss_tot < 1e-30:
        return np.nan
    return 1.0 - ss_res / ss_tot


def _per_knot_stats(block_idx, label):
    """Per-knot corr(near, sci), corr(far, sci) and 2-input OOS R^2."""
    rows = []
    for k in block_idx:
        n_tr = _pred_near[_pred_train_idx, k]
        f_tr = _pred_far[_pred_train_idx, k]
        s_tr = _pred_sci[_pred_train_idx, k]
        n_va = _pred_near[_pred_val_idx, k]
        f_va = _pred_far[_pred_val_idx, k]
        s_va = _pred_sci[_pred_val_idx, k]
        rows.append({
            'name': _pred_names_all[k],
            'k': int(k),
            'r_near': _r_pearson(n_tr, s_tr),
            'r_far':  _r_pearson(f_tr, s_tr),
            'r_mean': _r_pearson(0.5 * (n_tr + f_tr), s_tr),
            'oos_R2_two_input': _oos_r2_two_input(
                s_tr, s_va, n_tr, f_tr, n_va, f_va),
        })
    df = pd.DataFrame(rows)
    print(f'\n{label} block ({len(block_idx)} coefs):')
    for col_name, col_label in (
        ('r_near', 'per-knot corr(near, sci)'),
        ('r_far',  'per-knot corr(far,  sci)'),
        ('r_mean', 'per-knot corr(0.5(near+far), sci)'),
        ('oos_R2_two_input',
         'per-knot OOS R^2 of a + b*near + c*far   '),
    ):
        vals = df[col_name].dropna()
        if vals.empty:
            print(f'  {col_label}: no finite values')
            continue
        print(f'  {col_label}: '
              f'median={vals.median():+.3f}, '
              f'p25={vals.quantile(0.25):+.3f}, '
              f'p75={vals.quantile(0.75):+.3f}, '
              f'min={vals.min():+.3f}, max={vals.max():+.3f}')
    return df


_pred_mz_df = _per_knot_stats(_pred_mz_idx, 'MoonZodi_bs')
_pred_oh_df = _per_knot_stats(_pred_oh_idx, 'OH lines')


# Chebyshev-mode predictability for the moon+zodi block: a_k(row) = <coef, T_k> / <T_k, T_k>.
_n_moon = len(_pred_mz_idx)
_x_cheb = (np.arange(_n_moon, dtype=np.float64) - (_n_moon - 1) / 2.0) / ((_n_moon - 1) / 2.0)
_T_modes = {
    'T_1': _x_cheb,
    'T_2': 2.0 * _x_cheb ** 2 - 1.0,
    'T_3': 4.0 * _x_cheb ** 3 - 3.0 * _x_cheb,
}
print('\nMoonZodi_bs Chebyshev-mode predictability (all rows in filtered_triplet):')
print(f"  {'mode':<6s} {'r_near':>8s} {'r_far':>8s} {'r_mean':>8s} {'OOS_R2':>8s}")
_near_mz = _pred_near[:, _pred_mz_idx]
_far_mz  = _pred_far[:,  _pred_mz_idx]
_sci_mz  = _pred_sci[:,  _pred_mz_idx]
for _mn, _Tk in _T_modes.items():
    _tk_sq = float(np.sum(_Tk ** 2))
    _a_near = (_near_mz @ _Tk) / _tk_sq
    _a_far  = (_far_mz  @ _Tk) / _tk_sq
    _a_sci  = (_sci_mz  @ _Tk) / _tk_sq
    _r_n = _r_pearson(_a_near[_pred_train_idx], _a_sci[_pred_train_idx])
    _r_f = _r_pearson(_a_far[_pred_train_idx],  _a_sci[_pred_train_idx])
    _r_m = _r_pearson(0.5 * (_a_near[_pred_train_idx] + _a_far[_pred_train_idx]),
                      _a_sci[_pred_train_idx])
    _r2 = _oos_r2_two_input(
        _a_sci[_pred_train_idx], _a_sci[_pred_val_idx],
        _a_near[_pred_train_idx], _a_far[_pred_train_idx],
        _a_near[_pred_val_idx], _a_far[_pred_val_idx],
    )
    print(f'  {_mn:<6s} {_r_n:>+8.3f} {_r_f:>+8.3f} {_r_m:>+8.3f} {_r2:>+8.3f}')

# Curvature-of-spline predictability (d^2 of the spline knot vector, per row).
# corr(d^2_near, d^2_sci) integrated across knots -- tells if the CURVATURE
# pattern of the sci spline is anywhere in the near / far splines.
def _d2(mat):
    return mat[:, 2:] - 2.0 * mat[:, 1:-1] + mat[:, :-2]

_d2_near = _d2(_near_mz)
_d2_far  = _d2(_far_mz)
_d2_sci  = _d2(_sci_mz)

# Flatten all (row, knot) pairs for a pooled correlation across the whole
# moon+zodi block.  Also compute per-row correlation between d^2_near and
# d^2_sci as a spread indicator.
_r_pool_near = _r_pearson(_d2_near[_pred_train_idx].ravel(),
                          _d2_sci[_pred_train_idx].ravel())
_r_pool_far  = _r_pearson(_d2_far[_pred_train_idx].ravel(),
                          _d2_sci[_pred_train_idx].ravel())
_per_row_corr_near = np.array([
    _r_pearson(_d2_near[i], _d2_sci[i]) for i in _pred_train_idx])
_per_row_corr_near = _per_row_corr_near[np.isfinite(_per_row_corr_near)]

print('\nMoonZodi_bs d^2 (curvature) predictability:')
print(f'  pooled corr(d^2 near, d^2 sci) across all (row,knot): '
      f'{_r_pool_near:+.3f}')
print(f'  pooled corr(d^2 far,  d^2 sci) across all (row,knot): '
      f'{_r_pool_far:+.3f}')
if _per_row_corr_near.size:
    print(f'  per-row corr(d^2 near, d^2 sci): '
          f'median={np.median(_per_row_corr_near):+.3f}, '
          f'p25={np.quantile(_per_row_corr_near, 0.25):+.3f}, '
          f'p75={np.quantile(_per_row_corr_near, 0.75):+.3f}')

print()
print('Interpretation:')
print('  * r_near / r_far close to 1 -> that arm is a near-perfect linear')
print('    predictor of the sci coefficient for that knot / mode.')
print('  * r ~ 0 -> that arm has NO predictive information; the per-row sci')
print('    coefficient is uncorrelated with the same-knot near/far value and')
print('    is dominated by per-arm fit noise or degeneracy.')
print('  * OOS_R2 is the fraction of sci variance a 2-input linear model')
print('    (a + b*near + c*far) captures on a held-out set.  Any network with')
print('    the same two inputs cannot exceed this on that specific knot / mode')
print('    without additional features -- so if OOS_R2 near 0, the residual we')
print('    see in reconstruction is a fundamental limit of the transfer, not')
print('    a network deficiency.')
print('  * A sharp contrast between MoonZodi_bs (near-zero R^2 expected) and OH')
print('    lines (high R^2 expected via shared gravity-wave modulation)')
print('    supports the hypothesis that the moon+zodi spline correction is fit')
print('    noise rather than a signal that transfers between arms.')

_fig_r = make_subplots(
    rows=1, cols=2,
    subplot_titles=('MoonZodi_bs per knot', 'OH per coef'),
    horizontal_spacing=0.10,
)
for _fig_col, _df, _label in (
    (1, _pred_mz_df, 'MoonZodi_bs'),
    (2, _pred_oh_df, 'OH'),
):
    _xk = np.arange(len(_df))
    for _col_name, _color in (
        ('r_near',           '#7f7f7f'),
        ('r_far',            '#bdbdbd'),
        ('oos_R2_two_input', '#e41a1c'),
    ):
        _fig_r.add_trace(
            go.Scattergl(
                x=_xk, y=_df[_col_name].to_numpy(),
                mode='lines+markers',
                marker=dict(size=4),
                line=dict(color=_color, width=1.2),
                name=_col_name,
                legendgroup=_col_name,
                showlegend=(_fig_col == 1),
            ),
            row=1, col=_fig_col,
        )
    _fig_r.add_hline(y=0, line=dict(color='black', width=0.6, dash='dot'),
                     row=1, col=_fig_col)
    _fig_r.update_xaxes(title_text='coefficient index within block',
                        row=1, col=_fig_col)
    _fig_r.update_yaxes(title_text='corr / R^2', range=[-0.15, 1.05],
                        row=1, col=_fig_col)
_fig_r.update_layout(
    template='plotly_white', height=440, width=1200,
    title=('Sci coefficient predictability from (near, far) -- '
           'per-knot correlations and 2-input OOS R^2 '
           '(OH shown for comparison; higher = more transferable)'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='left', x=0.0),
)
_fig_r.show()


In [ ]:
# Coefficient-vs-context scatter matrix (2026-08-10):
# one row per coefficient (or median-of-group for the two big aggregates),
# one column per context variable, four series overlaid per panel:
#   near (blue), far (purple), sci_true (green), sci_pred_default (red).
# Shared y-axis per row, shared x-axis per column.
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

required = ["mlp_artifacts", "filtered_triplet", "predict_sci_coefficients_default"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run training + prediction cells first. Missing: " + ", ".join(missing))

coef_near_all = np.asarray(filtered_triplet["coef_near"], dtype=np.float32)
coef_far_all = np.asarray(filtered_triplet["coef_far"], dtype=np.float32)
coef_sci_all = np.asarray(filtered_triplet["coef_sci"], dtype=np.float32)
ctx_near_all = np.asarray(filtered_triplet["ctx_near"], dtype=np.float32)
ctx_far_all = np.asarray(filtered_triplet["ctx_far"], dtype=np.float32)
ctx_sci_all = np.asarray(filtered_triplet["ctx_sci"], dtype=np.float32)
coef_names_all = [str(n) for n in filtered_triplet["coef_names"]]
ctx_names_all = [str(n) for n in filtered_triplet["ctx_names"]]

coef_pred_all = predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=coef_near_all,
    coef_far_phys=coef_far_all,
    ctx_near_phys=ctx_near_all,
    ctx_far_phys=ctx_far_all,
    ctx_sci_phys=ctx_sci_all,
)

# Cyclic sin/cos pairs collapsed to a single degree axis so column headers stay concise.
_ctx_names_display, _ctx_near_disp = _decode_cyclic_context(ctx_names_all, ctx_near_all)
_, _ctx_far_disp = _decode_cyclic_context(ctx_names_all, ctx_far_all)
_, _ctx_sci_disp = _decode_cyclic_context(ctx_names_all, ctx_sci_all)

# Row definitions: two aggregates (moon, OH) then every remaining named coefficient
# from COEF_SCHEMA -- the 3 continuum, 6 atomic/ionospheric line species, and O2 b-band.
_names_lower = [n.lower() for n in coef_names_all]
_moon_idx = np.array([i for i, n in enumerate(_names_lower)
                      if n.startswith("moon_bs")], dtype=int)
_oh_idx = np.array([i for i, n in enumerate(_names_lower)
                    if n.startswith("oh_")], dtype=int)

_INDIVIDUAL = [
    "HO2", "FeO", "O2Ac",
    "ATOM_K", "ATOM_Na", "ATOM_Og",
    "ATOM_N", "ATOM_Or", "ATOM_Orc_OI0777", "ATOM_Orc_OI0845",
    "O2_b01",
]


def _find_col(name):
    lname = name.lower()
    matches = [i for i, n in enumerate(_names_lower) if n == lname]
    if not matches:
        raise KeyError(f"coefficient {name!r} not in coef_names_all")
    return matches[0]


_individual_idx = {name: _find_col(name) for name in _INDIVIDUAL}

# Downsample rows so ~3.5M points across all panels stay renderable.
_MAX_POINTS = 3000
_n_rows_data = coef_near_all.shape[0]
if _n_rows_data > _MAX_POINTS:
    _rng = np.random.default_rng(42)
    _row_sel = np.sort(_rng.choice(_n_rows_data, size=_MAX_POINTS, replace=False))
else:
    _row_sel = np.arange(_n_rows_data)

_ROW_DEFS = [
    (f"moon median (n={_moon_idx.size})", "median_moon"),
    (f"OH median (n={_oh_idx.size})",     "median_oh"),
]
for name in _INDIVIDUAL:
    _ROW_DEFS.append((name, name))


def _series_values(coef_mat, kind):
    """One value per selected row: either an aggregate median or the raw column."""
    if kind == "median_moon":
        return np.nanmedian(coef_mat[_row_sel][:, _moon_idx], axis=1)
    if kind == "median_oh":
        return np.nanmedian(coef_mat[_row_sel][:, _oh_idx], axis=1)
    return coef_mat[_row_sel, _individual_idx[kind]]


_SERIES = [
    ("near",             coef_near_all, _ctx_near_disp, "#1f77b4"),
    ("far",              coef_far_all,  _ctx_far_disp,  "#9467bd"),
    ("sci_true",         coef_sci_all,  _ctx_sci_disp,  "#2ca02c"),
    ("sci_pred_default", coef_pred_all, _ctx_sci_disp,  "#d62728"),
]

n_rows = len(_ROW_DEFS)
n_cols = len(_ctx_names_display)

# Column headers on the top row only; row labels on the leftmost column via yaxis title.
_subplot_titles = []
for r in range(n_rows):
    for c in range(n_cols):
        _subplot_titles.append(_ctx_names_display[c].replace("_", " ") if r == 0 else "")

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=_subplot_titles,
    shared_xaxes="columns",
    shared_yaxes="rows",
    vertical_spacing=0.015,
    horizontal_spacing=0.004,
)

_marker_base = dict(size=3.0, opacity=0.20)

for r, (row_label, kind) in enumerate(_ROW_DEFS, start=1):
    for series_name, coef_mat, ctx_disp, color in _SERIES:
        y_vals = _series_values(coef_mat, kind)
        for c in range(n_cols):
            # sci_sep is 0 by construction on the sci arm; the two sci series collapse
            # to a vertical line at 0 there. Only near/far carry information for sci_sep.
            if (_ctx_names_display[c] == "sci_sep"
                    and series_name in ("sci_true", "sci_pred_default")):
                continue
            x_vals = ctx_disp[_row_sel, c]
            fig.add_trace(
                go.Scattergl(
                    x=x_vals,
                    y=y_vals,
                    mode="markers",
                    marker=dict(color=color, **_marker_base),
                    name=series_name,
                    legendgroup=series_name,
                    showlegend=(r == 1 and c == 0),
                    hoverinfo="skip",
                ),
                row=r, col=c + 1,
            )
    fig.update_yaxes(title_text=row_label, row=r, col=1,
                     title_font=dict(size=9))

# Column axis labels appear as the top-row subplot titles; bottom-row x-axes get the
# same labels so the reader can identify columns from either end of the tall figure.
for c in range(n_cols):
    fig.update_xaxes(title_text=_ctx_names_display[c].replace("_", " "),
                     row=n_rows, col=c + 1, title_font=dict(size=9))

fig.for_each_annotation(lambda a: a.update(font=dict(size=9)))
fig.update_xaxes(showline=True, mirror=True, ticks="outside", ticklen=3,
                 tickfont=dict(size=7), tickangle=30)
fig.update_yaxes(showline=True, mirror=True, ticks="outside", ticklen=3,
                 tickfont=dict(size=7))

_panel_w = 180
_panel_h = 155
fig.update_layout(
    template="plotly_white",
    title=(f"Coefficient vs context ({n_rows} rows x {n_cols} cols, "
           f"n={_row_sel.size} rows sampled). "
           "near/far are inputs; sci_true and sci_pred_default are the target and the default prediction."),
    height=_panel_h * n_rows + 180,
    width=_panel_w * n_cols + 200,
    legend=dict(orientation="h", yanchor="bottom", y=1.005, xanchor="left", x=0.0,
                itemsizing="constant"),
    margin=dict(l=100, r=30, t=100, b=100),
)
fig.show()


In [ ]:
# Full-spectrum reconstruction test for a single requested row using default coefficients
import plotly.graph_objects as go

_e10_stem = "lvmsframe_median_stack_1.2.1_p40_p70_every10"
_e10_suffix = _DECOMP_SUFFIX  # inherited from cell 6
EVERY10_INPUT = f"{_e10_stem}.fits"
EVERY10_NEAR = f"{_e10_stem}_decomp_sky1{_e10_suffix}.fits"
EVERY10_FAR = f"{_e10_stem}_decomp_sky2{_e10_suffix}.fits"
EVERY10_SCI = f"{_e10_stem}_decomp_sci{_e10_suffix}.fits"

def expnum_to_row(expnum, meta_fits_path=EVERY10_INPUT):
    """Return the row index in `meta_fits_path` for the given exposure number.

    Raises IndexError if the exposure number is not present.  If more than
    one row shares the expnum (should not happen in every10 but is
    tolerated), the first match is returned and a note is printed.
    """
    with fits.open(meta_fits_path) as _h:
        _ex = np.asarray(_h["META"].data["expnum"], dtype=np.int64)
    _matches = np.flatnonzero(_ex == int(expnum))
    if _matches.size == 0:
        raise IndexError(
            f"expnum={expnum} not present in META['expnum'] of {meta_fits_path}"
        )
    if _matches.size > 1:
        print(f"  note: expnum={expnum} matches {_matches.size} rows in META; "
              f"using the first (row {int(_matches[0])})")
    return int(_matches[0])

# Set the row to reconstruct and inspect.
# REQUESTED_ROW = 612
# REQUESTED_ROW = 500
# REQUESTED_ROW = 978
# REQUESTED_ROW = 98   # f'cked up
# REQUESTED_ROW = 1605 #wiggle/tilt
REQUESTED_ROW = 830 #wiggle

required = [
    "mlp_artifacts",
    "predict_sci_coefficients_default",
    "context_cols",
    "build_triplet_coef_dataset",
    "reconstruct_component_spectra",
    "reconstruct_with_lsf",
    "load_lsf_state_if_available",
    "load_moon_zodi_state_if_available",
    "load_o2_vector_if_available",
    "_infer_base_dir_for_reconstruction",
    "_moon_bs_indices_from_names",
    "_row_spline_roughness",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run the training + residual-correction cells first. Missing: " + ", ".join(missing))

# 1) Load coefficients/context from every10 decomposition products.
e10_triplet = build_triplet_coef_dataset(
    input_fits_path=EVERY10_INPUT,
    sky_near_decomp_fits_path=EVERY10_NEAR,
    sky_far_decomp_fits_path=EVERY10_FAR,
    sci_decomp_fits_path=EVERY10_SCI,
    context_columns=context_cols,
    return_chi2=False,
)
# ECLIPTIC-CTX-V1: match training-time ctx layout on the e10 triplet.
if '_augment_triplet_with_ecliptic' in globals():
    _augment_triplet_with_ecliptic(e10_triplet)
n_e10 = int(e10_triplet["n_rows"])
row_index_e10 = np.asarray(e10_triplet["row_index"], dtype=np.int64)
coef_names_e10 = [str(n) for n in e10_triplet["coef_names"]]
moon_idx = _moon_bs_indices_from_names(coef_names_e10)

if moon_idx.size < 3:
    raise RuntimeError("Expected at least 3 Moon_bs coefficients for spline diagnostics")

# 2) Load observed spectra, wavelength grid, and LSF from every10 input.
with fits.open(EVERY10_INPUT) as hdul:
    for ext in ("FLUX_SKY_NEAR", "FLUX_SKY_FAR", "FLUX_SCI", "WAVE", "LSF_SCI", "LSF_SKY_NEAR", "LSF_SKY_FAR"):
        if ext not in hdul:
            raise KeyError(f"Missing extension {ext} in {EVERY10_INPUT}")

    wave_arr = np.asarray(hdul["WAVE"].data, dtype=np.float64)
    flux_near_all = np.asarray(hdul["FLUX_SKY_NEAR"].data, dtype=np.float64)
    flux_far_all = np.asarray(hdul["FLUX_SKY_FAR"].data, dtype=np.float64)
    flux_sci_true_all = np.asarray(hdul["FLUX_SCI"].data, dtype=np.float64)
    lsf_sci_arr = np.asarray(hdul["LSF_SCI"].data, dtype=np.float64)
    lsf_sky_near_arr = np.asarray(hdul["LSF_SKY_NEAR"].data, dtype=np.float64)
    lsf_sky_far_arr = np.asarray(hdul["LSF_SKY_FAR"].data, dtype=np.float64)

n_spec, n_wave = flux_sci_true_all.shape
if row_index_e10.size != n_e10:
    raise ValueError(f"Triplet row_index length mismatch: {row_index_e10.size} vs n_rows={n_e10}")
if np.any(row_index_e10 < 0) or np.any(row_index_e10 >= n_spec):
    raise ValueError(
        f"Triplet row_index contains values outside [0, {n_spec - 1}] for {EVERY10_INPUT}"
    )

idx_row = int(REQUESTED_ROW)
triplet_pos = np.flatnonzero(row_index_e10 == idx_row)
if triplet_pos.size == 0:
    raise IndexError(
        f"REQUESTED_ROW={idx_row} is not available in aligned triplet rows. "
        f"Choose one of e10_triplet['row_index'] (size={row_index_e10.size})."
    )
triplet_pos = int(triplet_pos[0])

# Normalize WAVE/LSF arrays to per-row vectors, then select requested row.
wave_row = wave_arr if wave_arr.ndim == 1 else wave_arr[idx_row]
lsf_row = lsf_sci_arr if lsf_sci_arr.ndim == 1 else lsf_sci_arr[idx_row]
lsf_row_near = lsf_sky_near_arr if lsf_sky_near_arr.ndim == 1 else lsf_sky_near_arr[idx_row]
lsf_row_far = lsf_sky_far_arr if lsf_sky_far_arr.ndim == 1 else lsf_sky_far_arr[idx_row]

flux_near_row = flux_near_all[idx_row]
flux_far_row = flux_far_all[idx_row]
flux_sci_true_row = flux_sci_true_all[idx_row]

# 3) Predict SCI coefficients for the requested row using global default path.
coef_pred_row_batch = predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=e10_triplet["coef_near"][triplet_pos: triplet_pos + 1],
    coef_far_phys=e10_triplet["coef_far"][triplet_pos: triplet_pos + 1],
    ctx_near_phys=e10_triplet["ctx_near"][triplet_pos: triplet_pos + 1],
    ctx_far_phys=e10_triplet["ctx_far"][triplet_pos: triplet_pos + 1],
    ctx_sci_phys=e10_triplet["ctx_sci"][triplet_pos: triplet_pos + 1],
)
coef_pred_row = np.asarray(coef_pred_row_batch[0], dtype=np.float64)

# 3a) Context values for the near-sky, far-sky and science pointings at this row.
#     Cyclic sin/cos pairs are folded back to 0-360 degree axes for readability.
_ctx_names_e10 = list(e10_triplet["ctx_names"])
_ctx_row_stack = np.stack([
    e10_triplet["ctx_near"][triplet_pos],
    e10_triplet["ctx_far"][triplet_pos],
    e10_triplet["ctx_sci"][triplet_pos],
], axis=0)
_ctx_disp_names, _ctx_disp_stack = _decode_cyclic_context(_ctx_names_e10, _ctx_row_stack)
_ctx_row_df = pd.DataFrame({
    "feature": _ctx_disp_names,
    "sky_near": _ctx_disp_stack[0],
    "sky_far": _ctx_disp_stack[1],
    "science": _ctx_disp_stack[2],
})
print(f"Context values at row {idx_row} (sin/cos pairs decoded to degrees):")
print(_ctx_row_df.to_string(index=False, float_format=lambda v: f'{v:.4g}'))
print()

# 3b) Moon_bs coefficient diagnostics (global prior already applied).
coef_near_row = np.asarray(e10_triplet["coef_near"][triplet_pos], dtype=np.float64)
coef_far_row = np.asarray(e10_triplet["coef_far"][triplet_pos], dtype=np.float64)
coef_sci_row = np.asarray(e10_triplet["coef_sci"][triplet_pos], dtype=np.float64)

# Per-row coef_err arrays (LSF-propagated sigma feeds off these when the
# decomposition FITS lacks a FLUX_SIGMA_TOTAL HDU).  Missing HDU -> None,
# and the downstream WRMSE degrades to the median-floor path.
def _row_coef_err(triplet_key):
    _arr = e10_triplet.get(triplet_key)
    if _arr is None:
        return None
    _row = np.asarray(_arr[triplet_pos], dtype=np.float64)
    return _row if np.any(np.isfinite(_row)) else None

coef_err_near_row = _row_coef_err("coef_err_near")
coef_err_far_row  = _row_coef_err("coef_err_far")
coef_err_sci_row  = _row_coef_err("coef_err_sci")
moon_pred = coef_pred_row[moon_idx]
moon_near = coef_near_row[moon_idx]
moon_far = coef_far_row[moon_idx]
moon_true = coef_sci_row[moon_idx]

# 4) Reconstruct this row for SCI prediction and near/far self-consistency checks.
base_dir_guess = _infer_base_dir_for_reconstruction()

# Prefer the fitted wavelength-dependent LSF surface from each decomp file's
# LSF_COEF/LSF_KNOTS/LSF_META extensions (written by sky_decomp.lsf_surface_iterative);
# fall back to the input FITS's Gaussian LSF_SCI if that isn't present.
_lsf_state_near = load_lsf_state_if_available(EVERY10_NEAR, idx_row)
_lsf_state_far  = load_lsf_state_if_available(EVERY10_FAR,  idx_row)
_lsf_state_sci  = load_lsf_state_if_available(EVERY10_SCI,  idx_row)
_mz_state_near = load_moon_zodi_state_if_available(EVERY10_NEAR, idx_row)
_mz_state_far  = load_moon_zodi_state_if_available(EVERY10_FAR,  idx_row)
_mz_state_sci  = load_moon_zodi_state_if_available(EVERY10_SCI,  idx_row)
print(f"  Moon/Zodi state per arm (row {idx_row}): "
      f"near={'yes' if _mz_state_near is not None else 'no (moon spline)'}, "
      f"far={'yes' if _mz_state_far is not None else 'no (moon spline)'}, "
      f"sci={'yes' if _mz_state_sci is not None else 'no (moon spline)'}")
_lsf_sigma_fallback = lsf_row / 2.35
print(f"  LSF source per arm (row {idx_row}): "
      f"near={'surface' if _lsf_state_near is not None else 'gaussian (LSF_SCI)'}, "
      f"far={'surface' if _lsf_state_far is not None else 'gaussian (LSF_SCI)'}, "
      f"sci={'surface' if _lsf_state_sci is not None else 'gaussian (LSF_SCI)'}")

# Per-arm unit-integrated O2 templates from the decomposition FITS
# (VECTOR_O2 extension). If absent (older decomp), the O2 basis stays at
# zero -- matching pre-2026-08-10 behaviour. For the predicted-sci
# reconstruction, use the sci-arm template so the shape is anchored on the
# same layer temperature the science pointing had; the amplitude comes
# from the predicted coef['O2_b01'].
_o2_vec_near = load_o2_vector_if_available(EVERY10_NEAR, idx_row)
_o2_vec_far  = load_o2_vector_if_available(EVERY10_FAR,  idx_row)
_o2_vec_sci  = load_o2_vector_if_available(EVERY10_SCI,  idx_row)
print(f"  O2 template per arm (row {idx_row}): "
      f"near={'VECTOR_O2' if _o2_vec_near is not None else 'zero'}, "
      f"far={'VECTOR_O2' if _o2_vec_far is not None else 'zero'}, "
      f"sci={'VECTOR_O2' if _o2_vec_sci is not None else 'zero'}")

comps_sci = reconstruct_with_lsf(
    wave=wave_row,
    coef=coef_pred_row,
    lsf=_lsf_state_sci if _lsf_state_sci is not None else _lsf_sigma_fallback,
    n_spline_knots=25,
    base_dir=base_dir_guess,
    o2_vector=_o2_vec_sci,
    moon_zodi_state=_mz_state_sci,
    detector_lsf_fwhm=lsf_row,
    physical_to_fit_flux_scale=FACTOR,
)
comps_near_from_near = reconstruct_with_lsf(
    wave=wave_row,
    coef=coef_near_row,
    lsf=_lsf_state_near if _lsf_state_near is not None else _lsf_sigma_fallback,
    n_spline_knots=25,
    base_dir=base_dir_guess,
    o2_vector=_o2_vec_near,
    coef_err=coef_err_near_row,
    moon_zodi_state=_mz_state_near,
    detector_lsf_fwhm=lsf_row_near,
    physical_to_fit_flux_scale=FACTOR,
)
comps_far_from_far = reconstruct_with_lsf(
    wave=wave_row,
    coef=coef_far_row,
    lsf=_lsf_state_far if _lsf_state_far is not None else _lsf_sigma_fallback,
    n_spline_knots=25,
    base_dir=base_dir_guess,
    o2_vector=_o2_vec_far,
    coef_err=coef_err_far_row,
    moon_zodi_state=_mz_state_far,
    detector_lsf_fwhm=lsf_row_far,
    physical_to_fit_flux_scale=FACTOR,
)
# Reconstruction of the observed sci spectrum from the fitted sci coefs, so panel 3
# can separate the sky-decomposition fit residual (obs vs recon-from-sci-coef)
# from our transfer-model error (recon-from-pred vs recon-from-sci-coef).
comps_sci_true = reconstruct_with_lsf(
    wave=wave_row,
    coef=coef_sci_row,
    lsf=_lsf_state_sci if _lsf_state_sci is not None else _lsf_sigma_fallback,
    n_spline_knots=25,
    base_dir=base_dir_guess,
    o2_vector=_o2_vec_sci,
    coef_err=coef_err_sci_row,
    moon_zodi_state=_mz_state_sci,
    detector_lsf_fwhm=lsf_row,
    physical_to_fit_flux_scale=FACTOR,
)

flux_sci_pred_row = np.asarray(comps_sci["total"], dtype=np.float64) / FACTOR
flux_sci_true_recon_row = np.asarray(comps_sci_true["total"], dtype=np.float64) / FACTOR
flux_near_recon_row = np.asarray(comps_near_from_near["total"], dtype=np.float64) / FACTOR
flux_far_recon_row = np.asarray(comps_far_from_far["total"], dtype=np.float64) / FACTOR

# 5) Single-row metrics.
resid_row = flux_sci_pred_row - flux_sci_true_row
rmse_row = float(np.sqrt(np.mean(resid_row ** 2)))
rmse_row_display = float(rmse_row * FACTOR)
mae_row = float(np.mean(np.abs(resid_row)))
rel_resid_row = resid_row / np.where(flux_sci_true_row != 0, flux_sci_true_row, np.nan)

rmse_near_recon = float(np.sqrt(np.mean((flux_near_recon_row - flux_near_row) ** 2)))
rmse_far_recon = float(np.sqrt(np.mean((flux_far_recon_row - flux_far_row) ** 2)))

# Pixel-space WRMSE for the same three arms.  Sigma source order:
#   1. FLUX_SIGMA_TOTAL HDU in the decomposition FITS (future LSF-aware
#      propagator output);
#   2. sigma_total returned by `reconstruct_component_spectra(coef_err=...)`
#      when the corresponding COEF_ERR array is available on this row;
#   3. None -> pixel_wrmse_per_row degrades to a per-row median-floor RMSE.
def _sigma_for_single_row(fits_path, hdu_row_idx, comps_dict, comps_true_dict=None):
    """Prefer FITS HDU sigma; else use comps sigma_total (from coef_err)."""
    _fits_sigma = load_pixel_sigma_if_available(fits_path,
                                                row_indices=[int(hdu_row_idx)])
    if _fits_sigma is not None:
        return np.asarray(_fits_sigma[0], dtype=np.float64) / FACTOR
    # Fallback: use the sigma_total attached to the reconstructed comps dict.
    _sig = comps_dict.get("sigma_total") if isinstance(comps_dict, dict) else None
    if _sig is not None:
        return np.asarray(_sig, dtype=np.float64) / FACTOR
    # As a last resort, try the "true" recomposition's sigma (only meaningful
    # for the sci arm where the true decomposition sigma is more directly
    # comparable to the observation noise floor).
    _sig_alt = (comps_true_dict.get("sigma_total")
                if isinstance(comps_true_dict, dict) else None)
    if _sig_alt is not None:
        return np.asarray(_sig_alt, dtype=np.float64) / FACTOR
    return None

_sig_near_pix = _sigma_for_single_row(EVERY10_NEAR, idx_row, comps_near_from_near)
_sig_far_pix  = _sigma_for_single_row(EVERY10_FAR,  idx_row, comps_far_from_far)
_sig_sci_pix  = _sigma_for_single_row(EVERY10_SCI,  idx_row, comps_sci, comps_sci_true)

wrmse_near_recon = float(pixel_wrmse_per_row(
    flux_near_recon_row, flux_near_row, _sig_near_pix)[0])
wrmse_far_recon  = float(pixel_wrmse_per_row(
    flux_far_recon_row,  flux_far_row,  _sig_far_pix)[0])
wrmse_row_pix    = float(pixel_wrmse_per_row(
    flux_sci_pred_row,   flux_sci_true_row, _sig_sci_pix)[0])

print("Single-row reconstruction summary (every10, default coefficients)")
print(f"  row index (input file) = {idx_row}")
print(f"  row index (triplet pos) = {triplet_pos}")
print(f"  n_wave     = {n_wave}")
print("  predictor  = deep group-head MLP")
print(
    "  Moon_bs roughness: "
    f"pred={_row_spline_roughness(moon_pred):.4g}, "
    f"near={_row_spline_roughness(moon_near):.4g}, "
    f"far={_row_spline_roughness(moon_far):.4g}, "
    f"sci_true={_row_spline_roughness(moon_true):.4g}"
)
print(f"  near self-recon pRMSE  = {rmse_near_recon:.6g}")
print(f"  near self-recon pWRMSE = {wrmse_near_recon:.6g}")
print(f"  far  self-recon pRMSE  = {rmse_far_recon:.6g}")
print(f"  far  self-recon pWRMSE = {wrmse_far_recon:.6g}")
print(f"  sci row pRMSE          = {rmse_row:.6g}")
print(f"  sci row pWRMSE         = {wrmse_row_pix:.6g}")
print(f"  sci row pRMSE (x{FACTOR:.3g} display units) = {rmse_row_display:.6g}")
print(f"  sci row MAE            = {mae_row:.6g}")

# 6) Four-panel diagnostic plot:
#    row1: near observed vs reconstruction from near coefficients
#    row2: far observed vs reconstruction from far coefficients
#    row3: science true vs science prediction
#    row4: science relative residual
fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=(
        "Near: observed vs reconstructed from near coefficients",
        "Far: observed vs reconstructed from far coefficients",
        "Science: observed / recon(sci coef) / recon(pred)",
        "Science residual: (pred - true) / true",
    ),
    row_heights=[0.24, 0.24, 0.34, 0.18],
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_near_row * FACTOR,
        mode="lines",
        name="near true",
        line=dict(color="#7f7f7f", width=1.0),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_near_recon_row * FACTOR,
        mode="lines",
        name="near recon(from near coef)",
        line=dict(color="#e41a1c", width=1.4),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_far_row * FACTOR,
        mode="lines",
        name="far true",
        line=dict(color="#7f7f7f", width=1.0),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_far_recon_row * FACTOR,
        mode="lines",
        name="far recon(from far coef)",
        line=dict(color="#ff7f00", width=1.4),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_sci_true_row * FACTOR,
        mode="lines",
        name="science observed",
        line=dict(color="#7f7f7f", width=1.0),
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_sci_true_recon_row * FACTOR,
        mode="lines",
        name="science recon(from sci coef)",
        line=dict(color="#2ca02c", width=1.2, dash="dash"),
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=flux_sci_pred_row * FACTOR,
        mode="lines",
        name="science recon(pred)",
        line=dict(color="#1f78b4", width=1.4),
    ),
    row=3,
    col=1,
)

fig.add_trace(
    go.Scattergl(
        x=wave_row,
        y=rel_resid_row,
        mode="lines",
        name="science residual",
        line=dict(color="#d62728", width=1.0),
    ),
    row=4,
    col=1,
)
fig.add_hline(y=0, line=dict(color="black", width=0.8, dash="dash"), row=4, col=1)

fig.update_yaxes(type="log", title_text="Near flux", row=1, col=1)
fig.update_yaxes(type="log", title_text="Far flux", row=2, col=1)
fig.update_yaxes(type="log", title_text="Science flux", row=3, col=1)
fig.update_yaxes(type="linear", title_text="(pred-true)/true", row=4, col=1)
fig.update_xaxes(title_text="Wavelength [A]", row=4, col=1)

fig.update_layout(
    template="plotly_white",
    title=(
        f"Every10 row {idx_row} | near pRMSE={rmse_near_recon:.3g} "
        f"(pWRMSE={wrmse_near_recon:.3g}), far pRMSE={rmse_far_recon:.3g} "
        f"(pWRMSE={wrmse_far_recon:.3g}), sci pRMSE={rmse_row:.3g} "
        f"(pWRMSE={wrmse_row_pix:.3g}, disp={rmse_row_display:.3g})"
    ),
    height=1160,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig.show()

# 7) Moon spline coefficient diagnostic figure (global-prior result).
moon_axis = np.arange(moon_idx.size)
fig_moon = go.Figure()
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_near,
        mode="lines+markers",
        name="near",
        line=dict(color="#7f7f7f"),
    )
)
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_far,
        mode="lines+markers",
        name="far",
        line=dict(color="#bdbdbd"),
    )
)
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_true,
        mode="lines+markers",
        name="sci true",
        line=dict(color="#1f78b4"),
    )
)
fig_moon.add_trace(
    go.Scatter(
        x=moon_axis,
        y=moon_pred,
        mode="lines+markers",
        name="pred default",
        line=dict(color="#e41a1c"),
    )
)
fig_moon.update_layout(
    template="plotly_white",
    title="Moon spline coefficients (base prediction) for selected row",
    xaxis_title="Moon_bs coefficient index",
    yaxis_title="coefficient value",
    height=420,
)
fig_moon.show()

# 8) Per-component reconstructions for all four arms, mirroring the
#    four-trace layout of the Moon spline diagnostic above but as
#    spectra over wavelength rather than coefficients over index.
#    comps_sci_true was reconstructed in section 4 so row 3 can use it.
_comps_by_arm = {
    "near": comps_near_from_near,
    "far": comps_far_from_far,
    "sci true": comps_sci_true,
    "pred default": comps_sci,
}
_arm_colors = {
    "near": "#7f7f7f",
    "far": "#bdbdbd",
    "sci true": "#1f78b4",
    "pred default": "#e41a1c",
}

# comps["*"] is in the same display scale as comps["total"], so no *FACTOR here.
def _non_moon_continuum(comps):
    return np.asarray(comps["diffuse"], dtype=np.float64)

def _line_component(comps):
    return (np.asarray(comps["oh"], dtype=np.float64)
            + np.asarray(comps["atom"], dtype=np.float64)
            + np.asarray(comps["orc"], dtype=np.float64)
            + np.asarray(comps["o2"], dtype=np.float64))

def _moon_spectrum(comps):
    # 2026-08-18 (moon+zodi variant): combine moon + zodi so this diagnostic
    # is apples-to-apples with the previous moon-only model, where a single
    # "moon" component absorbed both moon and diffuse zodiacal light.
    moon = np.asarray(comps["moon"], dtype=np.float64)
    if "zodi" in comps:
        moon = moon + np.asarray(comps["zodi"], dtype=np.float64)
    return moon

# Sanity: total is defined by reconstruct_component_spectra as
#   oh + moon + diffuse + atom + orc + o2
# so lines + moon_spectrum + non_moon_continuum must equal it.
_total_pred = np.asarray(comps_sci["total"], dtype=np.float64)
_sum_pred = (_line_component(comps_sci)
             + _moon_spectrum(comps_sci)
             + _non_moon_continuum(comps_sci))
_max_diff = float(np.nanmax(np.abs(_total_pred - _sum_pred)))
_max_rel = float(np.nanmax(np.abs(_total_pred - _sum_pred)
                          / np.clip(np.abs(_total_pred), 1e-30, None)))
print(f"Component-sum check (pred): max abs diff = {_max_diff:.3g}, "
      f"max rel diff = {_max_rel:.3g}")

fig_continuum = go.Figure()
for _arm, _comps in _comps_by_arm.items():
    fig_continuum.add_trace(
        go.Scattergl(
            x=wave_row,
            y=_non_moon_continuum(_comps),
            mode="lines",
            name=_arm,
            line=dict(color=_arm_colors[_arm], width=1.2),
        )
    )
fig_continuum.update_layout(
    template="plotly_white",
    title="Reconstructed non-moon continuum (diffuse = HO2 + FeO + O2ac) for selected row",
    xaxis_title="Wavelength [A]",
    yaxis_title=f"Flux (display units x{FACTOR:.3g})",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig_continuum.show()

fig_moon_spectrum = go.Figure()
for _arm, _comps in _comps_by_arm.items():
    fig_moon_spectrum.add_trace(
        go.Scattergl(
            x=wave_row,
            y=_moon_spectrum(_comps),
            mode="lines",
            name=_arm,
            line=dict(color=_arm_colors[_arm], width=1.2),
        )
    )
fig_moon_spectrum.update_layout(
    template="plotly_white",
    title="Reconstructed moon spline spectrum (comps['moon']) for selected row",
    xaxis_title="Wavelength [A]",
    yaxis_title=f"Flux (display units x{FACTOR:.3g})",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig_moon_spectrum.show()

fig_lines = go.Figure()
for _arm, _comps in _comps_by_arm.items():
    fig_lines.add_trace(
        go.Scattergl(
            x=wave_row,
            y=_line_component(_comps),
            mode="lines",
            name=_arm,
            line=dict(color=_arm_colors[_arm], width=1.2),
        )
    )
fig_lines.update_layout(
    template="plotly_white",
    title="Reconstructed line emission (OH + atom + ORC + O2) for selected row",
    xaxis_title="Wavelength [A]",
    yaxis_title=f"Flux (display units x{FACTOR:.3g})",
    height=460,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig_lines.show()

# 9) Per-component (pred - sci_true) residual spectrum. Row 3 shows only
#    the total residual; this decomposes it into moon / diffuse / lines
#    so a small broadband deficit in a component that spans the whole
#    wavelength range is visible even when the component panels above
#    make it look "close" on a linear-y axis. By construction the three
#    traces must sum to the blue-minus-green curve of row 3, and that
#    sum is drawn as a black dashed reference.
_delta_moon = _moon_spectrum(comps_sci) - _moon_spectrum(comps_sci_true)
_delta_diffuse = _non_moon_continuum(comps_sci) - _non_moon_continuum(comps_sci_true)
_delta_lines = _line_component(comps_sci) - _line_component(comps_sci_true)
_delta_total = _delta_moon + _delta_diffuse + _delta_lines

fig_deltas = go.Figure()
for _label, _y, _color in (
    ("moon (pred - sci recon)", _delta_moon, "#e41a1c"),
    ("diffuse (pred - sci recon)", _delta_diffuse, "#377eb8"),
    ("lines (pred - sci recon)", _delta_lines, "#4daf4a"),
    ("total (pred - sci recon)", _delta_total, "#000000"),
):
    fig_deltas.add_trace(
        go.Scattergl(
            x=wave_row,
            y=_y,
            mode="lines",
            name=_label,
            line=dict(color=_color, width=1.2,
                      dash="dash" if _label.startswith("total") else "solid"),
        )
    )
fig_deltas.add_hline(y=0, line=dict(color="rgba(0,0,0,0.4)", width=0.8, dash="dot"))
fig_deltas.update_layout(
    template="plotly_white",
    title="Per-component prediction minus sci-arm reconstruction (linear)",
    xaxis_title="Wavelength [A]",
    yaxis_title=f"Delta flux (display units x{FACTOR:.3g})",
    height=460,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig_deltas.show()

# 10) Numeric integrated deltas per component in three wavelength bands, so
#     the sign and magnitude of each contribution to the red deficit is
#     visible even where the plot traces are noisy line-by-line.
_bands = [
    ("blue  (< 5500 A)", wave_row < 5500.0),
    ("green (5500-7500)", (wave_row >= 5500.0) & (wave_row < 7500.0)),
    ("red   (>= 7500 A)", wave_row >= 7500.0),
]
print()
print("Integrated (pred - sci recon) per component and wavelength band:")
print(f"  {'band':<18s} {'moon':>12s} {'diffuse':>12s} {'lines':>12s} {'total':>12s}")
for _bname, _mask in _bands:
    if not _mask.any():
        continue
    _sm = float(np.nansum(_delta_moon[_mask]))
    _sd = float(np.nansum(_delta_diffuse[_mask]))
    _sl = float(np.nansum(_delta_lines[_mask]))
    _st = float(np.nansum(_delta_total[_mask]))
    print(f"  {_bname:<18s} {_sm:>+12.4g} {_sd:>+12.4g} {_sl:>+12.4g} {_st:>+12.4g}")



In [ ]:
# Row-specific diagnostic: is the sci-arm decomposition inflating moon+zodi at
# red wavelengths, or is the truth-fit consistent with near/far AND consistent
# with the observed sci flux? Consumes globals from the single-row diagnostic
# cell above.
_required_mz_probe = [
    "wave_row", "flux_sci_true_row", "comps_sci_true", "comps_sci",
    "comps_near_from_near", "comps_far_from_far", "FACTOR", "idx_row",
]
_miss = [k for k in _required_mz_probe if k not in globals()]
if _miss:
    raise RuntimeError(
        f"Missing globals: {_miss}. Run the single-row diagnostic cell above first.")

def _mz_flux(_comps):
    _m = np.asarray(_comps["moon"], dtype=np.float64)
    if "zodi" in _comps:
        _m = _m + np.asarray(_comps["zodi"], dtype=np.float64)
    return _m

_probe_bands = [
    ("B_cont",   4000.0, 4500.0),
    ("G_cont",   5450.0, 5700.0),
    ("R_cont",   7000.0, 7300.0),
    ("NIR_cont", 9600.0, 9900.0),
    ("red_full", 7500.0, 10000.0),
]

_wave = np.asarray(wave_row, dtype=np.float64)
# flux_sci_true_row is the observed sci spectrum in raw flux units; scale to
# match the FACTOR display units the reconstructions live in.
_obs = np.asarray(flux_sci_true_row, dtype=np.float64) * FACTOR
_mz_true = _mz_flux(comps_sci_true)
_mz_pred = _mz_flux(comps_sci)
_mz_near = _mz_flux(comps_near_from_near)
_mz_far = _mz_flux(comps_far_from_far)
_tot_true = np.asarray(comps_sci_true["total"], dtype=np.float64)
_tot_pred = np.asarray(comps_sci["total"], dtype=np.float64)

print(f"Row {idx_row}: sci moon+zodi vs sky arms vs observed sci "
      f"(band-mean flux in display units, x{FACTOR:.0e})")
print(f"  {'band':<10s} {'obs_sci':>10s} {'tot_true':>10s} {'unmod':>10s} "
      f"{'mz_true':>10s} {'mz_pred':>10s} {'mz_near':>10s} {'mz_far':>10s} "
      f"{'true/near':>10s} {'true/far':>9s}")
for _bname, _lo, _hi in _probe_bands:
    _bmask = (_wave >= _lo) & (_wave < _hi)
    if not _bmask.any():
        continue
    _o = float(_obs[_bmask].mean())
    _tt = float(_tot_true[_bmask].mean())
    _tp = float(_tot_pred[_bmask].mean())
    _mt = float(_mz_true[_bmask].mean())
    _mp = float(_mz_pred[_bmask].mean())
    _mn = float(_mz_near[_bmask].mean())
    _mf = float(_mz_far[_bmask].mean())
    _unmod = _o - _tt
    _rn = _mt / _mn if abs(_mn) > 1e-30 else np.nan
    _rf = _mt / _mf if abs(_mf) > 1e-30 else np.nan
    print(f"  {_bname:<10s} {_o:>10.3g} {_tt:>10.3g} {_unmod:>+10.3g} "
          f"{_mt:>10.3g} {_mp:>10.3g} {_mn:>10.3g} {_mf:>10.3g} "
          f"{_rn:>10.2f} {_rf:>9.2f}")

print()
print("Legend:")
print("  tot_true  = obs_sci reconstructed from ALL true sci coefs (moon+zodi + diffuse + lines).")
print("  unmod     = obs_sci - tot_true. Near-zero => sci decomposition adequately explains the obs.")
print("              Non-zero => decomposition is leaving flux on the table (fit residual).")
print("  true/near, true/far = sci moon+zodi vs sky moon+zodi in the SAME band.")
print("              ~1  => shared airglow/scattering physics; sci decomp is consistent with sky arms.")
print("              >>1 => sci decomp is inflating moon+zodi vs the two sky arms.")


In [ ]:
# Batch SSFR prep on a random subset of every10 rows -- 2026-08-18 (v3):
# per-row pRMSE / pWRMSE reporting was removed; the useful signal was the
# per-component per-band SSFR bias table produced by the next cell. This cell
# now just: (a) loads the every10 triplet, (b) applies field/chi2 filters,
# (c) predicts sci coefs for a phase-stratified sample, and (d) hoists the
# reconstruction helpers (_fast_reconstruct + LSF / MZ / O2 caches) so the
# SSFR cell can reuse them. See §5 in the top notebook doc for SSFR itself.
import numpy as np
import pandas as pd
import plotly.graph_objects as go

RUN_SSFR_PREP = True  # Set True to build the SSFR prep context. Formerly RUN_RMSE_SUBSET_EVAL.

required = [
    "mlp_artifacts",
    "predict_sci_coefficients_default",
    "context_cols",
    "build_triplet_coef_dataset",
    "reconstruct_component_spectra",
    "reconstruct_with_lsf",
    "load_lsf_state_if_available",
    "load_moon_zodi_state_if_available",
    "load_o2_vector_if_available",
    "_infer_base_dir_for_reconstruction",
    "FACTOR",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run the training + residual-correction cells first. Missing: " + ", ".join(missing))

if not RUN_SSFR_PREP:
    print("SSFR prep skipped. Set RUN_SSFR_PREP = True to build the sample + hoist reconstruction helpers.")
else:
    # Use the same file inputs as the reconstruction diagnostic cell.
    _e10_stem = "lvmsframe_median_stack_1.2.1_p40_p70_every10"
    _e10_suffix = _DECOMP_SUFFIX  # inherited from cell 6
    EVERY10_INPUT = f"{_e10_stem}.fits"
    EVERY10_NEAR = f"{_e10_stem}_decomp_sky1{_e10_suffix}.fits"
    EVERY10_FAR = f"{_e10_stem}_decomp_sky2{_e10_suffix}.fits"
    EVERY10_SCI = f"{_e10_stem}_decomp_sci{_e10_suffix}.fits"

    # SSFR statistics use the full sample; the residual trace plot in cell 21
    # subsamples to ~100 rows (phase-stratified) so the overlay stays legible.
    n_sample = 1000
    rng_seed = 42

    # 1) Build aligned triplet rows, keeping chi2 so we can apply the same
    #    quality gates the training set went through.
    e10_triplet = build_triplet_coef_dataset(
        input_fits_path=EVERY10_INPUT,
        sky_near_decomp_fits_path=EVERY10_NEAR,
        sky_far_decomp_fits_path=EVERY10_FAR,
        sci_decomp_fits_path=EVERY10_SCI,
        context_columns=context_cols,
        return_chi2=True,
    )
    # ECLIPTIC-CTX-V1: match training-time ctx layout on the e10 triplet.
    if '_augment_triplet_with_ecliptic' in globals():
        _augment_triplet_with_ecliptic(e10_triplet)

    row_index_e10 = np.asarray(e10_triplet["row_index"], dtype=np.int64)
    _e10_n0 = int(e10_triplet["n_rows"])
    if _e10_n0 == 0:
        raise RuntimeError("No aligned rows available in e10_triplet")

    # 1a) Apply chi2 gating and LMC/SMC field exclusion BEFORE the random
    #     subsample, so the reconstructions we score are drawn from tiles
    #     that pass the same data-quality gates the training set went
    #     through.  Hard coefficient bounds and kappa-sigma clipping are
    #     intentionally NOT applied here -- those are training-time filters
    #     on the target that would remove the model's hardest true cases
    #     from the evaluation set.
    _e10_keep = np.ones(_e10_n0, dtype=bool)

    if "sci_ra" in e10_triplet and "sci_dec" in e10_triplet:
        _sci_ra = np.asarray(e10_triplet["sci_ra"], dtype=np.float64)
        _sci_dec = np.asarray(e10_triplet["sci_dec"], dtype=np.float64)
        _field_mask = np.ones(_e10_n0, dtype=bool)
        for _region in (LMC_EXCLUSION, SMC_EXCLUSION):
            _sep = _angular_separation_deg_vec(
                _sci_ra, _sci_dec,
                _region["ra_deg"], _region["dec_deg"])
            _inside = np.isfinite(_sep) & (_sep <= float(_region["radius_deg"]))
            print(
                f"  every10 field exclusion around {_region['name']}: "
                f"excluded {int(_inside.sum())}/{_e10_n0}"
            )
            _field_mask &= ~_inside
        _e10_keep &= _field_mask
    else:
        print("  every10 field exclusion skipped: no sci_ra/sci_dec in triplet.")

    if all(_k in e10_triplet for _k in ("chi2_near", "chi2_far", "chi2_sci")):
        _chi2_combined = np.nanmax(
            np.column_stack([
                np.asarray(e10_triplet["chi2_near"], dtype=np.float64),
                np.asarray(e10_triplet["chi2_far"], dtype=np.float64),
                np.asarray(e10_triplet["chi2_sci"], dtype=np.float64),
            ]),
            axis=1,
        )
        _chi2_finite = _chi2_combined[np.isfinite(_chi2_combined)]
        _chi2_hi = (float(np.nanpercentile(_chi2_finite, 90.0))
                    if _chi2_finite.size else np.inf)
        _chi2_upper = min(10.0, _chi2_hi)
        _chi2_mask = (np.isfinite(_chi2_combined)
                      & (_chi2_combined >= 0.0)
                      & (_chi2_combined <= _chi2_upper))
        print(
            f"  every10 chi2 filter (qmax=90% -> {_chi2_hi:.3g}, "
            f"upper={_chi2_upper:.3g}): "
            f"kept {int(_chi2_mask.sum())}/{_e10_n0}"
        )
        _e10_keep &= _chi2_mask
    else:
        print("  every10 chi2 filter skipped: chi2 columns not in triplet.")

    _e10_valid_pos = np.flatnonzero(_e10_keep)
    n_rows = int(_e10_valid_pos.size)
    print(
        f"  every10 rows passing chi2 + field exclusion: "
        f"{n_rows}/{_e10_n0} ({100.0 * n_rows / max(_e10_n0, 1):.1f}%)"
    )
    if n_rows == 0:
        raise RuntimeError(
            "No aligned rows survive chi2/field filtering; relax thresholds.")

    n_use = int(min(n_sample, n_rows))
    rng = np.random.default_rng(rng_seed)

    # Stratify the draw by lunar phase so the diagnostic set spans dark -> bright
    # roughly uniformly (matches split_indices_by_moon_phase in §8.1). A plain
    # rng.choice over _e10_valid_pos would inherit the phase distribution of the
    # dataset -- weighted toward whichever quantiles happen to hold more filtered
    # rows -- and the sci_pred_vs_true RMSE would then be dominated by that
    # region rather than being representative of deployment conditions.
    _e10_moon_phase = _moon_phase_deg_from_ctx(e10_triplet)
    _valid_phase = _e10_moon_phase[_e10_valid_pos]
    if not np.isfinite(_valid_phase).all():
        raise RuntimeError('Non-finite moon_phase in the valid every10 rows; '
                           'cannot stratify by lunar phase.')

    _n_phase_bins = int(min(10, n_use))
    _phase_edges = np.quantile(_valid_phase,
                               np.linspace(0.0, 1.0, _n_phase_bins + 1))
    _phase_edges[0], _phase_edges[-1] = -np.inf, np.inf
    _bin_id = np.digitize(_valid_phase, _phase_edges[1:-1], right=False)

    # Round-robin quota with the +1s scattered randomly so no bin is systematically favored.
    _quota = np.full(_n_phase_bins, n_use // _n_phase_bins, dtype=int)
    _quota[:n_use - int(_quota.sum())] += 1
    rng.shuffle(_quota)

    _picked = []
    for _b in range(_n_phase_bins):
        _in_bin = _e10_valid_pos[_bin_id == _b]
        _take = int(min(_quota[_b], _in_bin.size))
        if _take > 0:
            _picked.append(rng.choice(_in_bin, size=_take, replace=False))
    _selected = (np.concatenate(_picked).astype(int)
                 if _picked else np.array([], dtype=int))

    # If any bin was smaller than its quota, backfill from the remaining pool.
    _shortfall = n_use - _selected.size
    if _shortfall > 0:
        _remaining = np.setdiff1d(_e10_valid_pos, _selected, assume_unique=False)
        if _remaining.size >= _shortfall:
            _selected = np.concatenate(
                [_selected, rng.choice(_remaining, size=_shortfall, replace=False)])

    sel_pos = np.sort(_selected)
    sel_rows = row_index_e10[sel_pos]

    _sel_phases = _e10_moon_phase[sel_pos]
    print(f"  phase-stratified sample: n_use={n_use} across {_n_phase_bins} "
          f"quantile bins; phase deg quartiles (min / 25 / 50 / 75 / max) = "
          f"{float(np.min(_sel_phases)):.1f} / "
          f"{float(np.percentile(_sel_phases, 25)):.1f} / "
          f"{float(np.percentile(_sel_phases, 50)):.1f} / "
          f"{float(np.percentile(_sel_phases, 75)):.1f} / "
          f"{float(np.max(_sel_phases)):.1f}")

    # 2) Load observed spectra, wavelength grid, and LSF from the same input file.
    with fits.open(EVERY10_INPUT) as hdul:
        flux_near_all = np.asarray(hdul["FLUX_SKY_NEAR"].data, dtype=np.float64)
        flux_far_all = np.asarray(hdul["FLUX_SKY_FAR"].data, dtype=np.float64)
        flux_sci_all = np.asarray(hdul["FLUX_SCI"].data, dtype=np.float64)
        wave_arr = np.asarray(hdul["WAVE"].data, dtype=np.float64)
        lsf_sci_arr = np.asarray(hdul["LSF_SCI"].data, dtype=np.float64)
        lsf_sky_near_arr = np.asarray(hdul["LSF_SKY_NEAR"].data, dtype=np.float64)
        lsf_sky_far_arr = np.asarray(hdul["LSF_SKY_FAR"].data, dtype=np.float64)
        # expnum for the residual-trace + bias-vs-context tooltips (cells 21-22).
        _expnum_all = None
        if "META" in hdul:
            _meta_e10 = Table(hdul["META"].data)
            _meta_up_e10 = {c.upper(): c for c in _meta_e10.colnames}
            _expnum_col = next(
                (_meta_up_e10[k] for k in ("EXPNUM", "EXP_NUM", "EXPOSURE")
                 if k in _meta_up_e10), None)
            if _expnum_col is not None:
                _expnum_all = np.asarray(_meta_e10[_expnum_col])
                print(f"  expnum loaded from META[{_expnum_col!r}]: "
                      f"n_rows={_expnum_all.size}")
            else:
                print("  expnum column not found in META "
                      "(tried EXPNUM / EXP_NUM / EXPOSURE)")
        else:
            print("  META extension not found in every10 FITS; expnum unavailable.")

    n_spec = int(flux_sci_all.shape[0])
    if np.any(sel_rows < 0) or np.any(sel_rows >= n_spec):
        raise ValueError("Selected row index is outside the valid range of EVERY10_INPUT")


    # 3) Predict SCI coefficients for sampled rows.
    coef_sci_pred = predict_sci_coefficients_default(
        mlp_artifacts,
        coef_near_phys=e10_triplet["coef_near"][sel_pos],
        coef_far_phys=e10_triplet["coef_far"][sel_pos],
        ctx_near_phys=e10_triplet["ctx_near"][sel_pos],
        ctx_far_phys=e10_triplet["ctx_far"][sel_pos],
        ctx_sci_phys=e10_triplet["ctx_sci"][sel_pos],
    ).astype(np.float64)

    coef_near_sel = np.asarray(e10_triplet["coef_near"][sel_pos], dtype=np.float64)
    coef_far_sel = np.asarray(e10_triplet["coef_far"][sel_pos], dtype=np.float64)

    # Reconstruct via the SSFR cell -- keep model + precache + helpers here so
    # the next cell can reuse them.
    base_dir_guess = _infer_base_dir_for_reconstruction()

    # 4a) Optimization (2026-08-11): build ONE reconstruction model outside the loop and
    #     precache per-file LSF-availability + full VECTOR_O2 cubes. Previously the loop
    #     rebuilt SkyDecompLSFSurfaceIterative (basis + solar-reference + moon spline)
    #     3 x n_use times and re-read VECTOR_O2 on every call, both of which dominated
    #     the runtime. See §12 (2026-08-11 batch-RMSE cell reconstruction hoisted).
    import time as _time
    _t_recon0 = _time.perf_counter()
    _wave_ref_recon = (wave_arr if wave_arr.ndim == 1
                       else np.asarray(wave_arr[int(sel_rows[0])], dtype=np.float64))
    if wave_arr.ndim > 1:
        _wave_probe = np.asarray(wave_arr[int(sel_rows[-1])], dtype=np.float64)
        if _wave_probe.shape != _wave_ref_recon.shape or not np.allclose(
                _wave_probe, _wave_ref_recon, rtol=0.0, atol=1e-8):
            raise RuntimeError(
                "wave_arr rows differ between sampled rows; model hoisting assumes a shared grid. "
                "Fall back to per-row reconstruct_with_lsf if this ever triggers on your dataset.")
    _lsf_model = SkyDecompLSFSurfaceIterative(
        _wave_ref_recon, lsf_sigma=1.0, n_spline_knots=25, base_dir=base_dir_guess,
    )
    _mz_model = None

    def _precache_decomp_state(decomp_path):
        state = {"path": Path(decomp_path), "has_lsf": False, "o2_cube": None,
                 "is_moon_zodi": False}
        if not state["path"].exists():
            return state
        try:
            with fits.open(str(state["path"]), memmap=False) as _hdul_dec:
                _ext_names = {h.name for h in _hdul_dec}
                state["has_lsf"] = all(_e in _ext_names
                                       for _e in ("LSF_COEF", "LSF_KNOTS", "LSF_META"))
                if state["has_lsf"]:
                    _coef_rows = int(_hdul_dec["LSF_COEF"].data.shape[0])
                    _meta_rows = int(len(_hdul_dec["LSF_META"].data))
                    _expected_meta = 3 * _coef_rows
                    if _meta_rows != _expected_meta:
                        raise RuntimeError(
                            f"Inconsistent LSF HDUs in {state['path'].name}: "
                            f"LSF_COEF has {_coef_rows} rows but LSF_META has "
                            f"{_meta_rows} rows (expected 3 x {_coef_rows} = "
                            f"{_expected_meta}).")
                if "VECTOR_O2" in _ext_names:
                    _data = np.asarray(_hdul_dec["VECTOR_O2"].data, dtype=np.float64)
                    if _data.ndim == 2:
                        state["o2_cube"] = _data
                state["is_moon_zodi"] = all(
                    _e in _ext_names for _e in MOON_ZODI_HDU_NAMES
                )
        except (KeyError, IndexError, ValueError) as _exc:
            print(f"  precache failed for {state['path'].name}: "
                  f"{type(_exc).__name__}: {_exc}")
        return state

    _state_near = _precache_decomp_state(EVERY10_NEAR)
    _state_far  = _precache_decomp_state(EVERY10_FAR)
    _state_sci  = _precache_decomp_state(EVERY10_SCI)
    print(f"  precache: has_lsf near/far/sci = "
          f"{_state_near['has_lsf']}/{_state_far['has_lsf']}/{_state_sci['has_lsf']}, "
          f"VECTOR_O2 near/far/sci = "
          f"{_state_near['o2_cube'] is not None}/"
          f"{_state_far['o2_cube'] is not None}/"
          f"{_state_sci['o2_cube'] is not None}, "
          f"moon_zodi near/far/sci = "
          f"{_state_near['is_moon_zodi']}/{_state_far['is_moon_zodi']}/{_state_sci['is_moon_zodi']}")
    if any(s["is_moon_zodi"] for s in (_state_near, _state_far, _state_sci)):
        _first_mz_state_dict = next(s for s in (_state_near, _state_far, _state_sci)
                                   if s["is_moon_zodi"])
        _first_mz_state = load_moon_zodi_state(str(_first_mz_state_dict["path"]), 0)
        _mz_n_interior = max(1, len(_first_mz_state.correction_knots) - 8)
        _mz_model = SkyDecompMoonZodiLSFSurfaceIterative(
            wave=_wave_ref_recon,
            lsf_sigma=1.0,
            physical_to_fit_flux_scale=float(FACTOR),
            n_spline_knots=int(_mz_n_interior),
        )
        print(f"  Moon/Zodi model hoisted (n_spline_knots={_mz_n_interior}, "
              f"correction_knots={len(_first_mz_state.correction_knots)} full).")

    def _lsf_state_from_cache(state_dict, row_idx):
        if not state_dict["has_lsf"]:
            return None
        try:
            return load_lsf_surface_state(str(state_dict["path"]), int(row_idx))
        except (KeyError, IndexError, ValueError) as _exc:
            print(f"  LSF surface state unavailable in {state_dict['path'].name} "
                  f"row {int(row_idx)}: {type(_exc).__name__}: {_exc}")
            return None

    def _o2_vec_from_cache(state_dict, row_idx):
        cube = state_dict["o2_cube"]
        if cube is None or int(row_idx) >= cube.shape[0]:
            return None
        row = cube[int(row_idx)]
        if not np.isfinite(row).any() or float(np.nansum(np.abs(row))) == 0.0:
            return None
        return row

    def _mz_state_from_cache(state_dict, row_idx):
        if not state_dict["is_moon_zodi"]:
            return None
        try:
            return load_moon_zodi_state(str(state_dict["path"]), int(row_idx))
        except (KeyError, IndexError, ValueError) as _exc:
            print(f"  Moon/Zodi state unavailable in {state_dict['path'].name} "
                  f"row {int(row_idx)}: {type(_exc).__name__}: {_exc}")
            return None

    def _fast_reconstruct(coef, lsf_state, o2_vec, lsf_sigma_fallback, coef_err=None,
                          mz_state=None, detector_lsf_fwhm=None):
        if mz_state is not None:
            if _mz_model is None:
                raise RuntimeError("mz_state supplied but no Moon/Zodi model hoisted")
            if not isinstance(lsf_state, LSFSurfaceState):
                raise RuntimeError("Moon/Zodi decompositions always ship an LSF surface")
            _mz_model._install_prediction(
                mz_state.observation,
                np.asarray(detector_lsf_fwhm, dtype=np.float64),
            )
            _mz_model._set_lsf_state(lsf_state)
            _mats = _mz_model._assemble_refined_matrices()
            if o2_vec is not None:
                _o2_arr = np.asarray(o2_vec, float).ravel()
                if _o2_arr.shape != _mz_model.wave.shape:
                    raise ValueError(
                        f"o2_vector shape mismatch: expected {_mz_model.wave.shape}, "
                        f"got {_o2_arr.shape}")
                _mats["o2"] = _o2_arr[None, :]
            _coef_arr = np.asarray(coef, float).ravel()
            _comps = _mz_model._components_from_coef(_coef_arr, _mats)
            _comps["total"] = (_comps["oh"] + _comps["moon"] + _comps["zodi"]
                                + _comps["diffuse"] + _comps["atom"]
                                + _comps["orc"] + _comps["o2"])
            if coef_err is not None:
                _err_arr = np.asarray(coef_err, float).ravel()
                _sigmas = _mz_model._components_sigma_from_coef_err(_err_arr, _mats)
                _comps["sigma"] = _sigmas
                _comps["sigma_total"] = np.sqrt(
                    _sigmas["oh"] ** 2 + _sigmas["moon"] ** 2
                    + _sigmas["diffuse"] ** 2 + _sigmas["atom"] ** 2
                    + _sigmas["orc"] ** 2 + _sigmas["o2"] ** 2)
            return _comps
        if isinstance(lsf_state, LSFSurfaceState):
            _lsf_model._set_lsf_state(lsf_state)
            _mats = _lsf_model._assemble_refined_matrices()
            if o2_vec is not None:
                _o2_arr = np.asarray(o2_vec, float).ravel()
                if _o2_arr.shape != _lsf_model.wave.shape:
                    raise ValueError(
                        f"o2_vector shape mismatch: expected {_lsf_model.wave.shape}, "
                        f"got {_o2_arr.shape}")
                _mats["o2"] = _o2_arr[None, :]
            _coef_arr = np.asarray(coef, float).ravel()
            _comps = _lsf_model._components_from_coef(_coef_arr, _mats)
            _comps["total"] = (_comps["oh"] + _comps["moon"] + _comps["diffuse"]
                                + _comps["atom"] + _comps["orc"] + _comps["o2"])
            if coef_err is not None:
                _err_arr = np.asarray(coef_err, float).ravel()
                _sigmas = _lsf_model._components_sigma_from_coef_err(_err_arr, _mats)
                _comps["sigma"] = _sigmas
                _comps["sigma_total"] = np.sqrt(
                    _sigmas["oh"] ** 2 + _sigmas["moon"] ** 2
                    + _sigmas["diffuse"] ** 2 + _sigmas["atom"] ** 2
                    + _sigmas["orc"] ** 2 + _sigmas["o2"] ** 2)
            return _comps
        return reconstruct_with_lsf(
            wave=_lsf_model.wave, coef=coef, lsf=lsf_sigma_fallback,
            n_spline_knots=25, base_dir=base_dir_guess, o2_vector=o2_vec,
            coef_err=coef_err)

    def _reconstruct_pair(coef_pred, coef_true, lsf_state, o2_vec,
                            lsf_sigma_fallback, mz_state=None,
                            detector_lsf_fwhm=None):
        """Reconstruct two coefficient vectors sharing per-row mats assembly.

        `_install_prediction`, `_set_lsf_state`, and `_assemble_refined_matrices`
        dominate the per-row cost; running them once for both pred and true
        halves the SSFR-loop wall time relative to two `_fast_reconstruct`
        calls.
        """
        if mz_state is not None:
            if _mz_model is None:
                raise RuntimeError(
                    "mz_state supplied but no Moon/Zodi model hoisted")
            if not isinstance(lsf_state, LSFSurfaceState):
                raise RuntimeError(
                    "Moon/Zodi decompositions always ship an LSF surface")
            _mz_model._install_prediction(
                mz_state.observation,
                np.asarray(detector_lsf_fwhm, dtype=np.float64),
            )
            _mz_model._set_lsf_state(lsf_state)
            _mats = _mz_model._assemble_refined_matrices()
            if o2_vec is not None:
                _o2_arr = np.asarray(o2_vec, float).ravel()
                if _o2_arr.shape != _mz_model.wave.shape:
                    raise ValueError(
                        f"o2_vector shape mismatch: expected "
                        f"{_mz_model.wave.shape}, got {_o2_arr.shape}")
                _mats["o2"] = _o2_arr[None, :]
            _out = []
            for _coef in (coef_pred, coef_true):
                _c = np.asarray(_coef, float).ravel()
                _comps = _mz_model._components_from_coef(_c, _mats)
                _comps["total"] = (_comps["oh"] + _comps["moon"]
                                    + _comps["zodi"] + _comps["diffuse"]
                                    + _comps["atom"] + _comps["orc"]
                                    + _comps["o2"])
                _out.append(_comps)
            return _out[0], _out[1]
        if isinstance(lsf_state, LSFSurfaceState):
            _lsf_model._set_lsf_state(lsf_state)
            _mats = _lsf_model._assemble_refined_matrices()
            if o2_vec is not None:
                _o2_arr = np.asarray(o2_vec, float).ravel()
                if _o2_arr.shape != _lsf_model.wave.shape:
                    raise ValueError(
                        f"o2_vector shape mismatch: expected "
                        f"{_lsf_model.wave.shape}, got {_o2_arr.shape}")
                _mats["o2"] = _o2_arr[None, :]
            _out = []
            for _coef in (coef_pred, coef_true):
                _c = np.asarray(_coef, float).ravel()
                _comps = _lsf_model._components_from_coef(_c, _mats)
                _comps["total"] = (_comps["oh"] + _comps["moon"]
                                    + _comps["diffuse"] + _comps["atom"]
                                    + _comps["orc"] + _comps["o2"])
                _out.append(_comps)
            return _out[0], _out[1]
        return (
            _fast_reconstruct(coef_pred, lsf_state, o2_vec,
                                lsf_sigma_fallback),
            _fast_reconstruct(coef_true, lsf_state, o2_vec,
                                lsf_sigma_fallback),
        )

    print(f"  recon setup: {_time.perf_counter() - _t_recon0:.2f} s "
          f"(one-time basis build + FITS precache)")
    print(f"  SSFR-ready: {n_use} rows predicted, reconstruction helpers hoisted "
          f"(_fast_reconstruct + _lsf/o2/mz_state_from_cache + _state_sci).")


In [ ]:
# ============================================================================
# Sky-Subtraction Fractional Residual (SSFR) -- 2026-08-18
#
# Component-wise, band-resolved, flux-based prediction quality metric.
# Separates NETWORK error from DECOMPOSITION floor, and further attributes
# the network error to (moon+zodi, diffuse, lines) components.
#
#   SSFR_floor(b)      = RMS(f_true_recon_total - f_obs_sci)/median(|f_true_total|)  in band b
#   SSFR_network(b)    = RMS(f_pred_total       - f_true_recon_total)/median(|f_true_total|)
#   SSFR_deployment(b) = RMS(f_pred_total       - f_obs_sci)/median(|f_obs_sci|)
#   SSFR_c(b)          = RMS(f_pred_c - f_true_recon_c)/median(|f_true_total|)   per component group c
#
# BIAS_c(b) = mean(f_pred_c - f_true_recon_c)/median(|f_true_total|)  (signed)
#
# Requires: cell 27 (batch stats) has been run so the following globals exist:
#   coef_sci_pred, sel_pos, sel_rows, wave_arr, lsf_sci_arr, flux_sci_all,
#   _fast_reconstruct, _lsf_state_from_cache, _o2_vec_from_cache,
#   _mz_state_from_cache, _state_sci, e10_triplet, FACTOR
# ============================================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go

RUN_SSFR = True  # gate

if RUN_SSFR:
    _required_ssfr = [
        "coef_sci_pred", "sel_pos", "sel_rows", "wave_arr", "lsf_sci_arr",
        "flux_sci_all", "_fast_reconstruct", "_lsf_state_from_cache",
        "_o2_vec_from_cache", "_mz_state_from_cache", "_state_sci",
        "e10_triplet", "FACTOR",
    ]
    _miss = [k for k in _required_ssfr if k not in globals()]
    if _miss:
        raise RuntimeError(f"Missing globals for SSFR: {_miss}. Run cell 27 (batch stats) first.")

    SSFR_BANDS = [
        ("B_cont",   4000.0, 4500.0),
        ("B_OH",     5000.0, 5400.0),
        ("G_cont",   5450.0, 5700.0),
        ("G_OH",     5800.0, 6300.0),
        ("R_cont",   7000.0, 7300.0),
        ("R_OH",     7300.0, 9000.0),
        ("NIR_cont", 9600.0, 9900.0),
    ]

    # Component grouping matches cell 26's diagnostic (moon+zodi joined).
    COMPONENT_GROUPS = {
        "moon+zodi": ("moon", "zodi"),
        "diffuse":   ("diffuse",),
        "lines":     ("oh", "atom", "orc", "o2"),
    }

    _sel_pos_arr = np.asarray(sel_pos, dtype=int)
    _sel_rows_arr = np.asarray(sel_rows, dtype=int)
    _n_batch = int(_sel_rows_arr.size)

    # True sci coefs for the batch rows -- MUST come from e10_triplet
    # (positions match sel_pos and coef_sci_pred); filtered_triplet is a
    # different FITS product and its ordering does not align with sel_pos.
    _coef_sci_true_batch = np.asarray(e10_triplet["coef_sci"], dtype=np.float64)[_sel_pos_arr]

    _rows = []
    _resid_matrix = None
    _resid_comp_matrix = None
    _wave_ref_resid = None
    import time as _time
    _t0 = _time.perf_counter()
    for i in range(_n_batch):
        rr = int(_sel_rows_arr[i])
        wave_row = wave_arr if wave_arr.ndim == 1 else np.asarray(wave_arr[rr], dtype=np.float64)
        lsf_row = lsf_sci_arr if lsf_sci_arr.ndim == 1 else np.asarray(lsf_sci_arr[rr], dtype=np.float64)

        _lsf_state = _lsf_state_from_cache(_state_sci, rr)
        _o2_vec = _o2_vec_from_cache(_state_sci, rr)
        _mz_state = _mz_state_from_cache(_state_sci, rr)
        _lsf_fallback = lsf_row / 2.35

        # Reconstruct both pred and true (same sci LSF / MZ / O2 templates)
        try:
            comps_pred, comps_true = _reconstruct_pair(
                coef_sci_pred[i], _coef_sci_true_batch[i],
                _lsf_state, _o2_vec, _lsf_fallback,
                mz_state=_mz_state, detector_lsf_fwhm=lsf_row)
        except Exception as e:
            print(f"  row {rr}: reconstruction failed ({e}); skipping.")
            continue

        flux_obs = np.asarray(flux_sci_all[rr], dtype=np.float64)
        flux_pred_total = np.asarray(comps_pred["total"], dtype=np.float64) / FACTOR
        flux_true_total = np.asarray(comps_true["total"], dtype=np.float64) / FACTOR

        # Aggregate component fluxes per group
        pred_c = {g: np.zeros_like(flux_pred_total) for g in COMPONENT_GROUPS}
        true_c = {g: np.zeros_like(flux_true_total) for g in COMPONENT_GROUPS}
        for g, comp_keys in COMPONENT_GROUPS.items():
            for k in comp_keys:
                if k in comps_pred:
                    pred_c[g] += np.asarray(comps_pred[k], dtype=np.float64) / FACTOR
                if k in comps_true:
                    true_c[g] += np.asarray(comps_true[k], dtype=np.float64) / FACTOR

        # Per-row residuals for the ensemble plot: total network error
        # (pred - true_recon) plus one panel per component so continuum-
        # shape deficits can be attributed to moon+zodi / diffuse / lines.
        if _resid_matrix is None:
            _wave_ref_resid = np.asarray(wave_row, dtype=np.float64).copy()
            _resid_matrix = np.full((_n_batch, _wave_ref_resid.size),
                                     np.nan, dtype=np.float64)
            _resid_comp_matrix = {_g: np.full((_n_batch, _wave_ref_resid.size),
                                                np.nan, dtype=np.float64)
                                    for _g in COMPONENT_GROUPS}
            _frac_resid_matrix = np.full((_n_batch, _wave_ref_resid.size),
                                          np.nan, dtype=np.float64)
            _frac_resid_comp_matrix = {_g: np.full((_n_batch, _wave_ref_resid.size),
                                                    np.nan, dtype=np.float64)
                                        for _g in COMPONENT_GROUPS}
        if wave_row.size == _resid_matrix.shape[1]:
            _resid_matrix[i] = flux_pred_total - flux_true_total
            for _g in COMPONENT_GROUPS:
                _resid_comp_matrix[_g][i] = pred_c[_g] - true_c[_g]
            # Fractional residuals normalise by observed sci flux, masking pixels
            # where |obs| is below 1% of the row's median |obs| so ratios don't
            # blow up in deep absorption / between-line dips.
            _obs_row_abs_median = float(np.nanmedian(np.abs(flux_obs)))
            _obs_floor = max(1e-30, 0.01 * _obs_row_abs_median)
            _obs_safe = np.where(np.abs(flux_obs) > _obs_floor, flux_obs, np.nan)
            _frac_resid_matrix[i] = (flux_pred_total - flux_obs) / _obs_safe
            for _g in COMPONENT_GROUPS:
                _frac_resid_comp_matrix[_g][i] = (pred_c[_g] - true_c[_g]) / _obs_safe

        for band_name, wl_lo, wl_hi in SSFR_BANDS:
            mask = (wave_row >= wl_lo) & (wave_row < wl_hi)
            if int(mask.sum()) < 5:
                continue

            f_obs_b = flux_obs[mask]
            f_pred_total_b = flux_pred_total[mask]
            f_true_total_b = flux_true_total[mask]

            # Reference amplitude: RMS of observed sci flux in the band.
            # Same denominator for all three flavors so ratios are directly comparable.
            # Physical interpretation: RMS residual as fraction of RMS sky brightness.
            _ref = float(np.sqrt(np.mean(f_obs_b ** 2)))
            if _ref < 1e-30:
                continue

            ssfr_floor   = float(np.sqrt(np.mean((f_true_total_b - f_obs_b) ** 2))) / _ref
            ssfr_network = float(np.sqrt(np.mean((f_pred_total_b - f_true_total_b) ** 2))) / _ref
            ssfr_deploy  = float(np.sqrt(np.mean((f_pred_total_b - f_obs_b) ** 2))) / _ref

            row = {
                "row_idx": rr, "batch_i": i, "band": band_name,
                "wl_center": 0.5 * (wl_lo + wl_hi),
                "n_pix": int(mask.sum()),
                "ref_flux": _ref,
                "SSFR_floor": ssfr_floor,
                "SSFR_network": ssfr_network,
                "SSFR_deployment": ssfr_deploy,
            }
            for g in COMPONENT_GROUPS:
                _dc = pred_c[g][mask] - true_c[g][mask]
                row[f"SSFR_{g}"] = float(np.sqrt(np.mean(_dc ** 2))) / _ref
                row[f"BIAS_{g}"] = float(np.mean(_dc)) / _ref
            _rows.append(row)

    ssfr_df = pd.DataFrame(_rows)
    _t1 = _time.perf_counter()
    print(f"SSFR computation: {_t1 - _t0:.1f} s over {_n_batch} rows x {len(SSFR_BANDS)} bands = {len(ssfr_df)} row-band tuples.")

    # -----------------------------------------------------------------
    # 1) Per-band aggregated SSFR: median across rows (%)
    # -----------------------------------------------------------------
    _band_order = [b[0] for b in SSFR_BANDS]
    _agg_cols = ["SSFR_floor", "SSFR_network", "SSFR_deployment",
                 "SSFR_moon+zodi", "SSFR_diffuse", "SSFR_lines"]
    _summary_med = ssfr_df.groupby("band")[_agg_cols].median().reindex(_band_order)
    _summary_p95 = ssfr_df.groupby("band")[_agg_cols].quantile(0.95).reindex(_band_order)

    def _fmt_pct(df):
        return (df * 100.0).round(2)

    print()
    print("SSFR (median across rows) per band [%]:")
    print(_fmt_pct(_summary_med).to_string())
    print()
    print("SSFR (95th percentile across rows) per band [%]:")
    print(_fmt_pct(_summary_p95).to_string())

    # -----------------------------------------------------------------
    # 2) Signed bias per component per band (median across rows) [%]
    # -----------------------------------------------------------------
    _bias_cols = ["BIAS_moon+zodi", "BIAS_diffuse", "BIAS_lines"]
    _bias_med = ssfr_df.groupby("band")[_bias_cols].median().reindex(_band_order)
    print()
    print("Signed bias per component per band [%] (median; + = over-pred, - = under-pred):")
    print(_fmt_pct(_bias_med).to_string())

    # -----------------------------------------------------------------
    # 3) Headline numbers: single-value summaries
    # -----------------------------------------------------------------
    print()
    print("Headline SSFR numbers [%]:")
    print(f"  SSFR_floor      median across all row-band tuples : {100*ssfr_df['SSFR_floor'].median():.2f}")
    print(f"  SSFR_network    median across all row-band tuples : {100*ssfr_df['SSFR_network'].median():.2f}")
    print(f"  SSFR_deployment median across all row-band tuples : {100*ssfr_df['SSFR_deployment'].median():.2f}")
    print(f"  SSFR_deployment p95 (harder rows)               : {100*ssfr_df['SSFR_deployment'].quantile(0.95):.2f}")

    # Worst-band-per-row aggregation: for each row, take the worst band's SSFR_deployment
    _worst_per_row = ssfr_df.groupby("row_idx")["SSFR_deployment"].max()
    print(f"  worst-band-per-row deployment  median            : {100*_worst_per_row.median():.2f}")
    print(f"  worst-band-per-row deployment  p95               : {100*_worst_per_row.quantile(0.95):.2f}")

    # -----------------------------------------------------------------
    # 4) Diagnostic: which rows are FLOOR-limited vs NETWORK-limited?
    # -----------------------------------------------------------------
    _limit_ratio = ssfr_df["SSFR_network"] / (ssfr_df["SSFR_floor"] + 1e-30)
    ssfr_df["_limit_ratio"] = _limit_ratio
    ssfr_df["_limit"] = np.where(_limit_ratio > 2.0, "network",
                                  np.where(_limit_ratio < 0.5, "floor", "mixed"))
    print()
    print("Row-band-tuples classified by limiter (SSFR_network / SSFR_floor):")
    print(ssfr_df["_limit"].value_counts().to_string())

    # -----------------------------------------------------------------
    # 5) Top-10 worst rows (by max SSFR_deployment across bands)
    # -----------------------------------------------------------------
    _worst_row_ids = _worst_per_row.sort_values(ascending=False).head(10)
    print()
    print("Top-10 rows by worst-band deployment SSFR [%]:")
    for _row_id, _ssfr_val in _worst_row_ids.items():
        _sub = ssfr_df[ssfr_df["row_idx"] == _row_id].sort_values("SSFR_deployment", ascending=False)
        _worst_band = _sub.iloc[0]
        _dom_c = max(("moon+zodi", "diffuse", "lines"),
                     key=lambda c: _worst_band[f"SSFR_{c}"])
        print(f"  row {int(_row_id):5d}  band={_worst_band['band']:9s}  "
              f"SSFR_deploy={100*_ssfr_val:5.2f}%  floor={100*_worst_band['SSFR_floor']:5.2f}%  "
              f"network={100*_worst_band['SSFR_network']:5.2f}%  "
              f"dominant_comp={_dom_c}  ({_worst_band['_limit']})")

    # -----------------------------------------------------------------
    # 6) Plot: per-band violin of SSFR components
    # -----------------------------------------------------------------
    fig_ssfr = go.Figure()
    _colors = {"floor": "#7f7f7f", "network": "#1f78b4",
               "moon+zodi": "#e41a1c", "diffuse": "#377eb8", "lines": "#4daf4a"}
    for _var_name, _col_key in [("SSFR_floor", "floor"),
                                  ("SSFR_network", "network"),
                                  ("SSFR_moon+zodi", "moon+zodi"),
                                  ("SSFR_diffuse", "diffuse"),
                                  ("SSFR_lines", "lines")]:
        fig_ssfr.add_trace(
            go.Box(
                y=100.0 * ssfr_df[_var_name],
                x=ssfr_df["band"],
                name=_var_name.replace("SSFR_", ""),
                marker_color=_colors[_col_key],
                boxmean=True,
            )
        )
    fig_ssfr.update_layout(
        template="plotly_white",
        title="SSFR (%) per band -- network error attribution",
        yaxis_title="SSFR [%]",
        xaxis_title="wavelength band",
        boxmode="group",
        height=520,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
    )
    fig_ssfr.update_yaxes(range=[0, min(20.0, 1.1 * 100.0 * ssfr_df["SSFR_deployment"].quantile(0.99))])
    fig_ssfr.show()

    # -----------------------------------------------------------------
    # 7) Total SSFR statistics (aggregated over all row-band tuples).
    # -----------------------------------------------------------------
    _stat_cols = ["SSFR_floor", "SSFR_network", "SSFR_deployment",
                  "SSFR_moon+zodi", "SSFR_diffuse", "SSFR_lines"]
    _stats_tot = (100.0 * ssfr_df[_stat_cols]).describe(
        percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    _stats_tot = _stats_tot[["mean", "std", "min", "5%", "25%", "50%",
                              "75%", "95%", "99%", "max"]]
    print()
    print("Total SSFR statistics (aggregated across all row-band tuples), [%]:")
    print(_stats_tot.round(2).to_string())

    # -----------------------------------------------------------------
    # 8) Per-row residual panels: total (pred - true_recon = network)
    #    at top plus one panel per component group so continuum-shape
    #    deficits can be attributed to moon+zodi / diffuse / lines.
    # -----------------------------------------------------------------
    if _resid_matrix is not None:
        _resid_disp = _resid_matrix * FACTOR
        _finite_rows = np.isfinite(_resid_disp).all(axis=1)
        _resid_finite = _resid_disp[_finite_rows]
        _row_ids_finite = np.asarray(_sel_rows_arr)[_finite_rows]
        _resid_comp_finite = {_g: _resid_comp_matrix[_g][_finite_rows] * FACTOR
                              for _g in COMPONENT_GROUPS}
        # Fractional residuals stay dimensionless; NaN pixels (masked obs) render as gaps.
        _frac_resid_finite = _frac_resid_matrix[_finite_rows]
        _frac_resid_comp_finite = {_g: _frac_resid_comp_matrix[_g][_finite_rows]
                                    for _g in COMPONENT_GROUPS}
        # Color by lunation (moon phase 0-360 deg, cyclic HSV) so we can see
        # whether tail-row residuals correlate with bright-moon conditions.
        _moon_phase_e10 = _moon_phase_deg_from_ctx(e10_triplet)
        _row_phase_all = _moon_phase_e10[_sel_pos_arr]
        _row_phase = _row_phase_all[_finite_rows]
        # Worst-band deployment SSFR kept for hover text only.
        _worst_dict = _worst_per_row.to_dict()
        _row_worst_pct = np.array([
            100.0 * float(_worst_dict.get(int(_rid), np.nan))
            for _rid in _row_ids_finite])
        # expnum per row; None if META['EXPNUM'] wasn't attached in cell 20.
        _expnum_all_glob = globals().get("_expnum_all", None)
        _expnum_finite = (np.asarray(_expnum_all_glob)[_row_ids_finite]
                          if _expnum_all_glob is not None else None)
        _norm = np.clip((_row_phase % 360.0) / 360.0, 0.0, 1.0)
        from plotly.colors import sample_colorscale
        _colors = sample_colorscale("HSV", _norm.tolist())
        # Order so bright-moon rows draw on top, ties broken by worst-band SSFR.
        _bright_key = np.abs(((_row_phase + 180.0) % 360.0) - 180.0)
        _order = np.lexsort((
            np.where(np.isfinite(_row_worst_pct), _row_worst_pct, -np.inf),
            -_bright_key,
        ))
        # Phase-stratified subsample so the overlay stays legible with 1000+ rows.
        _n_plot_max = 100
        if _order.size > _n_plot_max:
            _plot_rng = np.random.default_rng(42)
            _phase_of_order = _row_phase[_order]
            _plot_bins = 10
            _plot_edges = np.quantile(_phase_of_order,
                                       np.linspace(0.0, 1.0, _plot_bins + 1))
            _plot_edges[0], _plot_edges[-1] = -np.inf, np.inf
            _plot_bin_id = np.digitize(
                _phase_of_order, _plot_edges[1:-1], right=False)
            _plot_per_bin = _n_plot_max // _plot_bins
            _plot_pick = []
            for _b in range(_plot_bins):
                _in_bin = np.flatnonzero(_plot_bin_id == _b)
                _take = int(min(_plot_per_bin, _in_bin.size))
                if _take > 0:
                    _plot_pick.append(_plot_rng.choice(
                        _in_bin, size=_take, replace=False))
            if _plot_pick:
                _order = _order[np.sort(np.concatenate(_plot_pick))]
            else:
                _order = _order[:_n_plot_max]
        # (subplot title, per-row matrix, y-axis title).
        _panel_defs = [
            ("total (pred - true_recon)", _resid_finite,
             f"Residual (x{FACTOR:.0e})"),
            ("moon+zodi (pred - true_recon)", _resid_comp_finite["moon+zodi"],
             f"Residual (x{FACTOR:.0e})"),
            ("diffuse (pred - true_recon)", _resid_comp_finite["diffuse"],
             f"Residual (x{FACTOR:.0e})"),
            ("lines (pred - true_recon)", _resid_comp_finite["lines"],
             f"Residual (x{FACTOR:.0e})"),
            ("total (pred - obs) / obs", _frac_resid_finite,
             "Fraction of obs_sci"),
            ("moon+zodi (pred - true_recon) / obs",
             _frac_resid_comp_finite["moon+zodi"], "Fraction of obs_sci"),
            ("diffuse (pred - true_recon) / obs",
             _frac_resid_comp_finite["diffuse"], "Fraction of obs_sci"),
            ("lines (pred - true_recon) / obs",
             _frac_resid_comp_finite["lines"], "Fraction of obs_sci"),
        ]
        _n_panels_resid = len(_panel_defs)
        fig_resid = make_subplots(
            rows=_n_panels_resid, cols=1, shared_xaxes=True,
            vertical_spacing=0.025,
            subplot_titles=[_lbl for _lbl, _, _ in _panel_defs],
        )
        for _panel_i, (_label, _mat, _ytitle) in enumerate(_panel_defs, start=1):
            for _i_ord in _order:
                _rid = int(_row_ids_finite[_i_ord])
                _wpct = _row_worst_pct[_i_ord]
                _phase = _row_phase[_i_ord]
                _base = _colors[_i_ord]
                _rgba = _base.replace("rgb(", "rgba(").replace(")", ",0.55)")
                _wpct_str = (f"{_wpct:.1f}%"
                             if np.isfinite(_wpct) else "n/a")
                _exp = (_expnum_finite[_i_ord]
                        if _expnum_finite is not None else None)
                _exp_str = (f" | expnum {int(_exp)}"
                            if _exp is not None else "")
                fig_resid.add_trace(
                    go.Scattergl(
                        x=_wave_ref_resid, y=_mat[_i_ord],
                        mode="lines",
                        line=dict(color=_rgba, width=0.8),
                        name=(f"row {_rid}{_exp_str} | moon_phase {_phase:.0f} deg | "
                              f"worst-band deploy {_wpct_str}"),
                        hovertemplate=("%{fullData.name}<br>"
                                       f"panel={_label}<br>"
                                       "lambda=%{x:.0f} A<br>"
                                       "residual=%{y:.3g}<extra></extra>"),
                        showlegend=False,
                    ),
                    row=_panel_i, col=1,
                )
            fig_resid.add_hline(y=0, line=dict(color="black", width=0.7,
                                                 dash="dash"),
                                row=_panel_i, col=1)
            fig_resid.update_yaxes(title_text=_ytitle,
                                     row=_panel_i, col=1)
        # Cap fractional panels at +/-1 so a few extreme pixels near band edges
        # don't compress the main signal to a flat line.
        for _panel_i in range(5, _n_panels_resid + 1):
            fig_resid.update_yaxes(range=[-1.0, 1.0], row=_panel_i, col=1)
        fig_resid.update_xaxes(title_text="Wavelength [A]",
                                row=_n_panels_resid, col=1)
        fig_resid.update_layout(
            template="plotly_white",
            title=(f"Per-component residuals for "
                   f"{int(_order.size)} rows (subsampled from "
                   f"{int(_finite_rows.sum())}) -- top 4: absolute "
                   f"(pred - true_recon); bottom 4: fractional vs observed "
                   f"sci flux -- color: moon phase (0-360 deg, cyclic HSV)"),
            height=2100,
        )
        fig_resid.show()


In [ ]:
# Per-row signed component bias vs context variables (grid).
# One dot per row_idx, labelled with the row number so systematic
# outliers can be cross-referenced with the SSFR trace plot.  A
# systematic slope in any panel = unmodelled context dependence.
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_required_ctxres = [
    "ssfr_df", "_sel_pos_arr", "_sel_rows_arr",
    "e10_triplet", "_decode_cyclic_context",
]
_miss = [k for k in _required_ctxres if k not in globals()]
if _miss:
    raise RuntimeError(f"Missing globals: {_miss}. Run the SSFR cell first.")

_row_bias_df = (ssfr_df.groupby("row_idx")
                [["BIAS_moon+zodi", "BIAS_diffuse", "BIAS_lines"]].mean())
_bias_row_ids = _row_bias_df.index.to_numpy()
# expnum lookup for hover tooltip; falls back gracefully if unavailable.
_expnum_all_glob = globals().get("_expnum_all", None)
if _expnum_all_glob is not None:
    _expnum_bias = np.asarray(_expnum_all_glob)[_bias_row_ids]
    _bias_customdata = np.column_stack([_bias_row_ids, _expnum_bias])
    _bias_hover_row = "row_idx=%{customdata[0]}<br>expnum=%{customdata[1]}<br>"
else:
    _bias_customdata = _bias_row_ids
    _bias_hover_row = "row_idx=%{customdata}<br>"
_row_idx_to_sel_i = {int(r): i for i, r in enumerate(_sel_rows_arr)}
_bias_sel_i = np.array(
    [_row_idx_to_sel_i[int(r)] for r in _bias_row_ids], dtype=int)
_bias_sel_pos = _sel_pos_arr[_bias_sel_i]

_ctx_sci_bias = np.asarray(
    e10_triplet["ctx_sci"], dtype=np.float64)[_bias_sel_pos]
_ctx_names_e10 = list(e10_triplet["ctx_names"])
_ctx_disp_names, _ctx_disp_mat = _decode_cyclic_context(
    _ctx_names_e10, _ctx_sci_bias)
_ctx_disp_lookup = {n: _ctx_disp_mat[:, i]
                    for i, n in enumerate(_ctx_disp_names)}

_context_vars_to_plot = [
    "moon_alt", "moon_phase", "moon_sep",
    "alt", "airmass", "sun_alt", "ecl_beta_deg",
]
_context_present = [c for c in _context_vars_to_plot
                     if c in _ctx_disp_lookup]
_component_rows_plot = [
    ("moon+zodi", "#e41a1c"),
    ("diffuse", "#377eb8"),
    ("lines", "#4daf4a"),
]
_n_rows_grid = len(_component_rows_plot)
_n_cols_grid = len(_context_present)

_subplot_titles_grid = []
for _r_i in range(_n_rows_grid):
    for _c_n in _context_present:
        _subplot_titles_grid.append(_c_n if _r_i == 0 else "")

fig_ctxres = make_subplots(
    rows=_n_rows_grid, cols=_n_cols_grid,
    subplot_titles=_subplot_titles_grid,
    horizontal_spacing=0.028,
    vertical_spacing=0.055,
    shared_yaxes="rows",
)
for _r_i, (_comp_name, _comp_color) in enumerate(
        _component_rows_plot, start=1):
    _y_bias = 100.0 * _row_bias_df[f"BIAS_{_comp_name}"].to_numpy()
    for _c_i, _ctx_name in enumerate(_context_present, start=1):
        _x_ctx = _ctx_disp_lookup[_ctx_name]
        fig_ctxres.add_trace(
            go.Scattergl(
                x=_x_ctx, y=_y_bias,
                mode="markers",
                marker=dict(size=4, color=_comp_color, opacity=0.55),
                customdata=_bias_customdata,
                showlegend=False,
                hovertemplate=(
                    _bias_hover_row
                    + f"{_ctx_name}: %{{x}}<br>"
                    + f"BIAS_{_comp_name}: %{{y:.2f}}%"
                    + "<extra></extra>"),
            ),
            row=_r_i, col=_c_i,
        )
        fig_ctxres.add_hline(
            y=0, line=dict(color="black", width=0.6, dash="dash"),
            row=_r_i, col=_c_i)
    fig_ctxres.update_yaxes(
        title_text=f"BIAS_{_comp_name} [%]",
        row=_r_i, col=1, title_font=dict(size=10))
for _c_i, _ctx_name in enumerate(_context_present, start=1):
    fig_ctxres.update_xaxes(
        title_text=_ctx_name, row=_n_rows_grid, col=_c_i,
        title_font=dict(size=10))
fig_ctxres.for_each_annotation(lambda a: a.update(font=dict(size=10)))
fig_ctxres.update_xaxes(showline=True, mirror=True, ticks="outside",
                          ticklen=3, tickfont=dict(size=8))
fig_ctxres.update_yaxes(showline=True, mirror=True, ticks="outside",
                          ticklen=3, tickfont=dict(size=8))
fig_ctxres.update_layout(
    template="plotly_white",
    title=(f"Per-row signed component bias (mean across 7 bands) vs "
           f"context (n_rows={len(_row_bias_df)}). "
           "Red = moon+zodi, blue = diffuse, green = lines. "
           "Hover a dot for row_idx and per-component bias."),
    height=_n_rows_grid * 240 + 120,
    width=min(2200, _n_cols_grid * 240 + 160),
    margin=dict(l=80, r=40, t=110, b=80),
)
fig_ctxres.show()


## Full-sky quality map

In [ ]:
# WRMSE vs galactic coordinates for all observations.
# Uses build_triplet_coef_dataset (no region filter) so LMC/SMC rows are also
# predicted and shown; membership in train / val-test / originally-excluded
# controls only the marker opacity and the color-scale reference range.
import plotly.graph_objects as go
from plotly.colors import sample_colorscale
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table

required = ['filtered_triplet', 'train_idx', 'val_idx', 'test_idx',
            'mlp_artifacts', 'predict_sci_coefficients_default',
            'build_triplet_coef_dataset', '_DECOMP_SUFFIX']
_missing = [k for k in required if k not in globals()]
if _missing:
    raise RuntimeError(
        'Missing kernel state: ' + ', '.join(_missing)
        + '. Run all prior cells (data loading + model training + split '
        'assignment) before executing this cell.'
    )

INPUT_FITS = 'lvmsframe_median_stack_1.2.1_p40_p70.fits'
NEAR_FITS = f'lvmsframe_median_stack_1.2.1_p40_p70_decomp_sky1{_DECOMP_SUFFIX}.fits'
FAR_FITS = f'lvmsframe_median_stack_1.2.1_p40_p70_decomp_sky2{_DECOMP_SUFFIX}.fits'
SCI_FITS = f'lvmsframe_median_stack_1.2.1_p40_p70_decomp_sci{_DECOMP_SUFFIX}.fits'
META_ONLY_FITS = 'lvmsframe_median_stack_1.2.1_p40_p70_meta_only.fits'

with fits.open(META_ONLY_FITS) as hdul:
    meta_tbl_all = Table(hdul['META'].data)

n_all_total = len(meta_tbl_all)
ra_col = next((c for c in ['sci_ra', 'ra', 'RA'] if c in meta_tbl_all.colnames), None)
dec_col = next((c for c in ['sci_dec', 'dec', 'DEC'] if c in meta_tbl_all.colnames), None)
if ra_col is None or dec_col is None:
    raise ValueError('RA/DEC not found in metadata')

ra_all = np.asarray(meta_tbl_all[ra_col], dtype=np.float64)
dec_all = np.asarray(meta_tbl_all[dec_col], dtype=np.float64)
coords_icrs = SkyCoord(ra=ra_all * u.deg, dec=dec_all * u.deg, frame='icrs')

lmc_cfg = globals().get('LMC_EXCLUSION', {'ra_deg': 80.894, 'dec_deg': -69.756, 'radius_deg': 10.0})
smc_cfg = globals().get('SMC_EXCLUSION', {'ra_deg': 13.187, 'dec_deg': -72.829, 'radius_deg': 10.0})

is_region_excluded = np.zeros(n_all_total, dtype=bool)
for cfg in (lmc_cfg, smc_cfg):
    center = SkyCoord(ra=float(cfg['ra_deg']) * u.deg,
                      dec=float(cfg['dec_deg']) * u.deg, frame='icrs')
    is_region_excluded |= (coords_icrs.separation(center).deg <= float(cfg['radius_deg']))

filtered_row_indices = np.asarray(filtered_triplet['row_index'], dtype=int)
is_train = np.zeros(n_all_total, dtype=bool)
is_valtest = np.zeros(n_all_total, dtype=bool)
train_idx_arr = np.asarray(train_idx, dtype=int)
valtest_idx_arr = np.unique(np.concatenate([np.asarray(val_idx, dtype=int),
                                            np.asarray(test_idx, dtype=int)]))
is_train[filtered_row_indices[train_idx_arr]] = True
is_valtest[filtered_row_indices[valtest_idx_arr]] = True
is_other = ~(is_train | is_valtest | is_region_excluded)

print(f'Row counts: total={n_all_total} '
      f'train={is_train.sum()} valtest={is_valtest.sum()} '
      f'lmcsmc={is_region_excluded.sum()} other={is_other.sum()}')

# Full aligned triplet without region filter -> includes LMC/SMC rows.
context_columns = list(globals().get('ctx_names_all', filtered_triplet['ctx_names']))
triplet_full = build_triplet_coef_dataset(
    input_fits_path=INPUT_FITS,
    sky_near_decomp_fits_path=NEAR_FITS,
    sky_far_decomp_fits_path=FAR_FITS,
    sci_decomp_fits_path=SCI_FITS,
    context_columns=context_columns,
    return_chi2=False,
)

full_rows = np.asarray(triplet_full['row_index'], dtype=int)
coef_true_full = np.asarray(triplet_full['coef_sci'], dtype=np.float64)
coef_pred_full = predict_sci_coefficients_default(
    mlp_artifacts,
    coef_near_phys=triplet_full['coef_near'],
    coef_far_phys=triplet_full['coef_far'],
    ctx_near_phys=triplet_full['ctx_near'],
    ctx_far_phys=triplet_full['ctx_far'],
    ctx_sci_phys=triplet_full['ctx_sci'],
).astype(np.float64)

# Per-row WRMSE for the WRMSE map + downstream correlation/scatter cells.
# Uses coef_err_sci from the full triplet if COEF_ERR was loaded; else
# falls back to NaN sigmas (which the helper treats as floor-only weights).
_sigma_full = triplet_full.get('coef_err_sci', None)
if _sigma_full is None:
    _sigma_full = np.full_like(coef_true_full, np.nan)
_sigma_full = np.asarray(_sigma_full, dtype=np.float64)
_gidx_map = (_group_indices_compress if '_group_indices_compress' in globals()
             else group_indices_sf)
wrmse_full = weighted_rmse_per_row(
    coef_true_full, coef_pred_full, _sigma_full,
    _gidx_map, dict(DEFAULT_COEF_ERR_SIGMA_FLOOR_BY_GROUP),
).astype(np.float32)

wrmse_all = np.full(n_all_total, np.nan, dtype=np.float32)
valid_rows = (full_rows >= 0) & (full_rows < n_all_total)
wrmse_all[full_rows[valid_rows]] = wrmse_full[valid_rows]

print(f'Finite sWRMSE_coef: total={np.isfinite(wrmse_all).sum()} '
      f'train={np.isfinite(wrmse_all[is_train]).sum()} '
      f'valtest={np.isfinite(wrmse_all[is_valtest]).sum()} '
      f'lmcsmc={np.isfinite(wrmse_all[is_region_excluded]).sum()} '
      f'other={np.isfinite(wrmse_all[is_other]).sum()}')

gl_all = coords_icrs.galactic.l.deg
gb_all = coords_icrs.galactic.b.deg
field_diameter_arcmin = 30.0
field_radius_deg = (field_diameter_arcmin / 2.0) / 60.0

def make_circle(center_l, center_b, radius_deg, n_points=32):
    a = np.linspace(0.0, 2.0 * np.pi, n_points)
    return center_l + radius_deg * np.cos(a), center_b + radius_deg * np.sin(a)

# Color scale fixed to train+val/test in-domain range so LMC/SMC out-of-domain
# behaviour cannot compress the visible dynamic range for the training set.
wrmse_in_domain = wrmse_all[(is_train | is_valtest) & np.isfinite(wrmse_all)]
if len(wrmse_in_domain) == 0:
    raise RuntimeError('No finite sWRMSE_coef available for in-domain (train+val/test) rows.')
wrmse_vmin, wrmse_vmax = np.percentile(wrmse_in_domain, [1, 99])
_wrmse_denom = max(wrmse_vmax - wrmse_vmin, 1e-30)
# Lower WRMSE -> lighter red (good); higher -> darker red (bad).
wrmse_norm = np.clip((wrmse_all - wrmse_vmin) / _wrmse_denom, 0, 1)

colors_list = ['rgb(235,235,235)'] * n_all_total
finite_norm = np.isfinite(wrmse_norm)
if np.any(finite_norm):
    sampled = sample_colorscale('Reds', wrmse_norm[finite_norm].tolist())
    for idx_i, color in zip(np.where(finite_norm)[0], sampled):
        colors_list[idx_i] = color

fig = go.Figure()
for i in range(n_all_total):
    cl, cb = make_circle(gl_all[i], gb_all[i], field_radius_deg)
    if is_train[i]:
        opacity, region = 0.88, 'Train'
    elif is_valtest[i]:
        opacity, region = 0.45, 'Val/Test'
    elif is_region_excluded[i]:
        opacity, region = 0.35, 'Originally excluded (LMC/SMC)'
    else:
        opacity, region = 0.10, 'Other'

    color_rgba = colors_list[i].replace('rgb(', 'rgba(').replace(')', f', {opacity})')
    wrmse_str = f'{wrmse_all[i]:.4g}' if np.isfinite(wrmse_all[i]) else 'N/A'
    fig.add_trace(go.Scatter(
        x=cl, y=cb, mode='lines', fill='toself',
        fillcolor=color_rgba, line=dict(color=color_rgba, width=0.5),
        hoverinfo='text',
        text=f'L={gl_all[i]:.2f} deg, B={gb_all[i]:.2f} deg<br>sWRMSE_coef={wrmse_str}<br>{region}',
        hovertemplate='%{text}<extra></extra>', showlegend=False,
    ))

fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(colorscale='Reds', showscale=True,
                colorbar=dict(title='sWRMSE_coef', thickness=15, len=0.7,
                              tickvals=[0, 0.25, 0.5, 0.75, 1.0],
                              ticktext=[f'{wrmse_vmin + (wrmse_vmax - wrmse_vmin) * t:.2g}'
                                        for t in [0, 0.25, 0.5, 0.75, 1.0]]),
                cmin=0, cmax=1, size=0),
    hoverinfo='none', showlegend=False,
))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=10, color='rgba(200,100,100,0.88)'), name='Train', showlegend=True))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=10, color='rgba(200,100,100,0.45)'), name='Val/Test', showlegend=True))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=10, color='rgba(200,100,100,0.35)'), name='Originally excluded (LMC/SMC)', showlegend=True))
fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=10, color='rgba(180,180,180,0.10)'), name='Other', showlegend=True))
fig.update_layout(
    title='sWRMSE_coef vs Galactic Coordinates (Train / Val-Test / LMC-SMC Excluded)',
    xaxis_title='Galactic Longitude [deg]', yaxis_title='Galactic Latitude [deg]',
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(scaleanchor='x', scaleratio=1),
    width=1200, height=900, template='plotly_white', hovermode='closest',
)
fig.show()

print('sWRMSE_coef summary by region:')
for name, mask in [('Train', is_train),
                   ('Val/Test', is_valtest),
                   ('Originally excluded (LMC/SMC)', is_region_excluded),
                   ('Other', is_other)]:
    vals = wrmse_all[mask & np.isfinite(wrmse_all)]
    if len(vals) == 0:
        print(f'  {name} (n={int(mask.sum())}): no finite sWRMSE_coef available')
    else:
        print(f'  {name} (n={int(mask.sum())}): '
              f'{vals.min():.4g} to {vals.max():.4g}, '
              f'median={np.median(vals):.4g}, finite={len(vals)}')


## Changelog

- **2026-08-18** (v4): fixed a physical/fit-unit mismatch in the basis-integrals precompute. Cell 14 hard-coded `physical_to_fit_flux_scale=1.0`, but the sci FITS COEF HDU was fit with `physical_to_fit_flux_scale=1e14`, so `mats['moon']` in the cache was in physical units (~1e-15 magnitude) while the sci coefs are in fit units. The bias-loss `_int_g x _diff_g` product for moon+zodi coefs was effectively zero even in the v3 cache (which had the mapping fix from v3). v4 derives the scale from `_first_mz.physical_to_fit_flux_scale` and passes it through all model constructors. Post-retrain results: **BIAS_moon+zodi dropped in every band** (B_cont 5.5%->3.5%, B_OH 9.9%->4.8%, G_OH 10.8%->7.4%, R_cont 11.0%->8.3%), **test mean_eRMSE 26.51->23.76 (-10%)**, seed std 2.18->0.30 (7x more stable), SSFR_deployment p95 24.6%->21.1%, worst-band-per-row p95 47.2%->29.4%. Also included in v4: n_workers 4->8 and the v3 optimizations (persistent per-worker FITS I/O with monkey-patched `fits.open`, matmul-based band-mean, uncompressed `.npz`, LSF-shared line-species hoist -- the last was auto-disabled because the detector LSF varies per row). Precompute time: 121 min -> 10 min (12x). Cache schema hash: 44977b8a0c48ce80.
- **2026-08-18** (v3): bumped `bias_loss_weight` from 1.0 to 100.0, added per-component-group bias breakdown (moon+zodi / diffuse / lines) to the per-epoch print, and stripped per-row pRMSE / pWRMSE reporting from the batch cell (the useful signal is the SSFR cell's per-band bias table; the batch cell now only prepares reconstruction helpers). **Post-run diagnosis**: SSFR bias table did NOT move (moon+zodi still +1.8 to +11.0%, diffuse -1.1 to -2.9%, lines 0 to -6.7%). Per-epoch diagnostic exposed the reason: the moon+zodi bias term was exactly 0.00000 across all seeds/epochs. Root cause: 31/442 coefficients in the basis integrals cache have all-zero integrals -- indices 402-430 (all 29 MoonZodi_bs), index 48, and index 441. The cache's unit-vector probe method (cell 14 `_discover_coef_to_mat_mapping`) requires `moon_predictor > 0` to detect moon+zodi coefs, but the first 20 probe rows all have moon-below-horizon in this dark-time dataset. So `coef_mapping[j] = []` for every MoonZodi_bs coef, `out[:, j] = 0`, and the bias-loss gradient for moon+zodi is identically zero. v3 settled at the same SSFR bias table as v0/v2 because the term physically can't push moon+zodi. Fix requires rebuilding the basis-integrals cache with either (a) probing until a moon-above-horizon row is found, or (b) structurally hard-coding MoonZodi_bs[k] -> (mats['moon'], k) + (mats['zodi'], k). See Open Issues #1.
- **2026-08-18** (v2): Added per-component per-band systematic flux bias loss term ($L_{\rm bias}$) to RETN training, driven by a precomputed basis-integrals cache (cell 14). Loss is dimensionless (relative bias normalized by RMS true flux), applied to `moon+zodi`, `diffuse`, `lines` groups over 7 wavelength bands. Default weight 1.0. Motivated by SSFR bias table showing systematic +3-11% moon+zodi over-prediction and -1-7% lines/diffuse under-prediction.
- **2026-08-18**: Notebook forked from `notebook_sky_interpolation_triplet_dual_encoder_group_mlp.ipynb` and rewritten around the Residual Emissivity Transfer Network (RETN). Removed the per-group PCA compressor, the dual-encoder group-head MLP, all hyperparameter sweep cells, and all evaluation cells anchored on coefficient-space eRMSE. Batch median pRMSE dropped from 0.447 (compressor-based) to **0.346** (RETN), essentially at the near-arm decomposition fidelity floor of 0.336. The eRMSE metric is retired in favor of the flux-based SSFR.


## Open Issues

1. **Systematic per-component bias.** SSFR bias table showed moon+zodi over-predicted by 3-11% and lines/diffuse under-predicted by 1-7% across bands before bias-loss retraining. Errors cancel at total-flux level but the imbalance is real. **Status (2026-08-18 v4 -- RESOLVED)**: fix (b) applied structurally (`MoonZodi_bs[k] -> [('moon', k)]`, since `mats['moon']` in the joint MoonZodi model is already the combined moon+zodi design block per `fit.py::_matrix_bundle`). Additional root cause discovered: cell 14 hard-coded `physical_to_fit_flux_scale=1.0` while the sci COEF HDU was fit with `physical_to_fit_flux_scale=1e14`, so moon+zodi basis integrals were 14 orders of magnitude too small. v4 derives the scale from `_first_mz` and passes it through. Post-fix: `BIAS_moon+zodi` reduced 1-5 percentage points in every band, test mean_eRMSE 26.51 -> 23.76, SSFR_deployment p95 24.6 -> 21.1, worst-band-per-row p95 47.2 -> 29.4. Ensemble seed std collapsed from 2.18 to 0.30 (7x more stable).

2. **Diffuse continuum in blue bands** now dominates the residual on the worst 10 rows in the SSFR diagnostic. Prior worst-case was full-moon; RETN solved that. The blue-diffuse residual is systematic and correlated with high-airmass rows — likely a decomposition-model issue rather than a network deficiency.

3. **Decomposition fidelity floor.** 672 of 700 row-band tuples in the SSFR analysis are "mixed-limited" (network error and decomposition floor comparable), 25 are floor-limited, only 3 are pure network-limited. Further pRMSE reductions on typical rows will require improving the decomposition itself, not the network.

4. **20% row-drop rate in the batch diagnostic** (`sci_pred_vs_true_pRMSE count=80/100`). The reconstruction path silently fails on ~20 rows; unclear which stage and whether these rows are pathological or just missed by upstream filters. Worth investigating.
